# 기후상권 전체분석 - 1회차 심사 취합 반영판

2025년 하반기 강남·춘천의 **선정 업종 카드 매출과 영업시간 강수의 관계**를 분석합니다.
인과적 기후변화 피해나 실제 사업체별 지원 자격을 확정하지 않습니다.

## 준비와 실행
1. 원본 카드·SKT·기상 zip 3종을 Drive의 `MyDrive/side_pjt/`에 놓습니다. 다른 경로는 설정 셀에서 지정합니다.
2. 기상 원본은 `chuncheon_asos_daily.csv`, `chuncheon_asos_hourly.csv`, `gangnam_aws_hourly.csv`입니다.
3. 위에서 아래로 실행합니다. 상권 API의 SERVICE_KEY가 없으면 공식 상권·통신 단계만 건너뜁니다.
4. 새 결과는 `out_round2`, 그림은 `fig_round2`, 보고서 그림은 `report_figs`에 저장합니다.
5. 마지막 셀에서 실행기록·결과 대응표·zip을 만들고, **실제 실행 출력을 포함한 노트북을 다운로드**합니다.

## 이번 변경
- 양수 피해 D와 낮을수록 취약한 회복탄력성 지수를 유지합니다.
- 회복일수는 실제 날짜 차이·연속 이틀·검열을 구분합니다.
- ERA5 모델과 09~22시 강수 시간창을 고정하고, 비 발생일 정밀도/재현율과 연간 변동성 검정을 추가합니다.
- 사건별 원화 매출 부족액 예측을 평균 손실 기준과 비교합니다. 공유 기상 블록 제외 검증과 과거만 쓰는 검증을 분리합니다.
- 손실의 방법 민감도, 보전율 20/30/50% 비용, 업종별 지원 경로, 지속시간 시나리오를 생성합니다.

**범위:** 기존 중첩 지수 검증은 전체기간 기준선을 이용한 사후 분석입니다. 새 순차 검증은 고정된 업종표본 내의 조건부 예측이며, 실제 예보 오차나 다른 지역·연도에 대한 외부 검증이 아닙니다.

**2026-10-21 4대 수정:** 기상 사건/매출 관측 분리, 8일째 회복 확인, 최종 지수 동일목표 순차검증, 기준선 맑은구간 진단 및 강수입력 제외 비교. 이전 v2 출력은 사후 참고용이며 최종 지수는 `5-5_회복탄력성지수_순차동일공식.csv`를 사용합니다. 실자료 실행 출력은 아직 포함하지 않았습니다.


In [ ]:
# 실행 설정. API 키는 결과 기록에 저장하지 않습니다.
import os
os.environ['BIGCON_V3'] = '0'   # 확장 달력 민감도는 '1'
SERVICE_KEY = ''
DIRECT_EXCLUDED_RY = ['실내/실외골프장', '할인점/슈퍼마켓/양판점']
# 아래 범위에서 설치하고 실제 사용 버전은 마지막 실행기록에 저장합니다.
!pip -q install "numpy>=1.26,<3" "pandas>=2.2,<3" "scipy>=1.12,<2" "statsmodels>=0.14,<0.15" "matplotlib>=3.8,<4" "pyproj>=3.6,<4" "openpyxl>=3.1,<4" koreanize-matplotlib


# 기후 취약 상권: 기후 반응 → 피해 → 회복 → 기후 회복탄력성 지수 (빅콘테스트 2026)

원본 3종(신한카드 데이터1·2, SKT 유동인구, 기상청)을 그대로 읽어 목차 4~7번을 한 번에 실행한다.

| 단계 | 목차 | 질문 |
|---|---|---|
| 0 | 데이터 및 분석 개요 | 무엇을, 어느 범위로 분석하는가 |
| 1 | 4. 기후 반응 분석 | 비·더위에 매출이 언제부터, 얼마나 변하는가 |
| 2 | 5. 피해 정량화 | 비 사건 하나가 얼마를 지우는가 |
| 3 | 6. 회복 분석 | 비가 그친 뒤 언제 돌아오는가 |
| 4 | 7. 지수 개발 | 기후 정보를 더한 지수(Model B)가 기존 지수(Model A)보다 다음 피해를 더 잘 맞히는가 |

모든 효과는 "같은 요일·같은 달·같은 달력 조건의 평범한 날 대비 %"다. 공휴일·추석·연말은 표본에서 뺐다.

## 0-1. 설정
팀에서 정한 값은 이 셀에 모두 있다. Colab에서는 Drive의 zip 3개가 있는 폴더를 `ZIP_DIR`에 적는다.

In [ ]:
import os, glob, zipfile, warnings, json
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
warnings.filterwarnings('ignore')

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import koreanize_matplotlib  # noqa: F401
    ZIP_DIR = os.environ.get('BIGCON_ZIP_DIR', '/content/drive/MyDrive/side_pjt/')          # zip 3개가 있는 폴더
    WORK = '/content/bigcon_final'
else:
    matplotlib.use('Agg')
    plt.rcParams['font.family'] = 'Malgun Gothic'
    ZIP_DIR = os.environ.get('BIGCON_ZIP_DIR', os.path.join(os.getcwd(), 'data'))
    WORK = os.environ.get('BIGCON_WORK', os.path.join(os.getcwd(), 'work', 'climate_round1'))
OUT = os.path.join(WORK, 'out_round2'); FIG = os.path.join(WORK, 'fig_round2')
OUT = os.environ.get('BIGCON_OUT', OUT); FIG = os.environ.get('BIGCON_FIG', FIG)      # 비교 실행용 출력 위치(기본은 work/out)
for p in (WORK, OUT, FIG): os.makedirs(p, exist_ok=True)
plt.rcParams.update({'axes.unicode_minus': False, 'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25})
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

GN, CC = '서울 강남구', '강원 춘천시'
REG = (GN, CC); SHORT = {GN: '강남', CC: '춘천'}; HOME = {GN: '서울', CC: '강원'}
COL = {GN: '#C2553F', CC: '#2F6F8F'}
DAYS = pd.date_range('2025-07-01', '2025-12-31')

# 표본
MIN_DAILY_CNT, MIN_OBS_DAYS = 30, 150      # 소규모 유닛 제외
CYCLE_R2 = 0.40                            # 날짜(1~31일)가 편차의 40% 이상을 설명하면 월중 결제 주기 유닛
EXCLUDE_RY = ['예식장/결혼서비스', '장례식장/묘지/장의사']   # 결제 시점이 날씨와 무관한 예약형 업종
# 결제일과 이용(방문)일이 어긋나는 업종: 그날 매출이 그날 방문을 뜻하지 않는다
TIMING_RY = {'정기·회원권형': ['헬스장', '스포츠시설'],
             '예약·선결제형': ['호텔/콘도', '여행사/항공사', '고속버스/철도/여객선'],
             '진료 예약형': ['동물병원']}

# 충격 기준
RAIN_BIZ_MM = 5.0     # 메인: 영업시간(09~22시) 강수 5mm 이상
HEAT_TMAX = 33.0      # 서브: 폭염 일최고 33℃ 이상
TROP_TMIN = 25.0      # 서브: 열대야 일최저 25℃ 이상
RAIN_BINS = [-0.01, 0, 5, 15, 30, 1e9]; RAIN_LAB = ['0', '0~5', '5~15', '15~30', '30+']

# 달력 (특수일은 표본에서 제외, 방학·축제는 통제 변수)
HOLIDAYS = ['2025-08-15', '2025-10-03', '2025-10-05', '2025-10-06', '2025-10-07',
            '2025-10-08', '2025-10-09', '2025-12-25']
CHUSEOK = ('2025-10-03', '2025-10-09'); BRIDGE = ['2025-10-10']; YEAR_END = ('2025-12-22', '2025-12-31')
# (v3 달력 규칙, 결과를 보기 전에 달력만으로 정함) 긴 연휴 블록 전체 + 연휴 직후 첫 영업일(10일 연휴는 2일) + 추석 직전 장보기 기간
LONG_BREAK = [('2025-08-16', '2025-08-18'), ('2025-09-30', '2025-10-01'), ('2025-10-11', '2025-10-14')]
USE_LONG_BREAK = os.environ.get('BIGCON_V3') == '1'
FESTIVAL_CC = [('2025-10-16', '2025-10-19')]       # 춘천막국수닭갈비축제
SCHOOL_VAC = [('2025-07-21', '2025-08-17'), ('2025-12-29', '2025-12-31')]   # 초중고 방학(추정)

# 결제분류 엑셀의 업종 매핑 (핵심 6분야만 분석)
SECTOR_MAP = {
    ('01', '외식·음료', '핵심'): ['한식', '기타요식', '커피전문점', '양식', '일식', '중식', '유흥업소', '제과점', '패스트푸드'],
    ('02', '식품·생활소매', '핵심'): ['할인점/슈퍼마켓/양판점', '편의점', '기타식품', '생활잡화/수입상품점', '정육점', '농수산물', '주류판매', '화원', '애완동물', '완구/아동용자전거', '중고품판매점', '수제용품점'],
    ('03', '패션·뷰티소매', '핵심'): ['의복/의류', '패션잡화', '화장품', '시계/귀금속'],
    ('04', '여가·문화·스포츠', '핵심'): ['스포츠/레저용품', '헬스장', '실내/실외골프장', '서점', '스포츠시설', '노래방', '문화용품', '영화/공연', '게임방/오락실', '종합레저타운/놀이동산', '악기/음반'],
    ('05', '숙박·여행·교통', '핵심'): ['호텔/콘도', '고속버스/철도/여객선', '모텔,여관,기타숙박', '여행사/항공사', '주차장', '택시'],
    ('06', '생활·개인서비스', '핵심'): ['미용실', '예식장/결혼서비스', '동물병원', '장례식장/묘지/장의사', '미용서비스', '세탁소', '안마/마사지', '싸우나/목욕탕'],
    ('07', '의료·건강', '보조'): ['일반병원', '기타의료', '종합병원', '치과병원', '약국', '한의원', '보건소'],
    ('08', '교육', '보조'): ['학원/학습지', '유치원', '독서실', '학교등록금'],
    ('09', '자동차·연료', '보조'): ['주유소', '자동차서비스', 'LPG가스', '자동차용품', '수입자동차', '오토바이', '중고차판매'],
    ('10', '내구재·정보통신', '보조'): ['컴퓨터/소프트웨어', '인테리어/건축자재/주방기구', '통신요금(이동,시내전화)', '가전', '사무기기/문구용품', '가구', '통신요금(PC통신,무선호출)'],
    ('11', '전문·사업서비스', '보조'): ['연구/번역서비스', '법률/사무서비스', '회계/변리서비스', '부동산중개'],
    ('12', '대형유통·특수채널', '보조'): ['쇼핑몰', '백화점', '면세점'],
    ('90', '비식별·공공·금융', '제외'): ['ZZ_나머지', '세금공과금', '보험', '상품권/복권', '결제대행(PG)'],
    ('91', '업태불명·검토필요', '제외'): ['방문판매/다단계판매', '기타유통', '전용매장', '체인점'],
}
# 상권 유형: 날씨를 보고 갈지 말지 고르는 소비 vs 매일 필요한 소비
TYPE_OF = {'03': '방문·선택형', '04': '방문·선택형', '05': '방문·선택형',
           '01': '생활밀착형', '02': '생활밀착형', '06': '생활밀착형'}
RY_INFO = pd.DataFrame([(ry, c, n, t) for (c, n, t), lst in SECTOR_MAP.items() for ry in lst],
                       columns=['MCT_RY_CD', '분야코드', '분야', '분석구분'])
RY_INFO['유형'] = RY_INFO.분야코드.map(TYPE_OF)
SECTORS = [n for (c, n, t) in SECTOR_MAP if t == '핵심']

def savefig(name):
    plt.tight_layout(); plt.savefig(os.path.join(FIG, name + '.png'), dpi=150, bbox_inches='tight')
    plt.show() if IN_COLAB else None; plt.close('all')

def in_ranges(s, ranges):
    out = pd.Series(False, index=s.index)
    for a, b in ranges: out |= s.between(pd.Timestamp(a), pd.Timestamp(b))
    return out

## 0-2. 원본 불러오기
zip을 풀어 `WORK/raw`에 두고, 파일 이름으로 찾는다.

In [ ]:
RAW = os.path.join(WORK, 'raw')
INPUT_FILES = []
def _name(info):
    n = info.filename
    if not info.flag_bits & 0x800:
        try: n = n.encode('cp437').decode('cp949')
        except Exception: pass
    return n
for zp in glob.glob(os.path.join(ZIP_DIR, '*.zip')):
    tag = os.path.splitext(os.path.basename(zp))[0]
    dst = os.path.join(RAW, tag)
    marker = os.path.join(dst, '.extracted_ok')
    if os.path.isfile(marker): continue
    with zipfile.ZipFile(zp) as z:
        for info in z.infolist():
            n = _name(info); p = os.path.abspath(os.path.join(dst, n))
            if os.path.commonpath([p, os.path.abspath(dst)]) != os.path.abspath(dst):
                raise ValueError('zip 경로가 추출 폴더를 벗어났습니다.')
            if n.endswith('/'): os.makedirs(p, exist_ok=True); continue
            os.makedirs(os.path.dirname(p), exist_ok=True)
            with z.open(info) as src, open(p, 'wb') as f: f.write(src.read())
    with open(marker, 'w', encoding='utf-8') as f: f.write('ok')

def find(pattern):
    hits = sorted(set(glob.glob(os.path.join(RAW, '**', pattern), recursive=True) +
                      glob.glob(os.path.join(ZIP_DIR, '**', pattern), recursive=True)))
    if not hits: raise FileNotFoundError(f'필요한 원본 없음: {pattern}. ZIP_DIR과 원본 zip 구성을 확인하세요.')
    INPUT_FILES.append(hits[0])
    return hits[0]

def load_card(pattern):
    d = pd.read_csv(find(pattern), sep='\t', encoding='cp949', dtype={'TA_YMD': str})
    d['date'] = pd.to_datetime(d.TA_YMD, format='%Y%m%d'); d['AGE_CCD'] = d.AGE_CCD.str.replace(' ', '')
    return d.drop(columns='TA_YMD')
d1 = load_card('*데이터1.txt')     # 시간대 4구간, 법인 포함
d2 = load_card('*데이터2.txt')     # 고객 거주 시도, 법인 제외
print(f'데이터1 {len(d1):,}행 / 데이터2 {len(d2):,}행')

## 0-3. 분석 표본: 업종 × 지역 = 상권 유닛
걸러내는 순서(깔때기)를 그대로 표로 남긴다.

In [ ]:
UNIT = ['MCT_SGG_CD', 'MCT_RY_CD']
d1['low'] = d1.USE_CNT <= 6
ud = (d1.groupby(UNIT + ['date'], as_index=False)
        .agg(amt=('TS_AT', 'sum'), cnt=('USE_CNT', 'sum'), n_cells=('USE_CNT', 'size'), n_low=('low', 'sum')))
info = ud.groupby(UNIT).agg(obs_days=('date', 'nunique'), cnt=('cnt', 'sum'), amt=('amt', 'sum'),
                            n_cells=('n_cells', 'sum'), n_low=('n_low', 'sum')).reset_index()
info = info.merge(RY_INFO, on='MCT_RY_CD', how='left')
assert info.분야.notna().all()
info['daily_cnt'] = info.cnt / info.obs_days
info['mask_share'] = info.n_low / info.n_cells        # 5~6건 경계 셀 비중(높을수록 감소가 과장될 위험)

# 달력
cal = pd.DataFrame({'date': DAYS})
cal['dow'] = cal.date.dt.dayofweek; cal['month'] = cal.date.dt.month; cal['dom'] = cal.date.dt.day
cal['weekend'] = cal.dow >= 5
hol = cal.date.isin(pd.to_datetime(HOLIDAYS))
cal['special'] = (hol | cal.date.between(*map(pd.Timestamp, CHUSEOK)) | (hol.shift(-1, fill_value=False) & ~hol)
                  | cal.date.isin(pd.to_datetime(BRIDGE)) | cal.date.between(*map(pd.Timestamp, YEAR_END))
                  | (in_ranges(cal.date, LONG_BREAK) if USE_LONG_BREAK else False))
cal['month_end'] = cal.date.dt.is_month_end
cal['school_vac'] = in_ranges(cal.date, SCHOOL_VAC)
cal['festival_cc'] = in_ranges(cal.date, FESTIVAL_CC)

# 월중 결제 주기 판정: 평범한 날의 요일 중앙값 대비 편차를 날짜(1~31일)가 얼마나 설명하나
tmp = ud.merge(cal[['date', 'dow', 'dom', 'special']], on='date')
tmp = tmp[~tmp.special].copy(); tmp['la'] = np.log(tmp.amt)
tmp['dev'] = tmp.la - tmp.groupby(UNIT + ['dow']).la.transform('median')
def dom_r2(g):
    fit = g.groupby('dom').dev.transform('mean')
    return 1 - ((g.dev - fit) ** 2).sum() / ((g.dev - g.dev.mean()) ** 2).sum()
info = info.merge(tmp.groupby(UNIT)[['dev', 'dom']].apply(dom_r2).rename('dom_r2').reset_index(), on=UNIT, how='left')

steps = [('전체 유닛', np.ones(len(info), bool))]
m = (info.daily_cnt >= MIN_DAILY_CNT) & (info.obs_days >= MIN_OBS_DAYS); steps.append(('소규모 제외', m))
m = m & info.분석구분.eq('핵심'); steps.append(('핵심 6분야만', m))
m = m & ~(info.dom_r2 >= CYCLE_R2); steps.append(('월중 결제 주기 제외', m))
m = m & ~info.MCT_RY_CD.isin(EXCLUDE_RY); steps.append(('예식장·장례식장 제외', m))
m = m & ~info.MCT_RY_CD.isin(sum(TIMING_RY.values(), [])); steps.append(('정기결제·예약형 제외', m))
tot_amt = info.amt.sum()
funnel = pd.DataFrame([{'단계': s, '유닛': int(k.sum()), '강남': int((k & info.MCT_SGG_CD.eq(GN)).sum()),
                        '춘천': int((k & info.MCT_SGG_CD.eq(CC)).sum()), '매출 비중(%)': info.amt[k].sum() / tot_amt * 100}
                       for s, k in steps]).round(1)
print(funnel.to_string(index=False))
cycle_excluded = info[info.분석구분.eq('핵심') & info.dom_r2.ge(CYCLE_R2) & info.daily_cnt.ge(MIN_DAILY_CNT)]
print('월중 주기로 빠진 핵심 유닛:', [f'{SHORT[r.MCT_SGG_CD]} {r.MCT_RY_CD}' for r in cycle_excluded.itertuples()])
units = info[m].reset_index(drop=True); units['uid'] = units.index
units['mask_risk'] = pd.cut(units.mask_share, [-1, .3, .5, 1], labels=['low', 'mid', 'high']).astype(str)

## 0-4. 기상: 영업시간 강수와 기온
춘천 ASOS는 강수의 `-9`만 0으로 바꾸고 기온은 그대로 둔다(영하 9.0℃가 실제 값인 날이 있음). 강남 AWS는 시간 자료를 일로 묶고, 하루 24시간이 다 관측되지 않은 날은 "기상 미확정"으로 표본에서 뺀다.

In [ ]:
ch = pd.read_csv(find('chuncheon_asos_daily.csv'), encoding='utf-8-sig')
ch['date'] = pd.to_datetime(ch.TM)
ch = ch.set_index('date').rename(columns={'TA_MAX': 'ta_max', 'TA_MIN': 'ta_min', 'RN_DAY': 'rn_day'})[['ta_max', 'ta_min', 'rn_day']].reindex(DAYS)
ch['rn_day'] = ch.rn_day.mask(ch.rn_day.eq(-9), 0.).where(ch.rn_day.ge(0) | ch.rn_day.eq(-9))
ch['wx_ok'] = ch[['ta_max', 'ta_min', 'rn_day']].notna().all(axis=1)
h = pd.read_csv(find('gangnam_aws_hourly.csv'), encoding='utf-8-sig', parse_dates=['TM'])
h['RN_HR1'] = h.RN_HR1.where(h.RN_HR1 >= 0)
h = h.drop_duplicates('TM').copy()
h['date'] = (h.TM - pd.Timedelta(minutes=1)).dt.normalize()
gn = h.groupby('date').agg(ta_max=('TA_MAX', 'max'), ta_min=('TA_MIN', 'min'), rn_day=('RN_HR1', 'sum'),
                           n_t=('TA', 'count'), n_r=('RN_HR1', 'count')).reindex(DAYS)
gn['wx_ok'] = (gn.n_t.fillna(0) >= 24) & (gn.n_r.fillna(0) >= 24); gn = gn.drop(columns=['n_t', 'n_r'])
hc = pd.read_csv(find('chuncheon_asos_hourly.csv'), encoding='utf-8-sig', parse_dates=['TM'])
def rain_biz(tm, rn):          # 관측 시각은 "그 시각까지 1시간": 10~22시 관측 = 09~22시 강수
    d = pd.DataFrame({'date': (tm - pd.Timedelta(minutes=1)).dt.normalize(), 'hr': tm.dt.hour, 'rn': rn})
    return d[d.hr.between(10, 22)].groupby('date').rn.sum(min_count=13).reindex(DAYS)
ch['rn_biz'] = rain_biz(hc.TM, hc.RN.mask(hc.RN.eq(-9), 0.).where(hc.RN.ge(0) | hc.RN.eq(-9)))
ch['wx_ok'] &= ch.rn_biz.notna()
gn['rn_biz'] = rain_biz(h.TM, h.RN_HR1)
wx = pd.concat({GN: gn, CC: ch}, names=['MCT_SGG_CD', 'date']).reset_index()
wx['rain'] = wx.rn_biz >= RAIN_BIZ_MM
wx['heat'] = wx.ta_max >= HEAT_TMAX
wx['trop'] = wx.ta_min >= TROP_TMIN
wx['rain_bin'] = pd.cut(wx.rn_biz, RAIN_BINS, labels=RAIN_LAB).astype(str)
wx = wx.merge(cal, on='date')
wx['use'] = wx.wx_ok & ~wx.special & wx.rn_biz.notna()           # 분석에 쓰는 날
wx['festival'] = wx.festival_cc & wx.MCT_SGG_CD.eq(CC)
print(wx[wx.use].groupby('MCT_SGG_CD').agg(분석일=('date', 'size'), 비_5mm=('rain', 'sum'), 폭염=('heat', 'sum'), 열대야=('trop', 'sum')))

## 0-5. 분석용 패널
유닛 × 일 매출에 거주·외지 고객(데이터2), 법인, 시간대 매출을 붙인다. 지역·유형·분야 합계 시계열도 만든다.

In [ ]:
idx = pd.MultiIndex.from_product([units.uid, DAYS], names=['uid', 'date'])
panel = (ud.merge(units[UNIT + ['uid']]).set_index(['uid', 'date'])[['amt', 'cnt']].reindex(idx).reset_index()
           .merge(units[['uid'] + UNIT + ['분야코드', '분야', '유형', 'mask_risk']], on='uid'))
d2['outside'] = d2.CLN_SGG_CD != d2.MCT_SGG_CD.map(HOME)
seg = d2.groupby(UNIT + ['date', 'outside']).TS_AT.sum().unstack(fill_value=0).rename(columns={False: 'amt_res', True: 'amt_out'})
corp = d1[d1.SEX_CCD.eq('법인')].groupby(UNIT + ['date']).TS_AT.sum().rename('amt_corp')
tb = d1.groupby(UNIT + ['date', 'TIME_GB']).TS_AT.sum().unstack(fill_value=0); tb.columns = ['tb_' + c for c in tb.columns]
panel = panel.merge(pd.concat([seg, corp, tb], axis=1).reset_index(), on=UNIT + ['date'], how='left')
panel = panel.merge(wx[['MCT_SGG_CD', 'date', 'use', 'rain', 'heat', 'trop', 'rn_biz', 'rain_bin', 'ta_max', 'ta_min',
                        'dow', 'month', 'month_end', 'school_vac', 'festival', 'weekend']], on=['MCT_SGG_CD', 'date'])
panel['missing'] = panel.amt.isna()
print(f'패널 {len(panel):,}행 (유닛 {units.uid.nunique()} × {len(DAYS)}일), 매출 누락 {int(panel.missing.sum())}')

VALS = ['amt', 'cnt', 'amt_res', 'amt_out', 'amt_corp', 'tb_00_05', 'tb_06_11', 'tb_12_17', 'tb_18_23']
def series(level):
    """level: 'unit' | '분야' | '유형' | '합계' → 지역×그룹×일 합계"""
    p = panel[~panel.missing]
    keys = ['MCT_SGG_CD'] + ({'unit': ['uid'], '분야': ['분야'], '유형': ['유형'], '합계': []}[level]) + ['date']
    s = p.groupby(keys)[VALS].sum().reset_index()
    if level == '합계': s['그룹'] = '핵심 합계'
    elif level != 'unit': s = s.rename(columns={level: '그룹'})
    return s.merge(wx[['MCT_SGG_CD', 'date', 'use', 'rain', 'heat', 'trop', 'rn_biz', 'rain_bin', 'ta_max', 'ta_min',
                       'dow', 'month', 'month_end', 'school_vac', 'festival']], on=['MCT_SGG_CD', 'date'])
S = pd.concat([series('합계'), series('유형'), series('분야')], ignore_index=True)
S['y'] = np.log(S.amt.where(S.amt.gt(0)))
SU = series('unit'); SU['y'] = np.log(SU.amt.where(SU.amt.gt(0)))
GROUPS = ['핵심 합계', '방문·선택형', '생활밀착형'] + SECTORS
print(S.groupby(['MCT_SGG_CD', '그룹']).amt.mean().div(1e8).round(1).unstack(0).reindex(GROUPS).rename(columns=SHORT).to_string())

## 공통 도구: 기준선 회귀
로그 매출 = 요일 + 월 + 월말 + 방학 + 축제 + (날씨 항) 를 HAC(7일) 표준오차로 추정한다. 효과는 `exp(계수) − 1`.

In [ ]:
def calendar_X(d):
    X = pd.DataFrame(index=d.index)
    for c in ['month_end', 'school_vac', 'festival']:
        if d[c].astype(float).std() > 0: X[c] = d[c].astype(float)
    X = X.join(pd.get_dummies(d.month, prefix='m', drop_first=True, dtype=float))
    X = X.join(pd.get_dummies(d.dow, prefix='dow', drop_first=True, dtype=float))
    return sm.add_constant(X, has_constant='add')

def rain_bin_X(d):
    return pd.get_dummies(pd.Categorical(d.rain_bin, categories=RAIN_LAB), prefix='r', dtype=float).set_index(d.index).drop(columns='r_0')

def fit(y, X):
    ok = np.isfinite(y) & np.isfinite(X).all(axis=1)
    X = X[ok]; X = X.loc[:, (X.std() > 0) | (X.columns == 'const')]
    return sm.OLS(y[ok], X).fit(cov_type='HAC', cov_kwds={'maxlags': 7})

def pct(m, k, scale=1.0):
    if k not in m.params: return dict(효과=np.nan, 하한=np.nan, 상한=np.nan, t=np.nan, p=np.nan)
    b, se = m.params[k] * scale, m.bse[k] * scale
    if m.df_resid <= 0 or not np.isfinite(se) or se <= 0:
        return dict(효과=np.expm1(b)*100, 하한=np.nan, 상한=np.nan, t=np.nan, p=np.nan, 표본수=m.nobs, 잔차자유도=m.df_resid)
    return dict(효과=(np.exp(b) - 1) * 100, 하한=(np.exp(b - 1.96 * se) - 1) * 100, 상한=(np.exp(b + 1.96 * se) - 1) * 100,
                t=b / se, p=2 * stats.norm.sf(abs(b / se)), 표본수=m.nobs, 잔차자유도=m.df_resid)

def bh(p):   # Benjamini-Hochberg q값
    p = np.asarray(p, float); n = np.isfinite(p).sum(); q = np.full_like(p, np.nan)
    o = np.argsort(np.where(np.isfinite(p), p, np.inf))[:n]
    r = p[o] * n / np.arange(1, n + 1); q[o] = np.minimum.accumulate(r[::-1])[::-1].clip(max=1)
    return q

def each_series(level_df, keys=('MCT_SGG_CD', '그룹')):
    for k, g in level_df[level_df.use].groupby(list(keys)): yield k, g.sort_values('date')

def safe_wilcoxon(values):
    a = pd.Series(values).replace([np.inf, -np.inf], np.nan).dropna().to_numpy()
    if len(a) < 6: return np.nan
    if np.allclose(a, 0): return 1.0
    return float(stats.wilcoxon(a).pvalue)


# 1단계. 기후 반응 분석 (목차 4)
## 1-1. 강수량 구간별 반응
비가 오지 않은 날 대비. 폭염·열대야는 함께 통제한다.

In [ ]:
rows = []
for (sgg, grp), g in each_series(S):
    m = fit(g.y, calendar_X(g).join(rain_bin_X(g)).join(g[['heat', 'trop']].astype(float)))
    for lab in RAIN_LAB[1:]:
        rows.append({'지역': SHORT[sgg], '그룹': grp, '강수(mm)': lab, '일수': int((g.rain_bin == lab).sum()), **pct(m, 'r_' + lab)})
R11 = pd.DataFrame(rows); R11.round(2).to_csv(os.path.join(OUT, '1-1_강수구간_반응.csv'), index=False, encoding='utf-8-sig')
show = R11.assign(v=R11.효과.round(1).astype(str) + np.where(R11.p < .05, '*', ''))
print(show.pivot_table(index=['지역', '그룹'], columns='강수(mm)', values='v', aggfunc='first')[RAIN_LAB[1:]]
      .reindex(GROUPS, level=1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
pal = dict(zip(GROUPS, ['k', '#C2553F', '#2F6F8F', '#E07A5F', '#81B29A', '#8C6BB1', '#3D405B', '#F2CC8F', '#7F7F7F']))
for ax, sgg in zip(axes, REG):
    for grp in GROUPS:
        s = R11[(R11.지역.eq(SHORT[sgg])) & (R11.그룹.eq(grp))]
        yv = np.r_[0, s.효과.values]; x = np.arange(len(RAIN_LAB))
        ax.plot(x, yv, marker='o', color=pal[grp], lw=2.6 if grp == '핵심 합계' else 1.1, label=grp,
                ls='-' if grp in ('핵심 합계',) or grp in SECTORS else '--')
        sig = np.r_[False, (s.p < .05).values]; ax.scatter(x[sig], yv[sig], s=70, facecolor='none', edgecolor=pal[grp])
    n = R11[(R11.지역.eq(SHORT[sgg])) & (R11.그룹.eq('핵심 합계'))].일수.values
    ax.set_xticks(range(len(RAIN_LAB))); ax.set_xticklabels(['없음'] + [f'{l}\n({k}일)' for l, k in zip(RAIN_LAB[1:], n)])
    ax.axhline(0, color='k', lw=.8); ax.set(title=f'{SHORT[sgg]}: 영업시간 강수량별 매출 변화', xlabel='영업시간(09~22시) 강수량 (mm)')
axes[0].set_ylabel('비 안 온 날 대비 (%)'); axes[1].legend(fontsize=7.5, ncol=2, loc='lower left')
savefig('1-1_강수구간_반응')

## 1-2. 강수 임계값: "몇 mm 이상"으로 잘라야 매출을 가장 잘 설명하나
후보 기준마다 "그 이상 온 날" 더미를 넣고 AIC가 가장 낮은 값을 고른다. 주 단위 블록 부트스트랩(300회)으로 구간을 낸다.

In [ ]:
RAIN_CUT_CANDIDATES = [1, 2, 3, 4, 5, 6, 7.5, 10, 12.5, 15, 20, 30]
def best_cut(y, Xc, rn):
    aic = []
    for c in RAIN_CUT_CANDIDATES:
        X = Xc.copy(); X['wet'] = (rn >= c).astype(float)
        aic.append(sm.OLS(y, X).fit().aic)
    return RAIN_CUT_CANDIDATES[int(np.argmin(aic))], aic
rng = np.random.default_rng(42); rows = []; prof = {}
for (sgg, grp), g in each_series(S[S.그룹.isin(['핵심 합계', '방문·선택형', '생활밀착형'])]):
    g = g.reset_index(drop=True)
    Xc = calendar_X(g).join(g[['heat', 'trop']].astype(float)); Xc = Xc.loc[:, (Xc.std() > 0) | (Xc.columns == 'const')]
    c0, aic = best_cut(g.y, Xc, g.rn_biz); prof[(sgg, grp)] = aic
    wk = (g.date - g.date.min()).dt.days // 7; weeks = wk.unique(); bs = []
    for _ in range(300):
        ii = np.concatenate([np.where(wk == w)[0] for w in rng.choice(weeks, len(weeks))])
        Xb = Xc.iloc[ii].reset_index(drop=True); Xb = Xb.loc[:, (Xb.std() > 0) | (Xb.columns == 'const')]
        bs.append(best_cut(g.y.iloc[ii].reset_index(drop=True), Xb, g.rn_biz.iloc[ii].reset_index(drop=True))[0])
    X = Xc.copy(); X['wet'] = (g.rn_biz >= c0).astype(float); e = pct(fit(g.y, X), 'wet')
    rows.append({'지역': SHORT[sgg], '그룹': grp, '최적 기준(mm)': c0, '90% 구간': f'{np.percentile(bs, 5):g}~{np.percentile(bs, 95):g}',
                 '그 이상인 날': int((g.rn_biz >= c0).sum()), '그날 효과(%)': round(e['효과'], 1), 't': round(e['t'], 2)})
R12 = pd.DataFrame(rows); R12.to_csv(os.path.join(OUT, '1-2_강수_임계값.csv'), index=False, encoding='utf-8-sig')
print(R12.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
for (sgg, grp), aic in prof.items():
    if grp != '핵심 합계': continue
    ax.plot(RAIN_CUT_CANDIDATES, np.array(aic) - min(aic), marker='o', color=COL[sgg], label=SHORT[sgg])
ax.axvline(RAIN_BIZ_MM, color='k', ls=':', lw=1)
ax.set(xlabel='영업시간 강수 기준 (mm 이상)', ylabel='AIC − 최솟값 (낮을수록 잘 설명)', title='강수 기준별 설명력 (핵심 합계)'); ax.legend()
savefig('1-2_강수_임계값')

## 1-3. 비 온 날(5mm 이상) 효과: 그룹별
메인 충격 정의로 지역 × 그룹 효과를 정리한다. 방문(건수)과 객단가, 거주·외지·법인 고객으로도 나눈다.

In [ ]:
rows = []
for (sgg, grp), g in each_series(S):
    X = calendar_X(g).join(g[['rain', 'heat', 'trop']].astype(float))
    r = {'지역': SHORT[sgg], '그룹': grp, '비 온 날': int(g.rain.sum())}
    e = pct(fit(g.y, X), 'rain'); r.update({'매출(%)': e['효과'], '하한': e['하한'], '상한': e['상한'], 'p': e['p']})
    for nm, yv in [('건수', np.log(g.cnt)), ('객단가', np.log(g.amt / g.cnt)), ('거주 고객', np.log(g.amt_res)),
                   ('외지 고객', np.log(g.amt_out)), ('법인', np.log(g.amt_corp.where(g.amt_corp > 0)))]:
        r[nm + '(%)'] = pct(fit(yv, X), 'rain')['효과']
    rows.append(r)
R13 = pd.DataFrame(rows); R13['q'] = bh(R13.p)
R13.round(3).to_csv(os.path.join(OUT, '1-3_비_효과_그룹별.csv'), index=False, encoding='utf-8-sig')
print(R13.set_index(['지역', '그룹']).reindex(GROUPS, level=1).round(1).to_string())

## 1-4. 더위 (서브 주제): 1℃당 매출 변화와 폭염일 효과
7~9월만 쓴다. 방학 통제를 넣었을 때와 뺐을 때를 함께 보여 휴가철이 결과를 얼마나 바꾸는지 드러낸다.

In [ ]:
rows = []
for (sgg, grp), g in each_series(S[S.month.isin([7, 8, 9])]):
    for vac in (True, False):
        Xc = calendar_X(g) if vac else calendar_X(g).drop(columns=['school_vac'], errors='ignore')
        Xc = Xc.join(rain_bin_X(g))
        e1 = pct(fit(g.y, Xc.join(g[['ta_max']])), 'ta_max')
        e2 = pct(fit(g.y, Xc.join(g[['heat']].astype(float))), 'heat')
        rows.append({'지역': SHORT[sgg], '그룹': grp, '방학 통제': '포함' if vac else '제외',
                     '1℃당(%)': e1['효과'], '1℃ 하한': e1['하한'], '1℃ 상한': e1['상한'], '1℃ p': e1['p'],
                     '폭염일(%)': e2['효과'], '폭염 p': e2['p']})
R14 = pd.DataFrame(rows); R14.round(3).to_csv(os.path.join(OUT, '1-4_더위_반응.csv'), index=False, encoding='utf-8-sig')
print(R14.pivot_table(index=['지역', '그룹'], columns='방학 통제', values=['1℃당(%)', '폭염일(%)'])
         .reindex(GROUPS, level=1).round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharex=True)
for ax, sgg in zip(axes, REG):
    s = R14[(R14.지역.eq(SHORT[sgg])) & (R14['방학 통제'] == '포함')].set_index('그룹').reindex(GROUPS)
    yy = np.arange(len(GROUPS))[::-1]
    ax.errorbar(s['1℃당(%)'], yy, xerr=[s['1℃당(%)'] - s['1℃ 하한'], s['1℃ 상한'] - s['1℃당(%)']], fmt='o', color=COL[sgg], capsize=3)
    ax.axvline(0, color='k', lw=.8); ax.set_yticks(yy); ax.set_yticklabels(GROUPS)
    ax.set(title=f'{SHORT[sgg]}: 일최고기온 1℃당 매출 변화 (7~9월)', xlabel='% / ℃ (95% 구간)')
savefig('1-4_더위_기울기')

## 1-5. 같은 양의 비라면, 오래 오는 비와 세게 오는 비 중 무엇이 더 아픈가
비가 조금이라도 온 날만 놓고 총량(log)을 통제한 뒤, 영업시간 중 비 온 시간 수(0.5mm 이상인 시간)와 최대 시간당 강수(강도)를 함께 넣는다. 기후변화로 늘어나는 "지속 강우"와 "국지성 소나기" 가운데 상권에 무엇이 더 큰 충격인지 가린다.

In [ ]:
def rain_hourly(tm, rn):
    d = pd.DataFrame({'date': (tm - pd.Timedelta(minutes=1)).dt.normalize(), 'hr': tm.dt.hour, 'rn': rn})
    d = d[d.hr.between(10, 22)]
    return pd.DataFrame({'wet_h': (d.rn >= 0.5).groupby(d.date).sum(), 'rn_max': d.groupby('date').rn.max()}).reindex(DAYS)
RH = pd.concat({GN: rain_hourly(h.TM, h.RN_HR1), CC: rain_hourly(hc.TM, hc.RN.mask(hc.RN.eq(-9), 0.).where(hc.RN.ge(0) | hc.RN.eq(-9)))},
               names=['MCT_SGG_CD', 'date']).reset_index()
rows = []
for (sgg, grp), g in each_series(S.merge(RH, on=['MCT_SGG_CD', 'date'])):
    w = g[g.rn_biz > 0]
    X = calendar_X(w).join(pd.DataFrame({'log총량': np.log1p(w.rn_biz), '비온시간': w.wet_h, '최대강도': w.rn_max}, index=w.index)).join(w[['heat', 'trop']].astype(float))
    m = fit(w.y, X)
    rows.append({'지역': SHORT[sgg], '그룹': grp, '비 온 날': len(w),
                 '비 1시간 더(%)': pct(m, '비온시간')['효과'], 'p_시간': pct(m, '비온시간')['p'],
                 '강도 1mm/h 더(%)': pct(m, '최대강도')['효과'], 'p_강도': pct(m, '최대강도')['p']})
R15 = pd.DataFrame(rows); R15.round(3).to_csv(os.path.join(OUT, '1-5_지속vs강도.csv'), index=False, encoding='utf-8-sig')
print(R15.set_index(['지역', '그룹']).reindex(GROUPS, level=1).round(3).to_string())

# 2단계. 피해 정량화 (목차 5)
## 2-0. 비정상매출률과 비 사건
기준선 = 비 온 날을 뺀 날로 학습한 "요일·월·달력·방학·더위" 모형의 예측. 비정상매출률(abn) = 실제 로그매출 − 기준선. 이어진 비 온 날(하루 쉬어도 같은 사건)을 한 사건으로 묶는다.

In [ ]:
def add_abn(df, keys):
    out = []
    for k, g in df.groupby(keys):
        g = g.sort_values('date').copy()
        X = calendar_X(g).join(g[['heat', 'trop']].astype(float))
        tr = g.use & ~g.rain & g.y.notna()
        Xt = X[tr]; keep = [c for c in Xt.columns if c == 'const' or Xt[c].std() > 0]
        mdl = sm.OLS(g.y[tr], Xt[keep]).fit()
        g['base'] = X[keep].dot(mdl.params); g['abn'] = g.y - g.base
        g['sd_calm'] = g.abn[tr].std()
        out.append(g)
    return pd.concat(out, ignore_index=True)
S = add_abn(S, ['MCT_SGG_CD', '그룹'])
SU = add_abn(SU, ['MCT_SGG_CD', 'uid'])

ev_rows = []
for sgg in REG:
    w = wx[wx.MCT_SGG_CD.eq(sgg)].sort_values('date')
    eid = 0; last = None
    for d in w.date[w.rain & w.wx_ok]:
        if last is None or (d - last).days > 2: eid += 1
        ev_rows.append({'MCT_SGG_CD': sgg, 'event': f'{SHORT[sgg]}{eid:02d}', 'date': d}); last = d
EVD = pd.DataFrame(ev_rows)
EVENTS = EVD.groupby(['MCT_SGG_CD', 'event']).date.agg(start='min', end='max', n_days='size').reset_index()
print(EVENTS.groupby('MCT_SGG_CD').agg(사건=('event', 'size'), 비온날=('n_days', 'sum')))
print('주의: add_abn 기준선은 전체기간 사후 기준선입니다. 미래 예측 성능은 10-* 순차검증에서 별도로 평가합니다.')


## 2-1. 원화로 본 피해: 유닛별 효과 → 총손실·총이득·상쇄율
유닛별 비 효과를 같은 분야 평균 쪽으로 당겨 안정화(경험적 베이즈)한 뒤, 비 온 날 실제 매출에 적용한다.

In [ ]:
rows = []
for uid, g in SU[SU.use].groupby('uid'):
    m = fit(g.y, calendar_X(g).join(g[['rain', 'heat', 'trop']].astype(float)))
    if 'rain' not in m.params: continue
    rows.append({'uid': uid, 'b': m.params['rain'], 'se': m.bse['rain'], 'p': m.pvalues['rain'],
                 'rain_amt': g.amt[g.rain].sum(), 'n_rain': int(g.rain.sum())})
UE = pd.DataFrame(rows).merge(units[['uid', 'MCT_SGG_CD', 'MCT_RY_CD', '분야', '유형', 'mask_risk']], on='uid')

def eb_shrink(df, b='b', se='se', by=('MCT_SGG_CD', '분야'), out_col='b_eb'):
    out = df.copy(); out[out_col] = np.nan
    for k, g in df.groupby(list(by)):
        w = 1 / g[se] ** 2; mu = (w * g[b]).sum() / w.sum()
        tau2 = max(0.0, g[b].var(ddof=1) - (g[se] ** 2).mean()) if len(g) > 1 else 0.0
        out.loc[g.index, out_col] = mu if tau2 == 0 else (g[b] / g[se] ** 2 + mu / tau2) / (1 / g[se] ** 2 + 1 / tau2)
    return out
UE = eb_shrink(UE); UE['q'] = bh(UE.p)
UE['효과(%)'] = (np.exp(UE.b) - 1) * 100; UE['안정화 효과(%)'] = (np.exp(UE.b_eb) - 1) * 100
UE['원화(억)'] = UE.rain_amt * (np.exp(UE.b_eb) - 1) / np.exp(UE.b_eb) / 1e8
R21 = UE.groupby('MCT_SGG_CD').apply(lambda g: pd.Series({
    '비 온 날': int(g.n_rain.max()), '총손실(억)': g['원화(억)'][g['원화(억)'] < 0].sum(), '총이득(억)': g['원화(억)'][g['원화(억)'] > 0].sum(),
    '순변화(억)': g['원화(억)'].sum(), '비 온 날 매출(억)': g.rain_amt.sum() / 1e8,
    '유의 감소(FDR10%)': int(((g.q < .1) & (g.b < 0)).sum()), '유의 증가(FDR10%)': int(((g.q < .1) & (g.b > 0)).sum()), '유닛': len(g)}), include_groups=False)
R21['순변화율(%)'] = R21['순변화(억)'] / R21['비 온 날 매출(억)'] * 100
R21['상쇄율(%)'] = R21['총이득(억)'] / -R21['총손실(억)'] * 100
R21.rename(index=SHORT).round(1).to_csv(os.path.join(OUT, '2-1_원화피해.csv'), encoding='utf-8-sig')
print(R21.rename(index=SHORT).round(1).T.to_string())
R21b = UE.groupby(['MCT_SGG_CD', '분야'])['원화(억)'].sum().unstack(0).rename(columns=SHORT).round(1)
print('\n분야별 순변화(억)\n', R21b.reindex(SECTORS).to_string())
print('\n손실 상위 유닛\n', UE.sort_values('원화(억)').head(10)[['MCT_SGG_CD', 'MCT_RY_CD', '효과(%)', '안정화 효과(%)', '원화(억)', 'q']].round(2).to_string(index=False))
UE.round(4).to_csv(os.path.join(OUT, '2-1_유닛별_비효과.csv'), index=False, encoding='utf-8-sig')

## 2-2. 사건별 최대피해율과 누적손실 (지역·유형·분야)
사건 전 수준 = 사건 전 7일 중 비 안 온 날의 평균 abn. 최대피해율 = 사건 중 최저 abn − 사건 전 수준. 누적손실 = 비 온 날 + 그친 뒤 5일(다음 비 전까지)의 (abn − 사건 전 수준) 합, 하루 매출 대비 %.

In [ ]:
def event_metrics(df, keys):
    rows = []
    for key, data in df.groupby(keys):
        sgg = key[0]
        g = data.sort_values('date').set_index('date')
        sd = max(float(g.sd_calm.iloc[0]), .03)
        events = EVENTS[EVENTS.MCT_SGG_CD.eq(sgg)].sort_values('start').reset_index(drop=True)
        for i, e in events.iterrows():
            nxt = events.start.iloc[i + 1] if i + 1 < len(events) else g.index.max() + pd.Timedelta(days=1)
            pre = g.loc[e.start - pd.Timedelta(days=7):e.start - pd.Timedelta(days=1)]
            pre = pre[pre.use & ~pre.rain & pre.abn.notna()]
            during = g.loc[e.start:e.end]
            during = during[during.use & during.rain & during.abn.notna()]
            if len(pre) < 3 or during.empty:
                continue
            level = pre.abn.mean()
            last = min(e.end + pd.Timedelta(days=8), nxt - pd.Timedelta(days=1), g.index.max())
            dates = pd.date_range(e.end + pd.Timedelta(days=1), last)
            post = g.reindex(dates).copy()
            valid = post.use.fillna(False).astype(bool) & post.abn.notna()
            dev = (post.abn - level).where(valid)
            ok = dev.ge(-sd) & valid
            first_confirmed = next((int((dates[j] - e.end).days) for j in range(len(dates) - 1)
                if (dates[j] - e.end).days <= 7 and ok.iloc[j] and ok.iloc[j+1]), np.nan)
            # 앞에 관찰 불가 날짜가 있으면 정확한 최초 회복일은 알 수 없다.
            exact = np.isfinite(first_confirmed) and bool(valid.iloc[:int(first_confirmed)+1].all())
            recovery = first_confirmed if exact else np.nan
            first3 = dev.reindex(pd.date_range(e.end + pd.Timedelta(days=1), periods=3))
            first5 = dev.reindex(pd.date_range(e.end + pd.Timedelta(days=1), periods=5))
            within2 = 1. if np.isfinite(recovery) and recovery <= 2 else (0. if first3.notna().all() else np.nan)
            if exact:
                status = 'observed_exact_recovery'
            elif np.isfinite(first_confirmed):
                status = 'interval_censored_before_confirmed_recovery'
            elif len(dates) == 8 and valid.all():
                status = 'not_recovered_within_7days'
            elif last < e.end + pd.Timedelta(days=8):
                status = 'censored_next_event_or_data_end'
            else:
                status = 'missing_or_excluded_followup'
            during_loss = (during.abn - level).sum()
            rows.append({'MCT_SGG_CD': sgg, '그룹': key[1], 'event': e.event,
                'start': e.start, '비 일수': len(during),
                '최대피해율(%)': np.expm1(during.abn.min() - level) * 100,
                '피해 사건': during.abn.min() - level < -sd,
                '회복일수': recovery, '최초확인회복일': first_confirmed,
                '회복관찰상태': status, '이틀내회복여부': within2,
                '관찰일수': int(valid.sum()), '달력추적일수': len(dates),
                '회복수준_1~3일(%)': np.expm1(first3.mean()) * 100 if first3.notna().all() else np.nan,
                '여파_1~5일합(%)': first5.sum() * 100 if first5.notna().all() else np.nan,
                '누적손실(하루매출%)': (during_loss + first5.sum()) * 100 if first5.notna().all() else np.nan})
    return pd.DataFrame(rows)

EM = event_metrics(S, ['MCT_SGG_CD', '그룹'])
EM.round(3).to_csv(os.path.join(OUT, '2-2_사건별_지표.csv'), index=False, encoding='utf-8-sig')
R22 = EM.groupby(['MCT_SGG_CD', '그룹']).agg(사건=('event', 'size'),
    피해사건비율=('피해 사건', 'mean'), 최대피해율=('최대피해율(%)', 'mean'),
    누적손실=('누적손실(하루매출%)', 'mean')).reset_index()
R22['MCT_SGG_CD'] = R22.MCT_SGG_CD.map(SHORT)
R22['피해사건비율'] *= 100
print(R22.round(2).to_string(index=False))


# 3단계. 회복 분석
사건 종료일부터 실제 달력 날짜를 셉니다. 관측 가능한 연속 이틀이 필요합니다.
결측·제외일 전에 회복했을 가능성을 배제할 수 없으면 정확한 최초 회복일을 확정하지 않습니다.
회복일 평균, 이틀 내 판정 가능 사례, 관찰 중단을 함께 보고합니다.


In [ ]:
fe = EM[EM['피해 사건']]
R31 = fe.groupby(['MCT_SGG_CD', '그룹']).agg(
    피해사건=('event', 'size'), 회복일수_평균=('회복일수', 'mean'),
    정확회복관측사건=('회복일수', 'count'),
    이틀내회복=('이틀내회복여부', lambda s: s.mean() * 100),
    이틀내판정가능사건=('이틀내회복여부', 'count'),
    회복일_미확정=('회복일수', lambda s: s.isna().mean() * 100),
    회복수준=('회복수준_1~3일(%)', 'mean'), 여파합=('여파_1~5일합(%)', 'mean')).reset_index()
R31['MCT_SGG_CD'] = R31.MCT_SGG_CD.map(SHORT)
R31.round(3).to_csv(os.path.join(OUT, '3-1_회복_요약.csv'), index=False, encoding='utf-8-sig')
print(R31.round(2).to_string(index=False))
print('평균은 정확 회복일이 관측된 사건 기준이며, 미회복·검열을 제외합니다. 전체의 평균 회복시간으로 일반화하지 않습니다.')

# 종료일 기준 경과일 더미. 사흘 연속 비 뒤 첫 맑은 날은 +1일 하나만 켜진다.
rows = []
for (sgg, grp), g in S.groupby(['MCT_SGG_CD', '그룹']):
    g = g.sort_values('date').reset_index(drop=True)
    ex = pd.DataFrame({'당일': g.rain.astype(float), '+1일': 0., '+2일': 0., '+3일': 0.}, index=g.index)
    events = EVENTS[EVENTS.MCT_SGG_CD.eq(sgg)].sort_values('start').reset_index(drop=True)
    for i, e in events.iterrows():
        nxt = events.start.iloc[i+1] if i+1 < len(events) else pd.Timestamp('2026-01-01')
        for day in [1, 2, 3]:
            date = e.end + pd.Timedelta(days=day)
            if date < nxt:
                ex.loc[g.date.eq(date) & ~g.rain, f'+{day}일'] = 1.
    use = g.use
    m = fit(g.y[use], calendar_X(g)[use].join(g[['heat', 'trop']][use].astype(float)).join(ex[use]))
    for col in ex:
        rows.append({'지역': SHORT[sgg], '그룹': grp, '시점': col, **pct(m, col)})
R32 = pd.DataFrame(rows)
R32.round(3).to_csv(os.path.join(OUT, '3-2_그친뒤_여파.csv'), index=False, encoding='utf-8-sig')
print(R32[['지역', '그룹', '시점', '효과', 'p']].round(3).to_string(index=False))


# 4단계. 기후 회복탄력성 지수 (목차 7)
**질문**: 기후 정보를 더한 지수(Model B)가 기존 지수(Model A)보다 "다음 비의 피해"를 더 잘 맞히는가?

| 모델 | 구성 (모두 유닛 = 업종 × 지역, 높을수록 취약) |
|---|---|
| A 기존(구조) | 평소 변동성, 외지 고객 비중, 오후(12~17시) 매출 집중, 규모(작을수록 취약). 날씨 정보 없음 |
| C 기후만 | 비 민감도(과거 비 사건의 피해), 회복 지연(그친 뒤 1~3일), 더위 기울기(1℃당) |
| B 기후 회복탄력성 | A와 C를 반씩 |

**검증**: 지역별로 비 사건 하나를 빼고 나머지 사건으로 B·C를 만든 뒤, 뺀 사건에서 실제로 많이 빠진 유닛을 맞혔는지 본다(사건 단위 교차검증). 지표: 순위상관, 상위 20% 적중률.

**구버전 구분:** 이 단계의 A·C·B와 `4-2_최종지수.csv`는 높을수록 취약한 비교용 점수다. 보고서와 공식 상권에 사용하는 최종 회복탄력성 지수는 부록 B의 `5-5_회복탄력성지수_v2.csv`이며, 낮을수록 취약하다.

**검증 범위:** 이 단계는 전체기간 기준선을 사용하는 사후 사건 비교입니다. 순차 예측과 혼동하지 않습니다.


In [ ]:
# 유닛 × 사건 매출 이탈표: dmg/after는 감소 시 음수, 보고서 피해 D는 감소 시 양수
rows = []
for uid, g in SU.groupby('uid'):
    sgg = g.MCT_SGG_CD.iloc[0]; g = g.sort_values('date').set_index('date')
    evs = EVENTS[EVENTS.MCT_SGG_CD.eq(sgg)].sort_values('start').reset_index(drop=True)
    for i, e in evs.iterrows():
        nxt = evs.start.iloc[i + 1] if i + 1 < len(evs) else pd.Timestamp('2026-01-01')
        pre = g.loc[e.start - pd.Timedelta(days=7): e.start - pd.Timedelta(days=1)]
        pre = pre[pre.use & ~pre.rain & pre.abn.notna()]
        dur = g.loc[e.start:e.end]; dur = dur[dur.use & dur.rain & dur.abn.notna()]
        if len(pre) < 3 or len(dur) == 0: continue
        L = pre.abn.mean()
        post = g.loc[e.end + pd.Timedelta(days=1): min(e.end + pd.Timedelta(days=3), nxt - pd.Timedelta(days=1))]
        post = post[post.use & post.abn.notna()]
        rows.append({'uid': uid, 'MCT_SGG_CD': sgg, 'event': e.event, 'dmg': dur.abn.mean() - L,
                     'after': post.abn.mean() - L if len(post) else np.nan})
UD = pd.DataFrame(rows).merge(units[['uid', '분야', 'mask_risk']], on='uid')
# 평소 흔들림으로 나눈 표준화 피해: 작고 출렁이는 유닛의 잡음이 순위를 지배하지 않게 한다
UD = UD.merge(SU.groupby('uid').sd_calm.first().rename('sd'), left_on='uid', right_index=True)
UD['dmg_raw'] = UD.dmg; UD['dmg'] = UD.dmg / UD.sd; UD['after'] = UD.after / UD.sd
# 보고서 7-2절: D = (사건 전 7일 평균 abn − 비 온 날 평균 abn) / σ
# 피해가 클수록 D가 커진다. 증가한 사건은 음수로 보존한다.
UD['damage_D'] = -UD['dmg']
UD['damage_D_raw'] = -UD['dmg_raw']
print('유닛 × 사건 표:', UD.shape, UD.groupby('MCT_SGG_CD').event.nunique().to_dict())

# 구조 지표 (Model A): 비 온 날을 뺀 평범한 날로만 계산
calm = SU[SU.use & ~SU.rain]
A_raw = calm.groupby('uid').agg(변동성=('abn', 'std'), 규모=('amt', 'mean'))
p = SU[~SU.amt.isna()].groupby('uid')[['amt_res', 'amt_out', 'tb_12_17', 'amt']].sum()
A_raw['외지비중'] = p.amt_out / (p.amt_res + p.amt_out)
A_raw['오후집중'] = p.tb_12_17 / p.amt
A_raw['소규모'] = -np.log(A_raw.규모)
A_COLS = ['변동성', '외지비중', '오후집중', '소규모']

# 더위 기울기 (7~9월, 1℃당 로그매출 변화, 음수일수록 취약)
hs = {}
for uid, g in SU[SU.use & SU.month.isin([7, 8, 9])].groupby('uid'):
    m = fit(g.y, calendar_X(g).join(rain_bin_X(g)).join(g[['ta_max']]))
    hs[uid] = m.params.get('ta_max', np.nan)
A_raw['더위기울기'] = pd.Series(hs)

def z(s):
    s = pd.Series(s, copy=False).replace([np.inf, -np.inf], np.nan)
    sd = s.std()
    return (s - s.mean()) / sd if np.isfinite(sd) and sd > 1e-12 else s.where(s.isna(), 0.)
def shrink_mean(df, col, k=3.0):
    """유닛 평균을 같은 지역·분야 평균 쪽으로 당김 (사건이 적을수록 더 많이)"""
    g = df.groupby('uid').agg(v=(col, 'mean'), n=(col, 'count')).join(units.set_index('uid')[['MCT_SGG_CD', '분야']])
    mu = g.groupby(['MCT_SGG_CD', '분야']).v.transform('mean')
    return (g.n * g.v + k * mu) / (g.n + k)

def scores(ud_train, uids):
    """학습 사건으로 A·C·B 점수 계산 (지역 안에서 표준화)"""
    f = A_raw.loc[uids].copy()
    f['비민감도'] = shrink_mean(ud_train, 'damage_D').reindex(uids)  # D가 클수록 취약; 추가 부호 반전 없음
    f['회복지연'] = -shrink_mean(ud_train.dropna(subset=['after']), 'after').reindex(uids)
    f['더위취약'] = -f.더위기울기
    f = f.fillna(f.median())
    zA = sum(z(f[c]) for c in A_COLS) / len(A_COLS)
    zC = (z(f.비민감도) + z(f.회복지연) + z(f.더위취약)) / 3
    return pd.DataFrame({'A': z(zA), 'C': z(zC), 'B': z(0.5 * z(zA) + 0.5 * z(zC))}, index=uids), f

def top_hit(score, target, frac=.2):
    pair = pd.DataFrame({'score': score, 'target': target}).dropna()
    if len(pair) == 0 or pair.score.nunique() < 2 or pair.target.nunique() < 2:
        return np.nan
    count = max(1, int(round(len(pair) * frac)))
    def membership(values):
        threshold = values.nlargest(count).iloc[-1]
        high, tied = values.gt(threshold), values.eq(threshold)
        return high.astype(float) + tied.astype(float) * (count - high.sum()) / tied.sum()
    return float((membership(pair.score) * membership(pair.target)).sum() / count)

rows = []
for sgg in REG:
    uids = units.uid[units.MCT_SGG_CD.eq(sgg)].values
    ud_r = UD[UD.MCT_SGG_CD.eq(sgg)]
    eval_uids = units.uid[units.MCT_SGG_CD.eq(sgg) & units.mask_risk.ne('high')].values   # 검증은 마스킹 위험 high 제외
    for ev in ud_r.event.unique():
        test = ud_r[ud_r.event.eq(ev)].set_index('uid').dmg.reindex(eval_uids).dropna()
        if len(test) < 10: continue
        sc, _ = scores(ud_r[ud_r.event.ne(ev)], uids)
        target = -test                                   # 많이 빠질수록 큼
        r = {'지역': SHORT[sgg], 'event': ev, '유닛': len(test)}
        for mname in ['A', 'C', 'B']:
            s = sc[mname].reindex(target.index)
            r[f'{mname}_순위상관'] = stats.spearmanr(s, target)[0]
            r[f'{mname}_상위20%적중'] = top_hit(s, target)
        rows.append(r)
CV = pd.DataFrame(rows); CV.round(3).to_csv(os.path.join(OUT, '4-1_사건교차검증.csv'), index=False, encoding='utf-8-sig')

def summarize(df):
    out = {}
    for mname in ['A', 'C', 'B']:
        out[f'{mname} 순위상관'] = df[f'{mname}_순위상관'].mean()
        out[f'{mname} 상위20%적중(%)'] = df[f'{mname}_상위20%적중'].mean() * 100
    d = df.B_순위상관 - df.A_순위상관
    out['B−A 순위상관 차이'] = d.mean(); out['B>A 사건 비율(%)'] = (d > 0).mean() * 100
    out['B−A 차이 p(부호순위)'] = safe_wilcoxon(d)
    out['사건 수'] = len(df)
    return pd.Series(out)
R41 = pd.concat({'전체': summarize(CV), **{k: summarize(g) for k, g in CV.groupby('지역')}}, axis=1)
R41.round(3).to_csv(os.path.join(OUT, '4-1_지수검증_요약.csv'), encoding='utf-8-sig')
print(R41.round(3).to_string())

## 4-2. 최종 지수와 순위 안정성
모든 사건으로 최종 점수를 만들고, 사건을 무작위로 다시 뽑아(부트스트랩 200회) 다시 만들었을 때 순위가 얼마나 유지되는지 본다.

In [ ]:
final = []; feats = []
for sgg in REG:
    uids = units.uid[units.MCT_SGG_CD.eq(sgg)].values; ud_r = UD[UD.MCT_SGG_CD.eq(sgg)]
    sc, f = scores(ud_r, uids); final.append(sc); feats.append(f)
    evs = ud_r.event.unique(); rng = np.random.default_rng(7); stab = []
    for _ in range(200):
        pick = rng.choice(evs, len(evs))
        ub = pd.concat([ud_r[ud_r.event.eq(e)] for e in pick])
        stab.append(stats.spearmanr(scores(ub, uids)[0].B, sc.B)[0])
    print(f'{SHORT[sgg]} 순위 안정성(B, 사건 부트스트랩 순위상관): 중앙 {np.median(stab):.2f}, 5% {np.percentile(stab, 5):.2f}')
FIN = pd.concat(final).join(pd.concat(feats)).join(units.set_index('uid')[['MCT_SGG_CD', 'MCT_RY_CD', '분야', '유형', 'mask_risk']])
FIN['지수(0~100)'] = FIN.groupby('MCT_SGG_CD').B.rank(pct=True) * 100

# 유형: 비 민감도 × 회복 지연 (지역 안 중앙값 기준) + 구조 취약
def ftype(r, med):
    hi_s, hi_r = r.비민감도 > med.비민감도, r.회복지연 > med.회복지연
    if hi_s and hi_r: return '① 민감·지연형'
    if hi_s: return '② 민감·회복형'
    if r.A > 0.5: return '③ 구조 취약형'
    return '④ 안정형'
FIN['유형(지원)'] = None
for sgg in REG:
    k = FIN.MCT_SGG_CD.eq(sgg); med = FIN[k][['비민감도', '회복지연']].median()
    FIN.loc[k, '유형(지원)'] = FIN[k].apply(lambda r: ftype(r, med), axis=1)
FIN['등급'] = pd.cut(FIN['지수(0~100)'], [0, 60, 80, 100.01], labels=['관찰', '주의', '우선지원'], include_lowest=True).astype(str)
raw = UD.assign(after_raw=UD.after * UD.sd)
FIN['비 피해(%)'] = (np.exp(raw.groupby('uid').dmg_raw.mean()) - 1).reindex(FIN.index) * 100
FIN['그친뒤 1~3일(%)'] = (np.exp(raw.groupby('uid').after_raw.mean()) - 1).reindex(FIN.index) * 100
FIN = FIN.merge(UE.set_index('uid')[['원화(억)']], left_index=True, right_index=True, how='left')
FIN.round(3).to_csv(os.path.join(OUT, '4-2_최종지수.csv'), encoding='utf-8-sig')
cols = ['MCT_RY_CD', '분야', '지수(0~100)', '등급', '유형(지원)', '비 피해(%)', '그친뒤 1~3일(%)', '외지비중', '원화(억)', 'mask_risk']
for sgg in REG:
    print(f'\n[{SHORT[sgg]}] 지수 상위 12')
    print(FIN[FIN.MCT_SGG_CD.eq(sgg)].sort_values('B', ascending=False)[cols].head(12).round(2).to_string(index=False))
print('\n유형 × 등급 (유닛 수)\n', pd.crosstab([FIN.MCT_SGG_CD.map(SHORT), FIN['유형(지원)']], FIN.등급).to_string())

# 그림: 검증 결과
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
for ax, met, lab in [(axes[0], '순위상관', '다음 비 피해와의 순위상관'), (axes[1], '상위20%적중', '상위 20% 적중률')]:
    for i, reg in enumerate(['강남', '춘천']):
        s = CV[CV.지역.eq(reg)]
        vals = [s[f'{mm}_{met}'].mean() * (100 if met == '상위20%적중' else 1) for mm in 'ACB']
        ax.bar(np.arange(3) + (i - .5) * .38, vals, width=.38, color=[COL[GN], COL[CC]][i], label=reg)
    if met == '상위20%적중': ax.axhline(20, color='k', ls=':', lw=1)
    ax.axhline(0, color='k', lw=.6); ax.set_xticks(range(3)); ax.set_xticklabels(['A 구조', 'C 기후만', 'B 기후 회복탄력성'])
    ax.set_title(lab); ax.legend()
fig.suptitle('사건 단위 교차검증: 남은 사건으로 만든 지수가 뺀 사건의 피해를 맞히는가 (점선 = 무작위 20%)', fontsize=10)
savefig('4-1_지수검증')

---
# 추가 분석 (부록 A~I)
여기부터는 위 메인 분석의 결과(메모리에 있는 표와 변수)를 이어서 쓴다. 메인을 먼저 끝까지 실행해야 한다.

## 부록 A. 기후 기록과 전망 (ERA5 1981~2025, CMIP6 2041~2050)
보고서 4-6절(기후 추세에도 잡히지 않는다)과 5-3절(연간 손실 시나리오)의 재료. Open-Meteo에서 무료로 내려받는다(인증키 불필요).

In [ ]:
# 기후 기록은 ERA5 고정, 모든 시간 강수는 직전 1시간 합계(10~22시 -> 09~22시).
import urllib.request, urllib.parse, urllib.error, time, json
CLI = os.path.join(WORK, 'climate'); os.makedirs(CLI, exist_ok=True)
SITES = {'chuncheon': (37.9026, 127.7357), 'gangnam': (37.5134, 127.0469)}
MODELS = 'CMCC_CM2_VHR4,FGOALS_f3_H,HiRAM_SIT_HR,MRI_AGCM3_2_S,EC_Earth3P_HR,MPI_ESM1_2_XR,NICAM16_8S'
CLIMATE_SOURCE_VERSION = 'era5_fixed_business10-22_v2'


def get_climate(url, path):
    """URL을 확인할 수 없는 이전 캐시는 재사용하지 않는다. 실패한 응답은 저장하지 않는다."""
    meta_path = path + '.request.json'
    if os.path.exists(path) and os.path.exists(meta_path):
        try:
            meta = json.load(open(meta_path, encoding='utf-8'))
            data = json.load(open(path, encoding='utf-8'))
            if meta.get('url') == url and not data.get('error'):
                return data
        except (ValueError, OSError):
            pass
    failure = None
    for attempt in range(3):
        try:
            with urllib.request.urlopen(url, timeout=90) as response:
                data = json.loads(response.read())
            if data.get('error'):
                raise ValueError(data.get('reason', 'API error'))
            tmp_path = path + '.partial'
            with open(tmp_path, 'w', encoding='utf-8') as stream:
                json.dump(data, stream, ensure_ascii=False)
            os.replace(tmp_path, path)
            with open(meta_path, 'w', encoding='utf-8') as stream:
                json.dump({'url': url, 'source_version': CLIMATE_SOURCE_VERSION,
                           'collected_at': pd.Timestamp.now(tz='Asia/Seoul').isoformat()}, stream, ensure_ascii=False, indent=2)
            return data
        except (OSError, ValueError, urllib.error.HTTPError) as exc:
            failure = str(exc)
            if attempt < 2:
                time.sleep(3 * (attempt + 1))
    raise RuntimeError('기후 자료 수집 실패: ' + str(failure))


def era_daily(data):
    t = pd.to_datetime(data['hourly']['time'])
    d = pd.DataFrame({'t': t, 'rn': pd.to_numeric(pd.Series(data['hourly']['precipitation']), errors='coerce')})
    d['date'] = (d.t - pd.Timedelta(minutes=1)).dt.normalize()
    d = d[d.t.dt.hour.between(10, 22)].copy()
    d['rn'] = d.rn.where(d.rn >= 0)
    day = d.groupby('date').agg(rn_biz=('rn', lambda x: x.sum(min_count=13)),
                                observed_hours=('rn', 'count'))
    day['wet_h'] = (d.rn >= .5).groupby(d.date).sum().where(day.observed_hours.eq(13))
    day['complete'] = day.observed_hours.eq(13)
    return day


def occurrence_scores(observed, estimated):
    pairs = pd.DataFrame({'obs': observed, 'est': estimated}).dropna()
    ob, es = pairs.obs.ge(RAIN_BIZ_MM), pairs.est.ge(RAIN_BIZ_MM)
    tp, fp, fn, tn = int((ob & es).sum()), int((~ob & es).sum()), int((ob & ~es).sum()), int((~ob & ~es).sum())
    recall = tp / (tp + fn) if tp + fn else np.nan
    precision = tp / (tp + fp) if tp + fp else np.nan
    return {'paired_days': len(pairs), 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
            'precision': precision, 'recall': recall,
            'f1': 2 * precision * recall / (precision + recall) if precision + recall > 0 else np.nan,
            'overall_accuracy_descriptive_only': (tp + tn) / len(pairs) if len(pairs) else np.nan,
            'observed_rain_days': int(ob.sum()), 'era5_rain_days': int(es.sum()),
            'count_ratio': float(ob.sum() / es.sum()) if es.sum() else np.nan}


def rolling_variability_test(yearly, window=10, block=3, repetitions=999):
    """겹치는 이동 표준편차를 다시 계산하는 블록 순열 검정.

    10년 이동값 자체를 독립 표본으로 취급하지 않는다. 3년 블록 교환가능성 가정의
    탐색 검정이며, 45년은 분산 추세를 확정하기에 짧다는 한계를 남긴다.
    """
    series = yearly.dropna().sort_index()
    if len(series) < window + 5 or np.any(np.diff(series.index.to_numpy(int)) != 1):
        return {'status': 'insufficient_or_nonconsecutive_years', 'slope_per_decade': np.nan, 'p_block': np.nan}, pd.Series(dtype=float)
    values = series.to_numpy(float)
    rolled = series.rolling(window, min_periods=window).std(ddof=1).dropna()
    x = rolled.index.to_numpy(float)
    slope = float(np.polyfit(x - x.mean(), rolled.to_numpy(), 1)[0])
    blocks = [values[i:i+block] for i in range(0, len(values), block)]
    rng = np.random.default_rng(42)
    count = 0
    for _ in range(repetitions):
        shuffled = np.concatenate([blocks[i] for i in rng.permutation(len(blocks))])
        s = pd.Series(shuffled).rolling(window, min_periods=window).std(ddof=1).dropna().to_numpy()
        alt = np.polyfit(x - x.mean(), s, 1)[0]
        count += abs(alt) >= abs(slope)
    return {'status': 'ok', 'slope_per_decade': slope * 10,
            'p_block': (count + 1) / (repetitions + 1), 'window_years': window,
            'block_years': block, 'years': len(series)}, rolled


annual_rows, calibration_rows, trend_rows, rolling_rows = {}, [], [], []
for site, (lat, lon) in SITES.items():
    params = {'latitude': lat, 'longitude': lon, 'start_date': '1981-01-01', 'end_date': '2025-12-31',
              'hourly': 'precipitation', 'models': 'era5', 'timezone': 'Asia/Seoul'}
    url = 'https://archive-api.open-meteo.com/v1/archive?' + urllib.parse.urlencode(params)
    data = get_climate(url, os.path.join(CLI, f'era5_{site}.json'))
    daily = era_daily(data)
    sgg = CC if site == 'chuncheon' else GN
    observed = wx[wx.MCT_SGG_CD.eq(sgg) & wx.wx_ok].set_index('date').rn_biz
    occurrence = occurrence_scores(observed, daily.rn_biz.reindex(observed.index))
    calibration_rows.append({'site': site, 'MCT_SGG_CD': sgg, **occurrence})
    print(site, '비 온 날 정밀도/재현율:', occurrence['precision'], occurrence['recall'])
    y = daily.assign(year=daily.index.year).groupby('year').agg(
        rain5=('rn_biz', lambda x: int(x.ge(RAIN_BIZ_MM).sum())),
        wet_h=('wet_h', 'sum'), long_rain=('wet_h', lambda x: int(x.ge(6).sum())),
        observed_days=('complete', 'sum'))
    expected_days = pd.Series({year: len(pd.date_range(f'{year}-01-01', f'{year}-12-31')) for year in y.index})
    y['coverage'] = y.observed_days / expected_days
    y.loc[y.coverage.lt(.99), ['rain5', 'wet_h', 'long_rain']] = np.nan
    y.index.name = 'date'; annual_rows[site] = y
    for variable in ['rain5', 'wet_h', 'long_rain']:
        value = y[variable].dropna()
        slope = stats.theilslopes(value.to_numpy(), value.index.to_numpy())
        _, p = stats.kendalltau(value.index, value)
        trend_rows.append({'site': site, 'metric': variable, 'slope_per_decade': slope[0] * 10,
                           'p_MK_descriptive': p, 'years': len(value)})
    test, rolling = rolling_variability_test(y.rain5)
    trend_rows.append({'site': site, 'metric': 'rain5_rolling10_std', **test})
    for year, value in rolling.items():
        rolling_rows.append({'site': site, 'year': year, 'rolling10_std': value})

pd.concat(annual_rows, names=['site']).to_csv(os.path.join(CLI, 'era5_annual.csv'), encoding='utf-8-sig')
CALIBRATION = pd.DataFrame(calibration_rows)
CALIBRATION.to_csv(os.path.join(CLI, 'era5_observation_calibration.csv'), index=False, encoding='utf-8-sig')
TRENDS = pd.DataFrame(trend_rows)
mask = TRENDS.metric.eq('rain5_rolling10_std')
TRENDS.loc[mask, 'q_block_two_regions'] = bh(TRENDS.loc[mask, 'p_block'])
TRENDS.to_csv(os.path.join(CLI, 'climate_trends.csv'), index=False, encoding='utf-8-sig')
pd.DataFrame(rolling_rows).to_csv(os.path.join(CLI, 'rain5_rolling10_std.csv'), index=False, encoding='utf-8-sig')
print(TRENDS.round(4).to_string(index=False))
print('변동성 증가가 유의하지 않으면 증가 주장 없이 현재 기후 적응 문제로 설명합니다.')

# 일강수 전망은 영업시간 강수 전망으로 변환하지 않는다. 시간 해상도가 다르다.
projection_rows = []
for site, (lat, lon) in SITES.items():
    params = {'latitude': lat, 'longitude': lon, 'start_date': '1981-01-01', 'end_date': '2050-12-31',
              'models': MODELS, 'daily': 'precipitation_sum'}
    url = 'https://climate-api.open-meteo.com/v1/climate?' + urllib.parse.urlencode(params)
    data = get_climate(url, os.path.join(CLI, f'cmip6_{site}.json'))
    d = pd.DataFrame(data['daily']); d['time'] = pd.to_datetime(d.time); d = d.set_index('time')
    for col in d.columns:
        model_daily_values = d[col].dropna(); grouped = model_daily_values.groupby(model_daily_values.index.year)
        n5 = grouped.apply(lambda s: s.ge(5).sum()); n30 = grouped.apply(lambda s: s.ge(30).sum())
        projection_rows.append({'site': site, 'model': col, 'resolution': 'daily_total_not_business_hours',
            'daily5_change_pct': (n5.loc[2041:2050].mean() / n5.loc[1991:2020].mean() - 1) * 100,
            'daily30_change_pct': (n30.loc[2041:2050].mean() / n30.loc[1991:2020].mean() - 1) * 100})
pd.DataFrame(projection_rows).to_csv(os.path.join(CLI, 'cmip6_daily_scenarios.csv'), index=False, encoding='utf-8-sig')
print('CMIP6는 일 단위입니다. 09~22시 5mm 위험의 미래 증가를 직접 검증한 자료가 아닙니다.')


## 부록 B. 기후 회복탄력성 지수 v2, 재난 기준별 손실, 연간 손실 (보고서 4-6·5-3·7장)
비민감도 중심 지수(가중치는 중첩 교차검증으로 선택), 맑은 날 위약 검정, 최종 지수·우선지원 매트릭스.

### 보고서 7-2절과 같은 지수 방향
- 사건 피해 `damage_D = (사건 전 7일 평균 abn − 비 온 날 평균 abn) / σ`: 클수록 큰 피해.
- 비민감도는 `damage_D`를 같은 지역·분야 평균으로 안정화한 값이며, 다시 마이너스를 붙이지 않는다.
- 최종 회복탄력성 지수는 `100 − 취약 백분위`: 높을수록 강하고 낮을수록 우선지원 대상이다.
- 기존 점수를 유지하는 백분위 규칙은 `100 × (평균순위 − 1) / 지역별 업종 수`다. 동점은 평균순위를 사용한다.
- 회귀·검증용 매출 이탈 `dmg`, `dmg_raw`, `after`는 감소 시 음수인 원래 정의를 유지하고, 지수 학습에는 명시적인 양수 피해 `damage_D`를 사용한다.

**성능 해석:** 양의 상관 사건 비율은 정확도가 아닙니다. 이 중첩 검증은 사후 비교이며 10-*의 순차 검증과 평가 목표가 다릅니다.


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats

# ---------------------------------------------------------------- 5-1 관측소 수치 기준별 감소 추정(실제 특보 이력 아님)
def rolling_max(tm, rn):
    d = pd.DataFrame({'t': tm, 'rn': rn.astype(float)}).set_index('t').sort_index()
    d = d.rn.resample('h').sum(min_count=1)
    out = pd.DataFrame({'r3': d.rolling(3, min_periods=3).sum(), 'r12': d.rolling(12, min_periods=12).sum(), 'r1': d})
    out['date'] = (out.index - pd.Timedelta(minutes=1)).normalize()
    return out.groupby('date').agg(r1=('r1', 'max'), r3=('r3', 'max'), r12=('r12', 'max')).reindex(DAYS)
RM = pd.concat({GN: rolling_max(h.TM, h.RN_HR1), CC: rolling_max(hc.TM, hc.RN.mask(hc.RN.eq(-9), 0.).where(hc.RN.ge(0) | hc.RN.eq(-9)))},
               names=['MCT_SGG_CD', 'date']).reset_index()
core = S[S.그룹.eq('핵심 합계')].merge(RM, on=['MCT_SGG_CD', 'date']).merge(wx[['MCT_SGG_CD', 'date', 'rn_day']], on=['MCT_SGG_CD', 'date'])
eff = R11[R11.그룹.eq('핵심 합계')].assign(MCT_SGG_CD=lambda d: d.지역.map({v: k for k, v in SHORT.items()}))
eff = eff.set_index(['MCT_SGG_CD', '강수(mm)']).효과 / 100
core = core[core.use & core.rn_biz.gt(0)].copy()
core['e'] = [eff.get((s, b), np.nan) for s, b in zip(core.MCT_SGG_CD, core.rain_bin)]
core['loss'] = -core.amt * core.e / (1 + core.e) / 1e8                 # 그날 사라진 매출(억, 양수 = 손실)
core['기준'] = np.select([(core.r3 >= 60) | (core.r12 >= 110), core.rn_day >= 30, core.rain],
                        ['호우주의보 수준(3시간 60mm·12시간 110mm)', '일강수 30mm 이상(주의보 미만)', '영업시간 5mm 이상(특보·30mm 미만)'],
                        '영업시간 0~5mm')
R51 = core.groupby(['MCT_SGG_CD', '기준']).agg(일수=('date', 'size'), 손실억=('loss', 'sum')).reset_index()
R51['손실비중(%)'] = R51.손실억 / R51.groupby('MCT_SGG_CD').손실억.transform('sum') * 100
R51['MCT_SGG_CD'] = R51.MCT_SGG_CD.map(SHORT)
R51.round(2).to_csv(os.path.join(OUT, '5-1_재난기준별_손실.csv'), index=False, encoding='utf-8-sig')
print(R51.round(1).to_string(index=False))
heat_days = wx[wx.use].groupby('MCT_SGG_CD').heat.sum().rename(index=SHORT)
print('폭염일(일최고 33℃ 이상) 수:', heat_days.to_dict())

# ---------------------------------------------------------------- 5-2 연간 손실 시나리오
ea = pd.read_csv(os.path.join(WORK, 'climate', 'era5_annual.csv'), encoding='utf-8-sig')
calib = CALIBRATION.set_index('MCT_SGG_CD').count_ratio.to_dict()
rows = []
for sgg, site in [(GN, 'gangnam'), (CC, 'chuncheon')]:
    y = ea[ea.site.eq(site)].set_index('date').rain5
    n_obs = int(wx[wx.MCT_SGG_CD.eq(sgg) & wx.use].rain.sum())
    per_day = -R21.loc[sgg, '순변화(억)'] / R21.loc[sgg, '비 온 날']
    for lab, n in [('1981~2025 평균', y.mean()), ('최근 10년 평균(2016~2025)', y.loc[2016:2025].mean()),
                   ('2020년', y.loc[2020]), ('2023년', y.loc[2023]), ('45년 최대', y.max())]:
        rows.append({'지역': SHORT[sgg], '시나리오': lab, 'ERA5 비 온 날': round(n, 1), '관측 기준 환산 일수': round(n * calib[sgg], 1),
                     '비 온 날 1일 손실(억)': round(per_day, 2), '연간 손실(억)': round(per_day * n * calib[sgg], 0), '가정': '하반기 하루 손실 × 연간 강수일; 계절효과 고정 시나리오'})
R52 = pd.DataFrame(rows); R52.to_csv(os.path.join(OUT, '5-2_연간손실_시나리오.csv'), index=False, encoding='utf-8-sig')
print(R52.to_string(index=False))

# ---------------------------------------------------------------- 5-3 지수 v2
UD['dmg_std'] = UD.dmg
UD['tot'] = UD.dmg + UD.after.fillna(0)          # 종합 피해 = 당일 + 그친 뒤 1~3일 (표준화)
EXPO = ['오후집중', '외지비중']                 # 비 노출 구조 (체질 중 비에 노출되는 부분만)
INDEX_WEIGHT_CANDIDATES = {'비민감도만': (1, 0, 0), '저항 .75 + 노출 .25': (.75, 0, .25), '저항 .5 + 노출 .5': (.5, 0, .5),
         '저항 .75 + 회복 .25': (.75, .25, 0), '저항 .5 + 회복 .5': (.5, .5, 0),
         '저항 .5 + 회복 .25 + 노출 .25': (.5, .25, .25), '저항 .6 + 회복 .2 + 노출 .2': (.6, .2, .2), '균등 1/3': (1/3, 1/3, 1/3)}

def index_v2_features(ud_train, uids):
    f = A_raw.loc[uids, ['변동성', '외지비중', '오후집중', '소규모']].copy()
    f['비민감도'] = shrink_mean(ud_train, 'damage_D').reindex(uids)  # D가 클수록 취약; 추가 부호 반전 없음
    f['회복지연'] = -shrink_mean(ud_train.dropna(subset=['after']), 'after').reindex(uids)
    f = f.fillna(f.median())
    f['노출'] = (z(f.오후집중) + z(f.외지비중)) / 2
    return f

def vuln(f, w):          # 취약 점수 (높을수록 약함)
    ws, wr, we = w
    components = [(ws, f.비민감도), (wr, f.회복지연), (we, f.노출)]
    active = [weight * z(values) for weight, values in components if weight != 0]
    return z(sum(active)) if active else pd.Series(np.nan, index=f.index)

def v1_scores(f):        # 비교용: v1의 A(구조)와 B
    zA = sum(z(f[c]) for c in ['변동성', '외지비중', '오후집중', '소규모']) / 4
    zC = (z(f.비민감도) + z(f.회복지연) + z(-A_raw.loc[f.index, '더위기울기'].fillna(A_raw.더위기울기.median()))) / 3
    return z(zA), z(0.5 * z(zA) + 0.5 * z(zC))

def rho(s, t):
    k = t.dropna().index.intersection(s.index); return stats.spearmanr(s[k], t[k])[0] if len(k) >= 10 else np.nan

def loeo(ud_r, uids, eval_uids, cands, target='tot'):
    """사건 하나 빼기: 각 후보의 평균 순위상관"""
    res = {k: [] for k in cands}
    for ev in ud_r.event.unique():
        test = -ud_r[ud_r.event.eq(ev)].set_index('uid')[target].reindex(eval_uids)
        if test.notna().sum() < 10: continue
        f = index_v2_features(ud_r[ud_r.event.ne(ev)], uids)
        for k, w in cands.items(): res[k].append(rho(vuln(f, w), test))
    return {k: np.nanmean(v) for k, v in res.items()}

rows, chosen = [], []
for sgg in REG:
    uids = units.uid[units.MCT_SGG_CD.eq(sgg)].values
    eval_uids = units.uid[units.MCT_SGG_CD.eq(sgg) & units.mask_risk.ne('high')].values
    ud_r = UD[UD.MCT_SGG_CD.eq(sgg)]
    for ev in ud_r.event.unique():                       # 바깥: 평가 사건
        test_tot = -ud_r[ud_r.event.eq(ev)].set_index('uid').tot.reindex(eval_uids)
        test_dmg = -ud_r[ud_r.event.eq(ev)].set_index('uid').dmg.reindex(eval_uids)
        test_aft = -ud_r[ud_r.event.eq(ev)].set_index('uid').after.reindex(eval_uids)
        if test_tot.notna().sum() < 10: continue
        train = ud_r[ud_r.event.ne(ev)]
        inner = loeo(train, uids, eval_uids, INDEX_WEIGHT_CANDIDATES)       # 안쪽: 남은 사건만으로 가중치 선택
        valid_inner = {name: value for name, value in inner.items() if np.isfinite(value)}
        best = max(valid_inner, key=valid_inner.get) if valid_inner else '비민감도만'; chosen.append({'지역': SHORT[sgg], 'event': ev, '선택': best})
        f = index_v2_features(train, uids); v2 = vuln(f, INDEX_WEIGHT_CANDIDATES[best]); A1, B1 = v1_scores(f)
        for nm, s in [('v2 (중첩CV)', v2), ('v1 A 구조', A1), ('v1 B', B1), ('비민감도만', vuln(f, INDEX_WEIGHT_CANDIDATES['비민감도만']))]:
            rows.append({'지역': SHORT[sgg], 'event': ev, '지수': nm, '종합피해 ρ': rho(s, test_tot), '당일피해 ρ': rho(s, test_dmg),
                         '상위20% 적중': top_hit(s.reindex(test_tot.dropna().index), test_tot.dropna())})
        rows.append({'지역': SHORT[sgg], 'event': ev, '지수': '회복지연 단독(→그친 뒤 여파)', '종합피해 ρ': np.nan,
                     '당일피해 ρ': rho(f.회복지연, test_aft), '상위20% 적중': np.nan})
CV2 = pd.DataFrame(rows); CH = pd.DataFrame(chosen)
CV2.round(4).to_csv(os.path.join(OUT, '5-3_지수v2_중첩교차검증_사건별.csv'), index=False, encoding='utf-8-sig')
def s2(g):
    d = {'종합피해 ρ': g['종합피해 ρ'].mean(), '당일피해 ρ': g['당일피해 ρ'].mean(), '상위20% 적중(%)': g['상위20% 적중'].mean() * 100,
         '양의 상관 사건(%)': (g.loc[g['종합피해 ρ'].notna(), '종합피해 ρ'] > 0).mean() * 100 if g['종합피해 ρ'].notna().any() else np.nan}
    for r in ['강남', '춘천']: d[f'{r} 종합 ρ'] = g[g.지역 == r]['종합피해 ρ'].mean()
    return pd.Series(d)
R53 = CV2.groupby('지수').apply(s2, include_groups=False)
base = CV2[CV2.지수.eq('v1 A 구조')].set_index(['지역', 'event'])['종합피해 ρ']
for nm in ['v2 (중첩CV)', 'v1 B', '비민감도만']:
    d = CV2[CV2.지수.eq(nm)].set_index(['지역', 'event'])['종합피해 ρ'] - base
    R53.loc[nm, 'A 대비 p(부호순위)'] = safe_wilcoxon(d)
R53.round(3).to_csv(os.path.join(OUT, '5-3_지수v2_중첩교차검증_요약.csv'), encoding='utf-8-sig')
print(R53.round(3).to_string()); print(CH.선택.value_counts().to_string())

# 최종 가중치: 사건 전부로 같은 방식(사건 하나 빼기) 선택
FINAL_W = {}
for sgg in REG:
    uids = units.uid[units.MCT_SGG_CD.eq(sgg)].values
    eval_uids = units.uid[units.MCT_SGG_CD.eq(sgg) & units.mask_risk.ne('high')].values
    sc = loeo(UD[UD.MCT_SGG_CD.eq(sgg)], uids, eval_uids, INDEX_WEIGHT_CANDIDATES)
    FINAL_W[sgg] = max({k: v for k, v in sc.items() if np.isfinite(v)}, key=lambda k: sc[k]) if any(np.isfinite(v) for v in sc.values()) else '비민감도만'; print(SHORT[sgg], '후보별 ρ', {k: round(v, 3) for k, v in sc.items()})
# 두 지역에 같은 공식을 쓰기 위해, 두 지역 평균 ρ가 가장 높은 후보를 최종으로
both = {k: np.mean([loeo(UD[UD.MCT_SGG_CD.eq(s)], units.uid[units.MCT_SGG_CD.eq(s)].values,
                         units.uid[units.MCT_SGG_CD.eq(s) & units.mask_risk.ne('high')].values, {k: w})[k] for s in REG]) for k, w in INDEX_WEIGHT_CANDIDATES.items()}
valid_both = {k: v for k, v in both.items() if np.isfinite(v)}
W_NAME = max(valid_both, key=valid_both.get) if valid_both else '비민감도만'; W = INDEX_WEIGHT_CANDIDATES[W_NAME]
print('최종 가중치:', W_NAME, W, '| 두 지역 평균 ρ', {k: round(v, 3) for k, v in both.items()})

# ---------------------------------------------------------------- 5-4 위약 검정 (최종 공식)
rng = np.random.default_rng(11); rows = []
for sgg in REG:
    uids = units.uid[units.MCT_SGG_CD.eq(sgg)].values
    eval_uids = units.uid[units.MCT_SGG_CD.eq(sgg) & units.mask_risk.ne('high')].values
    w_ = wx[wx.MCT_SGG_CD.eq(sgg)].set_index('date').sort_index()
    wet = w_.index[w_.rain.fillna(False).astype(bool) | (w_.rn_day.fillna(0) >= 1)]
    g_all = SU[SU.MCT_SGG_CD.eq(sgg)]
    ud_days = g_all.groupby('date').use.mean(); ud_days = ud_days[ud_days > .9].index
    cand = [d for d in ud_days if not any(abs((d - x).days) <= 3 for x in wet)]
    pick = sorted(rng.choice(cand, size=min(30, len(cand)), replace=False))
    piv = g_all.pivot_table(index='date', columns='uid', values='abn')
    okm = g_all.assign(ok=g_all.use & ~g_all.rain).pivot_table(index='date', columns='uid', values='ok', aggfunc='first')
    sd = g_all.groupby('uid').sd_calm.first()
    f = index_v2_features(UD[UD.MCT_SGG_CD.eq(sgg)], uids); v2 = vuln(f, W); A1, B1 = v1_scores(f)
    for d in pick:
        pre = piv.loc[d - pd.Timedelta(days=7): d - pd.Timedelta(days=1)]
        pre = pre.where(okm.reindex_like(pre).fillna(False).astype(bool))
        L = pre.mean().where(pre.count() >= 3)
        tgt = -((piv.loc[d] - L) / sd).reindex(eval_uids).dropna()
        for nm, s in [('v2 최종', v2), ('v1 A 구조', A1), ('v1 B', B1)]:
            rows.append({'지역': SHORT[sgg], '지수': nm, 'ρ': rho(s, tgt), '적중': top_hit(s.reindex(tgt.index), tgt)})
PL = pd.DataFrame(rows)
R54 = PL.groupby('지수').agg(위약_ρ=('ρ', 'mean'), 위약_적중=('적중', lambda x: x.mean() * 100))
R54['위약 ρ≠0 p'] = PL.groupby('지수').ρ.apply(lambda x: safe_wilcoxon(x))
R54.round(3).to_csv(os.path.join(OUT, '5-4_지수v2_위약검정.csv'), encoding='utf-8-sig'); print(R54.round(3).to_string())

# ---------------------------------------------------------------- 5-5 최종 지수, 유형, 안정성
fin, stab_rows = [], []
for sgg in REG:
    uids = units.uid[units.MCT_SGG_CD.eq(sgg)].values; ud_r = UD[UD.MCT_SGG_CD.eq(sgg)]
    f = index_v2_features(ud_r, uids); v = vuln(f, W)
    evs = ud_r.event.unique(); rr = np.random.default_rng(7); st = []
    for _ in range(200):
        ub = pd.concat([ud_r[ud_r.event.eq(e)] for e in rr.choice(evs, len(evs))])
        st.append(stats.spearmanr(vuln(index_v2_features(ub, uids), W), v)[0])
    stab_rows.append({'지역': SHORT[sgg], '순위 안정성 중앙': np.median(st), '하위 5%': np.percentile(st, 5)})
    f['취약점수'] = v; fin.append(f)
F = pd.concat(fin).join(units.set_index('uid')[['MCT_SGG_CD', 'MCT_RY_CD', '분야', '유형', 'mask_risk']])
# 보고서의 지수 = 100 − 취약 백분위: 큰 피해 → 큰 취약 백분위 → 낮은 회복탄력성
# 기존 점수 눈금 보존: 평균순위 r, 지역별 업종 수 n에 대해 백분위 = 100 × (r − 1) / n
# 동점은 평균순위 사용. 동점이 없으면 회복탄력성 범위는 100/n~100이다.
vulnerability_rank = F.groupby('MCT_SGG_CD').취약점수.rank(method='average')
region_unit_count = F.groupby('MCT_SGG_CD').취약점수.transform('size')
identified = F.groupby('MCT_SGG_CD').취약점수.transform('nunique').gt(1)
F.loc[~identified, '취약점수'] = np.nan
vulnerability_rank = vulnerability_rank.where(identified)
F['취약 백분위(0~100)'] = 100 * (vulnerability_rank - 1) / region_unit_count
F['회복탄력성 지수'] = 100 - F['취약 백분위(0~100)']
F['회복탄력성 지수'] = F['회복탄력성 지수'].clip(upper=100).round(1)
F['등급'] = pd.cut(F['회복탄력성 지수'], [-1, 20, 40, 101], labels=['우선지원', '주의', '양호']).astype(str)
def typ(r, med):
    lo_res, lo_rec = r.비민감도 > med.비민감도, r.회복지연 > med.회복지연     # 값이 클수록 약함
    return {(True, True): '① 취약형(많이 빠지고 늦게 회복)', (True, False): '② 충격형(많이 빠지지만 빨리 회복)',
            (False, True): '③ 지연형(덜 빠지지만 늦게 회복)', (False, False): '④ 강건형'}[(lo_res, lo_rec)]
F['유형'] = None
for sgg in REG:
    k = F.MCT_SGG_CD.eq(sgg); med = F[k][['비민감도', '회복지연']].median()
    F.loc[k, '유형'] = F[k].apply(lambda r: typ(r, med), axis=1)
F['신뢰도'] = np.where(F.mask_risk.eq('high') | (F.변동성 > F.groupby('MCT_SGG_CD').변동성.transform(lambda s: s.quantile(.9))), '낮음(소규모·잡음)', '보통')
raw = UD.assign(after_raw=UD.after * UD.sd)
F['비 온 날 매출(%)'] = (np.exp(raw.groupby('uid').dmg_raw.mean()) - 1).reindex(F.index) * 100
F['그친 뒤 1~3일(%)'] = (np.exp(raw.groupby('uid').after_raw.mean()) - 1).reindex(F.index) * 100
F = F.join(UE.set_index('uid')[['원화(억)']])
F.round(3).to_csv(os.path.join(OUT, '5-5_회복탄력성지수_v2.csv'), encoding='utf-8-sig')
ST = pd.DataFrame(stab_rows); ST.round(3).to_csv(os.path.join(OUT, '5-5_순위안정성.csv'), index=False, encoding='utf-8-sig')
print(ST.round(2).to_string(index=False))
cols = ['MCT_RY_CD', '회복탄력성 지수', '등급', '유형', '비 온 날 매출(%)', '그친 뒤 1~3일(%)', '원화(억)', '신뢰도']
for sgg in REG:
    print(f'\n[{SHORT[sgg]}] 회복탄력성 하위 12'); print(F[F.MCT_SGG_CD.eq(sgg)].sort_values('회복탄력성 지수')[cols].head(12).round(1).to_string(index=False))
print(pd.crosstab([F.MCT_SGG_CD.map(SHORT), F.유형], F.등급).to_string())

# 우선지원 매트릭스 (신뢰도 낮음 제외)
M = F[F.신뢰도.eq('보통')].copy(); M['손실'] = -M['원화(억)'].clip(upper=0)
rows = []
for sgg in REG:
    s = M[M.MCT_SGG_CD.eq(sgg)]; q = s.손실.quantile(.8)
    for _, r in s.iterrows():
        lo, big = r['회복탄력성 지수'] <= 20, r.손실 >= q
        rows.append({'지역': SHORT[sgg], '업종': r.MCT_RY_CD, '지수': r['회복탄력성 지수'], '손실(억)': round(r.손실, 1), '유형': r.유형,
                     '구분': '1순위(약하고 손실 큼)' if lo and big else '선별 지원(약하지만 손실 작음)' if lo else '규모 지원(강하지만 손실 큼)' if big else '-'})
MX = pd.DataFrame(rows); MX.to_csv(os.path.join(OUT, '5-6_우선지원_매트릭스.csv'), index=False, encoding='utf-8-sig')
print(MX[MX.구분.ne('-')].sort_values(['지역', '구분', '지수']).to_string(index=False))

# 그림 1: v2 검증
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
order = ['v1 A 구조', 'v1 B', 'v2 (중첩CV)']
lab = ['기존 체질 지표\n(v1 A)', '첫 지수\n(v1 B)', '회복탄력성 지수\n(v2)']
ax = axes[0]; vals = R53.reindex(order)['종합피해 ρ']
pv = R54.reindex(['v1 A 구조', 'v1 B', 'v2 최종'])['위약_ρ'].values
x = np.arange(3); ax.bar(x - .19, vals, .38, color='#2F6F8F', label='실제 비 사건'); ax.bar(x + .19, pv, .38, color='#BBBBBB', label='맑은 날(위약)')
ax.axhline(0, color='k', lw=.8); ax.set_xticks(x); ax.set_xticklabels(lab); ax.set_title('사후 사건 검증의 종합 피해 순위상관'); ax.legend(fontsize=8)
ax = axes[1]; hv = R53.reindex(order)['상위20% 적중(%)']
ax.bar(x, hv, .5, color=['#BBBBBB', '#9DB8C7', '#2F6F8F']); ax.axhline(20, color='k', ls=':', lw=1)
for xx, v in zip(x, hv): ax.text(xx, v + 1, f'{v:.0f}%', ha='center')
ax.set_xticks(x); ax.set_xticklabels(lab); ax.set_title('가장 크게 다칠 20%를 맞힌 비율 (점선 = 무작위 20%)')
fig.suptitle('사건 하나를 빼고 만든 지수로 그 사건의 피해를 맞혀 본 결과 (가중치도 남은 사건만으로 선택)', fontsize=10)
plt.tight_layout(); plt.savefig(os.path.join(FIG, '5-3_지수v2_검증.png'), dpi=160, bbox_inches='tight'); (plt.show() if IN_COLAB else None); plt.close('all')
print('\n완료')

## 부록 C. 강수량별 분석 (보고서 4-3·7-4절)
세분 구간별 반응, 강수량 2배당 효과, 작은 비·큰 비 사건별 지수 예측력.

In [ ]:
import matplotlib
from scipy import stats

# 6-1 세분 구간: 비 없는 날 대비
FB = [-0.01, 0, 2, 5, 10, 20, 40, 1e9]; FL = ['0', '0~2', '2~5', '5~10', '10~20', '20~40', '40+']
rows = []
for (sgg, grp), g in each_series(S[S.그룹.isin(['핵심 합계', '방문·선택형', '생활밀착형'])]):
    g = g.copy(); g['fb'] = pd.cut(g.rn_biz, FB, labels=FL).astype(str)
    D = pd.get_dummies(pd.Categorical(g.fb, categories=FL), prefix='b', dtype=float).set_index(g.index).drop(columns='b_0')
    m = fit(g.y, calendar_X(g).join(D).join(g[['heat', 'trop']].astype(float)))
    for lab in FL[1:]:
        rows.append({'지역': SHORT[sgg], '그룹': grp, '영업시간 강수(mm)': lab, '일수': int((g.fb == lab).sum()), **pct(m, 'b_' + lab)})
R61 = pd.DataFrame(rows); R61.round(3).to_csv(os.path.join(OUT, '6-1_강수량_세분구간.csv'), index=False, encoding='utf-8-sig')
print(R61.assign(v=R61.효과.round(1).astype(str) + np.where(R61.p < .05, '*', '') + ' (' + R61.일수.astype(str) + '일)')
      .pivot_table(index=['지역', '그룹'], columns='영업시간 강수(mm)', values='v', aggfunc='first')[FL[1:]].to_string())

# 6-2 강수량이 2배가 될 때 (비 온 날만, log2 강수량), 5mm 이상 날만도 따로
rows = []
for (sgg, grp), g in each_series(S[S.그룹.isin(['핵심 합계', '방문·선택형', '생활밀착형'])]):
    for lab, w in [('비 온 날 전체(>0mm)', g[g.rn_biz > 0]), ('5mm 이상 날만', g[g.rn_biz >= 5])]:
        X = calendar_X(w).join(pd.DataFrame({'log2강수': np.log2(w.rn_biz)}, index=w.index)).join(w[['heat', 'trop']].astype(float))
        rows.append({'지역': SHORT[sgg], '그룹': grp, '표본': lab, '일수': len(w), **pct(fit(w.y, X), 'log2강수')})
R62 = pd.DataFrame(rows); R62.round(3).to_csv(os.path.join(OUT, '6-2_강수량2배당_효과.csv'), index=False, encoding='utf-8-sig')
print(R62.round(2).to_string(index=False))

# 6-3 사건 크기별 지수 예측력: 사건 하나 빼고 만든 비민감도 지수 → 그 사건 피해 순위상관
evrain = EVD.merge(wx[['MCT_SGG_CD', 'date', 'rn_biz']], on=['MCT_SGG_CD', 'date']).groupby('event').agg(일평균강수=('rn_biz', 'mean'), 일수=('date', 'size'), 총강수=('rn_biz', 'sum'))
rows = []
for sgg in REG:
    eval_uids = units.uid[units.MCT_SGG_CD.eq(sgg) & units.mask_risk.ne('high')].values
    ud_r = UD[UD.MCT_SGG_CD.eq(sgg)]
    for ev in ud_r.event.unique():
        test = -ud_r[ud_r.event.eq(ev)].set_index('uid').dmg.reindex(eval_uids).dropna()
        if len(test) < 10: continue
        sc = shrink_mean(ud_r[ud_r.event.ne(ev)], 'damage_D').reindex(test.index)  # 높을수록 취약
        rows.append({'지역': SHORT[sgg], 'event': ev, 'ρ': stats.spearmanr(sc, test)[0], **evrain.loc[ev].to_dict()})
R63 = pd.DataFrame(rows)
R63['사건 크기'] = np.where(R63.일평균강수 >= R63.groupby('지역').일평균강수.transform('median'), '큰 비 사건', '작은 비 사건')   # 지역 안 중앙값 기준
R63.round(3).to_csv(os.path.join(OUT, '6-3_사건크기별_지수예측력.csv'), index=False, encoding='utf-8-sig')
print(R63.groupby('사건 크기').agg(사건=('ρ', 'size'), 평균ρ=('ρ', 'mean'), 일평균강수=('일평균강수', 'mean')).round(3).to_string())
print('강수량과 예측력의 순위상관:', round(stats.spearmanr(R63.일평균강수, R63.ρ)[0], 3), 'p', round(stats.spearmanr(R63.일평균강수, R63.ρ)[1], 3))

# 6-4 약한 비 사건과 센 비 사건에서 업종 피해 순위가 같은가
rows = []
for sgg in REG:
    ud_r = UD[UD.MCT_SGG_CD.eq(sgg) & UD.mask_risk.ne('high')].merge(R63[['event', '사건 크기']], on='event')
    p = ud_r.groupby(['uid', '사건 크기']).dmg.mean().unstack()
    r = stats.spearmanr(p['작은 비 사건'], p['큰 비 사건'], nan_policy='omit')
    rows.append({'지역': SHORT[sgg], '업종 수': int(p.dropna().shape[0]), '작은 비 vs 큰 비 피해 순위상관': r[0], 'p': r[1]})
R64 = pd.DataFrame(rows); R64.round(3).to_csv(os.path.join(OUT, '6-4_약한비vs센비_순위일치.csv'), index=False, encoding='utf-8-sig')
print(R64.round(3).to_string(index=False))

## 부록 D. 첫 지수(v1) 진단: 피해 정의별 비교, 위약 검정, 예측 상한 (보고서 부록 그림 2)

In [ ]:
import matplotlib
from scipy import stats
RES = OUT

UD['dmg_std'] = UD.dmg                          # 표준화 피해 (평소 흔들림으로 나눔) = 본 분석의 검증 목표
TARGETS = {'표준화': 'dmg_std', '원값': 'dmg_raw'}
C3 = ['비민감도', '회복지연', '더위취약']
CFG = {'A 구조(현행)': (A_COLS, C3, 1.0), 'B 현행(0.5/0.5)': (A_COLS, C3, .5), 'C 기후(현행)': (A_COLS, C3, 0.0),
       'B w=0.75': (A_COLS, C3, .75), 'B w=0.25': (A_COLS, C3, .25),
       'C 더위 제외': (A_COLS, ['비민감도', '회복지연'], 0.0), '비민감도만': (A_COLS, ['비민감도'], 0.0),
       'A 변동성 제외': (['외지비중', '오후집중', '소규모'], C3, 1.0),
       '오후집중만': (['오후집중'], C3, 1.0), '변동성만': (['변동성'], C3, 1.0)}

def diagnostic_features(ud_train, uids, tcol):
    f = A_raw.loc[uids].copy()
    damage_col = {'dmg_std': 'damage_D', 'dmg_raw': 'damage_D_raw'}[tcol]
    f['비민감도'] = shrink_mean(ud_train, damage_col).reindex(uids)  # 양수 피해 D로 학습
    f['회복지연'] = -shrink_mean(ud_train.dropna(subset=['after']), 'after').reindex(uids)
    f['더위취약'] = -f.더위기울기
    return f.fillna(f.median())

def build(f, ac, cc, w):
    zA = sum(z(f[c]) for c in ac) / len(ac); zC = sum(z(f[c]) for c in cc) / len(cc)
    return z(zA) if w == 1 else (z(zC) if w == 0 else z(w * z(zA) + (1 - w) * z(zC)))

rows, ceil = [], []
rng = np.random.default_rng(11)
for sgg in REG:
    uids = units.uid[units.MCT_SGG_CD.eq(sgg)].values
    eval_uids = units.uid[units.MCT_SGG_CD.eq(sgg) & units.mask_risk.ne('high')].values
    ud_r = UD[UD.MCT_SGG_CD.eq(sgg)]
    # 위약일: 앞뒤 3일 안에 비(영업시간 5mm 이상 또는 일강수 1mm 이상)가 없고 특수일이 아닌 날 30일
    w = wx[wx.MCT_SGG_CD.eq(sgg)].set_index('date').sort_index()
    wet = w.index[w.rain.fillna(False).astype(bool) | (w.rn_day.fillna(0) >= 1)]
    g_all = SU[SU.MCT_SGG_CD.eq(sgg)]
    use_days = g_all.groupby('date').use.mean(); use_days = use_days[use_days > .9].index
    cand = [d for d in use_days if not any(abs((d - x).days) <= 3 for x in wet)]
    pick = sorted(rng.choice(cand, size=min(30, len(cand)), replace=False))
    piv = g_all.pivot_table(index='date', columns='uid', values='abn')
    okm = g_all.assign(ok=g_all.use & ~g_all.rain).pivot_table(index='date', columns='uid', values='ok', aggfunc='first')
    sd = g_all.groupby('uid').sd_calm.first()
    for tname, tcol in TARGETS.items():
        # 실제 비: 사건 하나를 빼고 나머지로 지수를 만들어 뺀 사건의 피해를 맞힘
        for ev in ud_r.event.unique():
            test = ud_r[ud_r.event.eq(ev)].set_index('uid')[tcol].reindex(eval_uids).dropna()
            if len(test) < 10: continue
            f = diagnostic_features(ud_r[ud_r.event.ne(ev)], uids, tcol); tgt = -test
            for k, v in CFG.items():
                s = build(f, *v).reindex(tgt.index)
                rows.append({'목표': tname, '지역': SHORT[sgg], '종류': '실제 비', 'event': ev, '구성': k,
                             'rho': stats.spearmanr(s, tgt)[0], 'hit': top_hit(s, tgt)})
        # 위약: 모든 비 사건으로 만든 지수가 맑은 날의 '피해'를 맞히는지
        f = diagnostic_features(ud_r, uids, tcol); sc = {k: build(f, *v) for k, v in CFG.items()}
        for d in pick:
            pre = piv.loc[d - pd.Timedelta(days=7): d - pd.Timedelta(days=1)]
            pre = pre.where(okm.reindex_like(pre).fillna(False).astype(bool))
            L = pre.mean().where(pre.count() >= 3)
            dm = piv.loc[d] - L
            if tcol == 'dmg_std': dm = dm / sd
            tgt = -dm.reindex(eval_uids).dropna()
            if len(tgt) < 10: continue
            for k, s in sc.items():
                rows.append({'목표': tname, '지역': SHORT[sgg], '종류': '위약(맑은 날)', 'event': str(d.date()), '구성': k,
                             'rho': stats.spearmanr(s.reindex(tgt.index), tgt)[0], 'hit': top_hit(s.reindex(tgt.index), tgt)})
        # 예측 상한: 서로 다른 두 사건의 피해 순위 상관 r → 학습 사건 k개 평균으로 사건 하나를 맞힐 때의 기대 상한
        M = ud_r[ud_r.mask_risk.ne('high')].pivot_table(index='uid', columns='event', values=tcol)
        cc = M.corr(method='spearman').values; r = np.nanmean(cc[np.triu_indices_from(cc, 1)]); k = M.shape[1] - 1
        ceil.append({'목표': tname, '지역': SHORT[sgg], '사건 수': M.shape[1], '사건 간 순위 일관성 r': r,
                     '예측 상한(Spearman-Brown)': r * np.sqrt(k) / np.sqrt(1 + (k - 1) * r) if r > 0 else np.nan})

R = pd.DataFrame(rows).dropna(subset=['rho'])
R.round(4).to_csv(os.path.join(RES, '4-3_지수진단_사건별.csv'), index=False, encoding='utf-8-sig')

# 요약 1: 실제 비 사건 교차검증 (지역별·전체)
def s1(g):
    out = {}
    for reg in ['전체', '강남', '춘천']:
        h = g if reg == '전체' else g[g.지역 == reg]
        out[f'순위상관_{reg}'] = h.rho.mean(); out[f'상위20%적중_{reg}(%)'] = h.hit.mean() * 100
    out['양의 상관 사건(%)'] = (g.rho > 0).mean() * 100
    out['순위상관>0 p'] = safe_wilcoxon(g.rho)
    return pd.Series(out)
S1 = R[R.종류 == '실제 비'].groupby(['목표', '구성']).apply(s1, include_groups=False)

# 요약 2: 실제 비 vs 위약 (비 특이 예측력)
def s2(g):
    a, b = g[g.종류 == '실제 비'], g[g.종류 == '위약(맑은 날)']
    return pd.Series({'실제 순위상관': a.rho.mean(), '위약 순위상관': b.rho.mean(), '비 특이(실제−위약)': a.rho.mean() - b.rho.mean(),
                      '실제>위약 p(Mann-Whitney)': stats.mannwhitneyu(a.rho, b.rho, alternative='greater').pvalue,
                      '위약 순위상관≠0 p': safe_wilcoxon(b.rho),
                      '실제 상위20%적중(%)': a.hit.mean() * 100, '위약 상위20%적중(%)': b.hit.mean() * 100})
S2 = R.groupby(['목표', '구성']).apply(s2, include_groups=False)

CE = pd.DataFrame(ceil)
bm = R[(R.종류 == '실제 비') & (R.구성 == '비민감도만')].groupby(['목표', '지역']).rho.mean()
CE['비민감도 순위상관'] = [bm.get((t, g)) for t, g in zip(CE.목표, CE.지역)]
CE['주의'] = '가정하의 신뢰도 진단; 엄밀한 성능 상한이나 77~93% 달성률로 사용하지 않음'

S1.round(3).to_csv(os.path.join(RES, '4-3_지수진단_구성비교.csv'), encoding='utf-8-sig')
S2.round(4).to_csv(os.path.join(RES, '4-3_지수진단_위약검정.csv'), encoding='utf-8-sig')
CE.round(3).to_csv(os.path.join(RES, '4-3_지수진단_예측상한.csv'), index=False, encoding='utf-8-sig')
pd.set_option('display.width', 250)
print('상한 대비 77~93% 주장은 사용하지 않습니다. 중첩 성능과 혼용하지 않습니다.'); print(S1.round(3).to_string()); print(S2.round(3).to_string()); print(CE.round(3).to_string(index=False))

## 부록 E. 강건성: 대형 업종(골프장·할인점) 제외, 한 업종 빼기 (보고서 V2 5-1절)
이 단계는 `panel`을 잠시 바꿨다가 되돌린다.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
EV = FIG
PANEL0 = panel.copy()
GOLF, MART = '실내/실외골프장', '할인점/슈퍼마켓/양판점'

def effects(drop):
    """drop 업종을 뺀 패널로 지역·유형 합계를 다시 만들어 비 효과를 추정"""
    global panel
    panel = PANEL0[~PANEL0.MCT_RY_CD.isin(drop)]
    s = pd.concat([series('합계'), series('유형')], ignore_index=True); s['y'] = np.log(s.amt)
    out = {}
    for (sgg, grp), g in each_series(s):
        e = pct(fit(g.y, calendar_X(g).join(g[['rain', 'heat', 'trop']].astype(float))), 'rain')
        out[(SHORT[sgg], grp)] = e
    panel = PANEL0
    return out

SCEN = {'전체': [], '골프장 제외': [GOLF], '할인점·슈퍼 제외': [MART], '골프장·할인점 모두 제외': [GOLF, MART]}
rows = []
for nm, drop in SCEN.items():
    for (reg, grp), e in effects(drop).items():
        rows.append({'시나리오': nm, '지역': reg, '그룹': grp, '효과(%)': e['효과'], '하한': e['하한'], '상한': e['상한'], 'p': e['p']})
R91 = pd.DataFrame(rows); R91.round(3).to_csv(os.path.join(OUT, '9-1_대형업종제외_비효과.csv'), index=False, encoding='utf-8-sig')
pd.set_option('display.width', 220)
print(R91.assign(v=R91['효과(%)'].round(1).astype(str) + ' (p=' + R91.p.round(3).astype(str) + ')')
      .pivot_table(index=['지역', '그룹'], columns='시나리오', values='v', aggfunc='first')[list(SCEN)].to_string())

# 9-2 한 업종 빼기: 지역 핵심 합계 비 효과가 업종 하나에 얼마나 기대는가
rows = []
for sgg in REG:
    for ry in units[units.MCT_SGG_CD.eq(sgg)].MCT_RY_CD:
        global_panel = PANEL0[~(PANEL0.MCT_RY_CD.eq(ry) & PANEL0.MCT_SGG_CD.eq(sgg))]
        panel = global_panel
        s = series('합계'); s['y'] = np.log(s.amt); g = s[s.use & s.MCT_SGG_CD.eq(sgg)].sort_values('date')
        e = pct(fit(g.y, calendar_X(g).join(g[['rain', 'heat', 'trop']].astype(float))), 'rain')
        rows.append({'지역': SHORT[sgg], '뺀 업종': ry, '효과(%)': e['효과'], 'p': e['p']})
    panel = PANEL0
R92 = pd.DataFrame(rows)
base = R91[(R91.시나리오 == '전체') & (R91.그룹 == '핵심 합계')].set_index('지역')['효과(%)']
R92['변화(%p)'] = R92['효과(%)'] - R92.지역.map(base)
R92.round(3).to_csv(os.path.join(OUT, '9-2_한업종빼기.csv'), index=False, encoding='utf-8-sig')
for reg, g in R92.groupby('지역'):
    g = g.sort_values('변화(%p)', ascending=False)
    print(f"[{reg}] 기준 {base[reg]:.2f}% | 한 업종 빼기 범위 {g['효과(%)'].min():.2f} ~ {g['효과(%)'].max():.2f}% | 유의 유지 {int((g.p < .05).sum())}/{len(g)}")
    print(g.head(4)[['뺀 업종', '효과(%)', '변화(%p)', 'p']].round(3).to_string(index=False))

# 9-3 손실: 업종별 원화(2-1)에서 대형 업종을 뺀 몫
UE = pd.read_csv(os.path.join(OUT, '2-1_유닛별_비효과.csv'), encoding='utf-8-sig')
UE['지역'] = UE.MCT_SGG_CD.map({GN: '강남', CC: '춘천'})
rows = []
for reg, g in UE.groupby('지역'):
    tot = g['원화(억)'].sum()
    for nm, drop in SCEN.items():
        k = g[~g.MCT_RY_CD.isin(drop)]
        rows.append({'지역': reg, '시나리오': nm, '순손실(억)': -k['원화(억)'].sum(), '전체 대비(%)': k['원화(억)'].sum() / tot * 100})
R93 = pd.DataFrame(rows); R93.round(1).to_csv(os.path.join(OUT, '9-3_대형업종제외_손실.csv'), index=False, encoding='utf-8-sig')
print(R93.round(1).to_string(index=False))

# 그림 16
plt.rcParams.update({'axes.unicode_minus': False, 'font.size': 10, 'axes.titleweight': 'bold',
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .2})
cols = ['#222222', '#2B6CB0', '#D35F1F', '#8A8A8A']
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), gridspec_kw={'width_ratios': [1.3, 1.3, 1]})
for ax, grp, ttl in [(axes[0], '핵심 합계', '지역 전체 비 효과'), (axes[1], None, '방문·선택형 vs 생활밀착형 (춘천)')]:
    if grp:
        x = np.arange(2)
        for i, nm in enumerate(SCEN):
            s = R91[(R91.시나리오 == nm) & (R91.그룹 == grp)].set_index('지역').reindex(['강남', '춘천'])
            xx = x + (i - 1.5) * .2; ax.bar(xx, s['효과(%)'], .19, color=cols[i], label=nm)
            for a, v, p in zip(xx, s['효과(%)'], s.p): ax.text(a, v - .3, f'{v:.1f}' + ('' if p < .05 else '\n(n.s.)'), ha='center', va='top', fontsize=7.5)
        ax.set_xticks(x); ax.set_xticklabels(['강남', '춘천']); ax.legend(frameon=False, fontsize=8, loc='lower left')
    else:
        x = np.arange(2)
        for i, nm in enumerate(SCEN):
            s = R91[(R91.시나리오 == nm) & (R91.지역 == '춘천')].set_index('그룹').reindex(['방문·선택형', '생활밀착형'])
            xx = x + (i - 1.5) * .2; ax.bar(xx, s['효과(%)'], .19, color=cols[i])
            for a, v in zip(xx, s['효과(%)']): ax.text(a, v - .3, f'{v:.1f}', ha='center', va='top', fontsize=7.5)
        ax.set_xticks(x); ax.set_xticklabels(['방문·선택형', '생활밀착형'])
    ax.axhline(0, color='k', lw=.8); ax.set_ylabel('비 온 날 매출 변화 (%)'); ax.set_title(ttl)
ax = axes[2]
for i, reg in enumerate(['강남', '춘천']):
    g = R92[R92.지역 == reg]
    ax.scatter(np.full(len(g), i) + np.random.default_rng(1).uniform(-.12, .12, len(g)), g['효과(%)'], s=18, color=['#C8553D', '#2B6CB0'][i], alpha=.6)
    ax.hlines(base[reg], i - .25, i + .25, color='k', lw=2)
ax.axhline(0, color='k', lw=.8); ax.set_xticks([0, 1]); ax.set_xticklabels(['강남', '춘천'])
ax.set_title('업종을 하나씩 빼 본 결과 (검은 선 = 전체)'); ax.set_ylabel('지역 전체 비 효과 (%)')
fig.suptitle('손실이 쏠린 대형 업종(골프장·할인점)을 빼도 결론이 유지되는가', fontsize=11)
plt.tight_layout(); plt.savefig(os.path.join(EV, '그림16_대형업종제외_강건성.png'), dpi=170, bbox_inches='tight', facecolor='white'); (plt.show() if IN_COLAB else None); plt.close('all')
print('완료')

# 5단계. 취합 1회차 - 피해 규모 예측·운영 시나리오

**목표:** 업종×비 사건의 과거 기준선 대비 매출 부족액(원)을 예측합니다. 이 값은 독립적으로 관측된 인과적 피해 정답이 아닙니다.

- 입력: 영업시간 강수량, 지속시간, 사건 일수, 요일 구성, 과거 비민감도, 업종 유형·분야, 기준매출.
- 고정 Ridge와 평균 손실·업종별 하루 평균을 동일 fold에서 비교합니다. MAPE는 0·증가 사건에 불안정하므로 MAE·WAPE를 씁니다.
- LOEO에서는 겹치는 지역 사건도 함께 제외합니다. 별도로 과거만 학습한 순차 검증을 제시합니다.
- 기준선은 사건 이전 평상일만 이용합니다. 전체기간 업종 선정 표본을 고정한 한계는 남습니다.
- 장마형 시나리오는 시간당 계수를 로그 매출비에서 결합합니다. 7.8억에 임의 가중치를 곱하지 않습니다.
- 562억은 취합본의 비교용 수치이며 확정 가용 재원이 아닙니다. 업종 제외분도 실제 소상공인 자격 확인값이 아닙니다.

## 4대 수정 반영
1. 지역×사건 기상은 매출 누락과 독립적으로 고정합니다. 완전 관측 사건만 원화 주 평가에 포함하고, 관측 0원은 포함합니다.
2. 회복 시작일은 1~7일, 연속 정상 여부는 최대 8일까지 확인합니다. 다음 강우·자료 끝·결측은 검열 상태로 남깁니다.
3. 최종 지수는 표준화 피해 D와 사후3일 피해의 합을 목표로 삼습니다. 기준선·노출·내부 가중치 선택은 모두 과거 자료만 씁니다. 내부 검증 부족 시 비민감도 단독으로 고정 대체합니다. 사후3일이 끝나기 전 사건은 학습에서 제외합니다. 10-6·10-7과 최종 지수·공식 상권·지원 후보는 같은 산출 절차를 사용합니다. 앞선 5-3~5-5 v2는 사후 비교용입니다.
4. 기준선 자체는 매주 고정한 3일 맑은 구간의 순차 홀드아웃으로 진단합니다. 강수입력 제외 모형과 같은 사건에서 비교하고 업종 단위 MAE·지역×사건 총액 MAE·쌍별 개선 구간을 분리합니다.

최종 지수의 목표는 `D_total = D + D_after3`입니다. 두 항 모두 사건 전 7일 평상일의 로그매출 잔차 평균에서 각 기간 평균을 뺀 뒤 사건 이전 평상일 σ로 나눕니다. 큰 값이 취약합니다. 후보 가중치는 기존 8개를 유지하며 시험 사건을 보지 않고 내부 순차 검증으로 선택합니다. 순차 검증은 가중치 한 번을 고정한 성능이 아니라 **시점마다 과거로 가중치를 선택하는 전체 절차의 성능**입니다. 최종 스냅숏은 자료 다음날 기준입니다. 기준선·평가 표본·사후 관측 완전성 조건도 바뀌었으므로 기존 0.21과 새 값을 바로 비교해 성능 향상이라고 해석하지 않습니다.


In [ ]:
# 취합 1회차: 조건부 매출 부족액 예측 및 사건별 검증. 인과적 피해의 정답은 아니다.
LOSS_MODEL_VERSION = 'round2-2026-10-21'
LOSS_MIN_CALM_DAYS = 21
LOSS_LOOKBACK_DAYS = 84
LOSS_MIN_TRAIN_CLUSTERS = 3
LOSS_RIDGE_ALPHA = 10.0  # 시험 결과로 선택하지 않는 고정값


def loss_calendar_matrix(d):
    """열 정의 고정. 미래 월 더미를 학습하는 대신 연중 주기항을 사용한다."""
    day = pd.to_datetime(d['date'])
    dow = day.dt.dayofweek.to_numpy()
    year_phase = 2 * np.pi * day.dt.dayofyear.to_numpy() / 365.25
    cols = [np.ones(len(d))] + [(dow == k).astype(float) for k in range(1, 7)]
    cols += [np.sin(year_phase), np.cos(year_phase)]
    for name in ['month_end', 'school_vac', 'festival', 'heat', 'trop']:
        cols.append(d[name].astype(float).to_numpy() if name in d else np.zeros(len(d)))
    return np.column_stack(cols)


def loss_fit_ridge(X, y, alpha=LOSS_RIDGE_ALPHA, weights=None):
    X, y = np.asarray(X, float), np.asarray(y, float)
    if not np.isfinite(X).all() or not np.isfinite(y).all() or not len(y):
        raise ValueError('회귀 입력은 유한한 관측이어야 합니다.')
    mean, scale = X.mean(axis=0), X.std(axis=0)
    mean[0], scale[0] = 0., 1.
    scale[scale < 1e-10] = 1.
    Z = (X - mean) / scale
    w = np.ones(len(y)) if weights is None else np.asarray(weights, float)
    if not np.isfinite(w).all() or (w < 0).any() or w.sum() <= 0:
        raise ValueError('학습 가중치 오류')
    w = w * len(w) / w.sum()
    penalty = np.eye(Z.shape[1]) * alpha
    penalty[0, 0] = 0.
    coef = np.linalg.lstsq(Z.T @ (w[:, None] * Z) + penalty, Z.T @ (w * y), rcond=None)[0]
    return {'mean': mean, 'scale': scale, 'coef': coef}


def loss_predict_ridge(model, X):
    return ((np.asarray(X, float) - model['mean']) / model['scale']) @ model['coef']


def loss_shared_clusters(events):
    """두 지역에서 겹치거나 하루 이내 이어진 강우는 같은 평가 블록으로 묶는다."""
    ordered = events.sort_values(['start', 'end', 'event']).copy()
    last_end, cluster = None, 0
    labels = {}
    for e in ordered.itertuples():
        if last_end is None or e.start > last_end + pd.Timedelta(days=1):
            cluster += 1
        labels[e.event] = f'weather_{cluster:02d}'
        last_end = e.end if last_end is None else max(last_end, e.end)
    return labels


def loss_reference(tr, rows):
    """과거 평상일의 양수 매출로 적합한 공통 기준선. 평가 매출은 사용하지 않는다."""
    X = loss_calendar_matrix(tr)
    model = loss_fit_ridge(X, np.log(tr.amt.to_numpy()))
    residual = np.log(tr.amt.to_numpy()) - loss_predict_ridge(model, X)
    smearing = float(np.exp(residual).mean())
    pred = np.exp(loss_predict_ridge(model, loss_calendar_matrix(rows))) * smearing
    return pred, model, residual


def build_loss_event_table(sales, events, rain_hours, weather):
    """기상 사건은 지역 기상표에서 고정하고 매출 완전 관측 사건만 평가한다.

    관측 0원은 원화 목표에 포함한다. 결측·제외일을 0원으로 대체하지 않는다.
    표준화 지수 목표는 로그가 필요한 양수 매출과 사건 후 3일이 관측될 때만 정의한다.
    """
    records, audit = [], []
    clusters = loss_shared_clusters(events)
    hour_lookup = rain_hours.set_index(['MCT_SGG_CD', 'date'])
    if hour_lookup.index.duplicated().any() or weather.duplicated(['MCT_SGG_CD', 'date']).any():
        raise ValueError('기상표 지역×날짜 중복을 먼저 해결하세요.')
    by_unit = {uid: g.sort_values('date').set_index('date', drop=False)
               for uid, g in sales.groupby('uid')}
    for e in events.sort_values('start').itertuples():
        w = weather[weather.MCT_SGG_CD.eq(e.MCT_SGG_CD)].set_index('date')
        window = w.reindex(pd.date_range(e.start, e.end))
        weather_ok = window.rain.notna() & window.rn_biz.notna()
        if 'wx_ok' in window:
            weather_ok &= window.wx_ok.fillna(False).astype(bool)
        event_weather_ok = bool(weather_ok.all())
        ed = window[window.rain.fillna(False).astype(bool)].copy()
        dates = ed.index
        hh = hour_lookup.reindex(pd.MultiIndex.from_arrays([[e.MCT_SGG_CD] * len(dates), dates]))
        duration_ok = bool(hh.wet_h.notna().all()) and len(dates) > 0
        fixed = {'n_days': len(dates), 'expected_event_days': len(dates),
                 'rain_mm': float(ed.rn_biz.mean()), 'wet_h': float(hh.wet_h.mean())}
        for dow in range(7):
            fixed[f'dow_{dow}'] = float((dates.dayofweek == dow).mean()) if len(dates) else np.nan
        for uid, g in by_unit.items():
            if g.MCT_SGG_CD.iloc[0] != e.MCT_SGG_CD:
                continue
            tr = g[g.use & ~g.rain & g.amt.gt(0) & g.date.lt(e.start)
                   & g.date.ge(e.start - pd.Timedelta(days=LOSS_LOOKBACK_DAYS))].copy()
            te = g.reindex(dates).copy()
            # Calendar/weather regressors come from weather even if sales is absent.
            for col in ed.columns:
                if col not in ['amt', 'uid', 'date']:
                    te[col] = ed[col].to_numpy()
            te['date'] = dates
            observed = te.amt.notna() & np.isfinite(te.amt) & te.amt.ge(0)
            usable = te.use.fillna(False).astype(bool)
            a = {'uid': uid, 'event': e.event, 'start': e.start, **fixed,
                 'observed_event_days': int(observed.sum()),
                 'usable_event_days': int((observed & usable).sum()),
                 'sales_coverage_pct': float(observed.mean() * 100) if len(te) else 0.,
                 'prior_calm_days': len(tr), 'reference_train_end': tr.date.max()}
            reason = ('missing_event_weather' if not event_weather_ok else
                      'missing_hourly_duration' if not duration_ok else
                      'incomplete_event_sales' if not observed.all() else
                      'excluded_event_day' if not usable.all() else
                      'insufficient_prior_calm_days' if len(tr) < LOSS_MIN_CALM_DAYS else 'ok')
            if reason != 'ok':
                audit.append({**a, 'status': reason})
                continue
            pred, model, residual = loss_reference(tr, te)
            if not np.isfinite(pred).all() or (pred <= 0).any():
                audit.append({**a, 'status': 'invalid_reference'})
                continue
            ref, actual = float(pred.sum()), float(te.amt.sum())
            sigma = max(float(np.std(residual, ddof=1)), .03)
            # Same D and total target for forward validation and the published final index.
            pre_dates = pd.date_range(e.start - pd.Timedelta(days=7), periods=7)
            pre = g.reindex(pre_dates)
            pre = pre[pre.use.fillna(False).astype(bool) & ~pre.rain.fillna(True).astype(bool) & pre.amt.gt(0)]
            after = g.reindex(pd.date_range(e.end + pd.Timedelta(days=1), periods=3))
            after_valid = (after.use.fillna(False).astype(bool) & ~after.rain.fillna(True).astype(bool)
                           & after.amt.gt(0))
            damage, after_damage, damage_raw, after_raw = [np.nan] * 4
            if len(pre) >= 3 and te.amt.gt(0).all():
                pre_abn = np.log(pre.amt) - loss_predict_ridge(model, loss_calendar_matrix(pre))
                level = float(pre_abn.mean())
                damage_raw = level - float((np.log(te.amt) - loss_predict_ridge(model, loss_calendar_matrix(te))).mean())
                damage = damage_raw / sigma
                if after_valid.all():
                    after_raw = level - float((np.log(after.amt) - loss_predict_ridge(model, loss_calendar_matrix(after))).mean())
                    after_damage = after_raw / sigma
            index_ok = np.isfinite(damage) and np.isfinite(after_damage)
            row = {'uid': uid, 'event': e.event, 'cluster': clusters[e.event],
                   'MCT_SGG_CD': e.MCT_SGG_CD, 'start': e.start, 'end': e.end,
                   '분야': g['분야'].iloc[0], '유형': g['유형'].iloc[0], 'mask_risk': g.mask_risk.iloc[0],
                   **fixed, 'observed_event_days': int(observed.sum()), 'sales_coverage_pct': 100.,
                   'baseline_krw': ref, 'actual_krw': actual, 'shortfall_krw': ref - actual,
                   'shortfall_rate': (ref - actual) / ref, 'prior_calm_days': len(tr),
                   'reference_train_end': tr.date.max(), 'reference_sigma': sigma,
                   'damage_D': damage, 'after_D': after_damage,
                   'damage_D_raw': damage_raw, 'after_D_raw': after_raw,
                   'total_D': damage + after_damage,
                   'index_available_end': e.end + pd.Timedelta(days=3),
                   'index_target_status': 'ok' if index_ok else 'incomplete_positive_pre_event_or_dry_after3',
                   'target_note': '과거 기준선 대비 매출 부족액(인과 피해 아님)'}
            records.append(row)
            audit.append({**a, 'status': 'ok', 'index_target_status': row['index_target_status']})
    cols = ['uid', 'event', 'cluster', 'MCT_SGG_CD', 'start', 'end', '분야', '유형',
            'mask_risk', 'n_days', 'expected_event_days', 'rain_mm', 'wet_h',
            'observed_event_days', 'sales_coverage_pct', 'baseline_krw', 'actual_krw',
            'shortfall_krw', 'shortfall_rate', 'prior_calm_days', 'reference_train_end',
            'reference_sigma', 'damage_D', 'after_D', 'damage_D_raw', 'after_D_raw',
            'total_D', 'index_available_end', 'index_target_status', 'target_note'] + [f'dow_{i}' for i in range(7)]
    audit_cols = ['uid', 'event', 'start', 'n_days', 'expected_event_days', 'rain_mm', 'wet_h', *[f'dow_{i}' for i in range(7)], 'observed_event_days', 'usable_event_days',
                  'sales_coverage_pct', 'prior_calm_days', 'reference_train_end', 'status', 'index_target_status']
    return pd.DataFrame(records, columns=cols), pd.DataFrame(audit, columns=audit_cols)


def loss_prior_sensitivity(reference, rows):
    """각 행보다 먼저 종료된 학습 사건만 사용한다. 현재 행/평가 정답은 참조하지 않는다."""
    values, counts = [], []
    for r in rows.itertuples():
        past = reference[reference.end.lt(r.start) & reference.event.ne(r.event)]
        own = past[past.uid.eq(r.uid)]
        sector = past[past.MCT_SGG_CD.eq(r.MCT_SGG_CD) & past['분야'].eq(r.분야)]
        region = past[past.MCT_SGG_CD.eq(r.MCT_SGG_CD)]
        pool = sector if len(sector) else region
        mu = float(pool.shortfall_rate.mean()) if len(pool) else 0.
        n = own.event.nunique()
        own_mean = float(own.shortfall_rate.mean()) if len(own) else mu
        values.append((n * own_mean + 3 * mu) / (n + 3))
        counts.append(n)
    return pd.Series(values, index=rows.index), pd.Series(counts, index=rows.index)


def loss_design(train, test, reference, include_weather=True):
    """범주·스케일은 학습 fold에서 결정한다. 미관측 범주는 0으로 처리한다."""
    tr, te = train.copy(), test.copy()
    tr['prior_sensitivity'], tr['sensitivity_events'] = loss_prior_sensitivity(reference, tr)
    te['prior_sensitivity'], te['sensitivity_events'] = loss_prior_sensitivity(reference, te)
    def numerical(d):
        sens = d.prior_sensitivity.to_numpy(float)
        rain = np.log1p(d.rain_mm.to_numpy(float))
        duration = d.wet_h.to_numpy(float)
        common = [np.ones(len(d)), np.log1p(d.n_days), np.log1p(d.baseline_krw / d.n_days)]
        weather_inputs = [rain, duration, sens, rain * sens, duration * sens] if include_weather else []
        return np.column_stack(common + weather_inputs + [d[f'dow_{k}'].to_numpy(float) for k in range(1, 7)])
    X, T = numerical(tr), numerical(te)
    for col in ['MCT_SGG_CD', '분야', '유형']:
        categories = sorted(tr[col].dropna().unique())
        for cat in categories[1:]:
            X = np.column_stack([X, tr[col].eq(cat).astype(float)])
            T = np.column_stack([T, te[col].eq(cat).astype(float)])
    return X, T, tr, te


def loss_fold_predictions(train, test):
    if train.cluster.nunique() < LOSS_MIN_TRAIN_CLUSTERS:
        return None
    X, T, tr, te = loss_design(train, test, train)
    sizes = tr.groupby('cluster').uid.transform('size').to_numpy(float)
    model = loss_fit_ridge(X, tr.shortfall_rate, weights=1 / sizes)
    raw_rate = loss_predict_ridge(model, T)
    prediction = np.minimum(raw_rate, 1.) * te.baseline_krw.to_numpy()
    # 평균 부족액 고정 기준과 업종별 하루 평균 기준을 모두 비교한다.
    region_mean = tr.groupby('MCT_SGG_CD').shortfall_krw.mean()
    average = te.MCT_SGG_CD.map(region_mean).fillna(tr.shortfall_krw.mean()).to_numpy()
    per_day = tr.assign(daily=tr.shortfall_krw / tr.n_days).groupby('uid').daily.mean()
    region_daily = tr.assign(daily=tr.shortfall_krw / tr.n_days).groupby('MCT_SGG_CD').daily.mean()
    unit_average = te.uid.map(per_day).fillna(te.MCT_SGG_CD.map(region_daily))
    unit_average = unit_average.fillna((tr.shortfall_krw / tr.n_days).mean()).to_numpy() * te.n_days
    result = te.copy()
    result['pred_ridge_krw'] = prediction
    C, CT, _, _ = loss_design(train, test, train, include_weather=False)
    calendar_model = loss_fit_ridge(C, tr.shortfall_rate, weights=1 / sizes)
    calendar_rate = loss_predict_ridge(calendar_model, CT)
    result['pred_calendar_krw'] = np.minimum(calendar_rate, 1.) * te.baseline_krw.to_numpy()
    result['calendar_upper_bound_applied'] = calendar_rate > 1.
    result['pred_rate_raw'] = raw_rate
    result['physical_upper_bound_applied'] = raw_rate > 1.
    result['pred_mean_krw'] = average
    result['pred_unit_mean_krw'] = np.asarray(unit_average)
    result['train_clusters'] = tr.cluster.nunique()
    result['train_end'] = tr.end.max()
    return result


def evaluate_loss_models(table):
    """사건 하나 빼기(겹치는 타지역 사건도 제외)와 시간 순차 검증을 분리한다."""
    predictions, coverage = [], []
    pred_cols = list(table.columns) + ['pred_ridge_krw', 'pred_calendar_krw', 'pred_mean_krw', 'pred_unit_mean_krw', 'scheme', 'held_out_event']
    metric_cols = ['scheme', 'event', 'cluster', '지역', 'model', 'units', 'MAE_krw', 'event_abs_error_krw', 'WAPE_pct', 'true_shortfall_krw', 'predicted_shortfall_krw', 'train_end', 'start']
    coverage_cols = ['scheme', 'event', 'cluster', 'start', 'eligible_units', 'train_clusters', 'status']
    if table.empty:
        return pd.DataFrame(columns=pred_cols), pd.DataFrame(columns=metric_cols), pd.DataFrame(columns=coverage_cols)
    for scheme in ['LOEO_shared_weather_block', 'forward_past_only']:
        for event, test_all in table.groupby('event', sort=False):
            cluster = test_all.cluster.iloc[0]
            start = test_all.start.min()
            train = table[table.cluster.ne(cluster)].copy()
            if scheme == 'forward_past_only':
                train = train[train.end.lt(start)]
            test = test_all[test_all.mask_risk.ne('high')].copy()
            res = loss_fold_predictions(train, test) if len(test) else None
            coverage.append({'scheme': scheme, 'event': event, 'cluster': cluster,
                'start': start, 'eligible_units': len(test), 'train_clusters': train.cluster.nunique(),
                'status': 'ok' if res is not None else 'insufficient_history_or_evaluation_units'})
            if res is None:
                continue
            res['scheme'], res['held_out_event'] = scheme, event
            predictions.append(res)
    if not predictions:
        return pd.DataFrame(columns=pred_cols), pd.DataFrame(columns=metric_cols), pd.DataFrame(coverage, columns=coverage_cols)
    pred = pd.concat(predictions, ignore_index=True)
    rows = []
    for (scheme, event), g in pred.groupby(['scheme', 'held_out_event']):
        for model, col in [('ridge', 'pred_ridge_krw'), ('calendar_only', 'pred_calendar_krw'), ('mean_loss', 'pred_mean_krw'),
                           ('unit_daily_mean', 'pred_unit_mean_krw')]:
            err = g[col] - g.shortfall_krw
            denominator = float(g.shortfall_krw.abs().sum())
            rows.append({'scheme': scheme, 'event': event, 'cluster': g.cluster.iloc[0],
                '지역': g.MCT_SGG_CD.iloc[0], 'model': model, 'units': len(g),
                'MAE_krw': float(err.abs().mean()), 'event_abs_error_krw': float(abs(err.sum())),
                'WAPE_pct': float(err.abs().sum() / denominator * 100) if denominator > 0 else np.nan,
                'true_shortfall_krw': float(g.shortfall_krw.sum()),
                'predicted_shortfall_krw': float(g[col].sum()),
                'train_end': g.train_end.iloc[0], 'start': g.start.min()})
    return pred, pd.DataFrame(rows), pd.DataFrame(coverage)


def loss_cluster_summary(metrics, repetitions=500):
    """업종 단위 MAE와 지역×사건 총액 오차를 분리하고 동일 사건의 차이를 비교한다."""
    cols = ['scheme', 'metric', 'mean', 'ci_low', 'ci_high', 'weather_clusters', 'events']
    result = []
    rng = np.random.default_rng(42)
    def add(scheme, label, values, events):
        values = values.dropna().to_numpy(float)
        if not len(values):
            return
        bs = np.array([rng.choice(values, len(values), replace=True).mean() for _ in range(repetitions)])
        result.append({'scheme': scheme, 'metric': label, 'mean': values.mean(),
            'ci_low': np.quantile(bs, .025) if len(values) >= 3 else np.nan,
            'ci_high': np.quantile(bs, .975) if len(values) >= 3 else np.nan,
            'weather_clusters': len(values), 'events': events})
    for scheme, group in metrics.groupby('scheme'):
        for source, label in [('MAE_krw', 'unit_MAE_krw'), ('event_abs_error_krw', 'region_event_total_MAE_krw')]:
            c = group.groupby(['cluster', 'model'])[source].mean().unstack('model')
            for model in c:
                add(scheme, model + '_' + label, c[model], group.event.nunique())
            for control in ['unit_daily_mean', 'calendar_only']:
                if {'ridge', control}.issubset(c.columns):
                    add(scheme, control + '_minus_ridge_' + label + '_positive_is_improvement',
                        c[control] - c.ridge, group.event.nunique())
    return pd.DataFrame(result, columns=cols)


def duration_scenario_loss(base_effect_pct, hourly_effect_pct, forecast_hours, reference_hours, reference_sales_krw):
    """가중치를 총손실에 임의 곱하지 않고 로그 매출비에 지속시간 차이를 더한다.

    시간당 효과는 총량·강도 통제 조건의 연관 추정치다. 단독 인과 효과가 아니다.
    반환값은 그 효과가 적용된다고 가정한 시나리오이며 별도 검증 성능이 아니다.
    """
    if not all(np.isfinite([base_effect_pct, hourly_effect_pct, forecast_hours, reference_hours, reference_sales_krw])):
        return np.nan
    if min(base_effect_pct, hourly_effect_pct) <= -100 or min(forecast_hours, reference_hours, reference_sales_krw) < 0:
        raise ValueError('효과율·시간·기준매출 범위를 확인하세요.')
    log_ratio = np.log1p(base_effect_pct / 100) + np.log1p(hourly_effect_pct / 100) * (forecast_hours - reference_hours)
    return float(reference_sales_krw * max(0., -np.expm1(log_ratio)))


INDEX_TARGET = 'total_D = (사전7일 평상일 abn평균 - 사건 abn평균 + 사전평균 - 사후3일 abn평균) / 과거평상일σ'
INDEX_FIXED_FALLBACK = '비민감도만'


def forward_index_features(history, sales, region, origin):
    """최종 산출·내부 가중치 선택·외부 평가에서 함께 쓰는 단일 특성 함수."""
    h = history[history.MCT_SGG_CD.eq(region) & history.index_available_end.lt(origin)
                & history.total_D.notna()].copy()
    cols = ['비민감도', '회복지연', '외지비중', '오후집중', '노출', '과거사건수']
    if h.empty:
        return pd.DataFrame(columns=cols).rename_axis('uid')
    f = h.groupby('uid').agg(과거사건수=('event', 'nunique'), 분야=('분야', 'first'))
    for source, name in [('damage_D', '비민감도'), ('after_D', '회복지연')]:
        means = h.groupby('uid')[source].agg(['mean', 'count'])
        mu = means['mean'].groupby(f['분야']).transform('mean')
        f[name] = (means['count'] * means['mean'] + 3 * mu) / (means['count'] + 3)
    prior = sales[sales.MCT_SGG_CD.eq(region) & sales.date.lt(origin)
                  & sales.date.ge(origin - pd.Timedelta(days=LOSS_LOOKBACK_DAYS)) & sales.use & sales.amt.gt(0)]
    for name, source_cols in [('외지', ['amt_out']), ('오후', ['tb_12_17'])]:
        # Missing raw composition remains unavailable, never silently set to 0.
        if all(col in prior for col in source_cols):
            x = prior.dropna(subset=source_cols).copy()
            num = x[source_cols].sum(axis=1).groupby(x.uid).sum(min_count=1)
            den = ((x.amt_res + x.amt_out).groupby(x.uid).sum(min_count=1)
                   if name == '외지' and 'amt_res' in x else x.groupby('uid').amt.sum(min_count=1))
            f['외지비중' if name == '외지' else '오후집중'] = (num / den.where(den.gt(0))).reindex(f.index)
        else:
            f['외지비중' if name == '외지' else '오후집중'] = np.nan
    f['노출'] = (z(f.외지비중) + z(f.오후집중)) / 2
    return f[cols]


def forward_index_score(features, weights):
    active = [(w, features[col]) for w, col in zip(weights, ['비민감도', '회복지연', '노출']) if w != 0]
    return z(sum(w * z(v) for w, v in active)).rename('취약점수')


def forward_index_pair(history, sales, test, origin):
    region = test.MCT_SGG_CD.iloc[0]
    f = forward_index_features(history, sales, region, origin)
    eligible = test[test.mask_risk.ne('high') & test.total_D.notna()].set_index('uid')
    ids = eligible.index.intersection(f.index)
    return f, eligible.loc[ids, 'total_D']


def choose_forward_index_weights(history, sales, origin):
    """외부 사건보다 앞서 관측 완료된 사건의 내부 순차 검증에서만 가중치를 고른다."""
    h = history[history.index_available_end.lt(origin) & history.total_D.notna()]
    results = []
    for event, test in h.groupby('event', sort=False):
        start, cluster = test.start.min(), test.cluster.iloc[0]
        train = h[h.index_available_end.lt(start) & h.cluster.ne(cluster)]
        if train.cluster.nunique() < LOSS_MIN_TRAIN_CLUSTERS:
            continue
        f, target = forward_index_pair(train, sales, test, start)
        if len(target) < 10:
            continue
        candidate_scores = {name: forward_index_score(f, weights).reindex(target.index)
                            for name, weights in INDEX_WEIGHT_CANDIDATES.items()}
        common_units = target.notna()
        for score in candidate_scores.values():
            common_units &= score.notna()
        if common_units.sum() < 10:
            continue
        target = target[common_units]
        for name, score in candidate_scores.items():
            score = score[common_units]
            if score.nunique() < 2 or target.nunique() < 2:
                continue
            results.append({'candidate': name, 'event': event, 'cluster': cluster,
                            'rho': score.rank().corr(target.rank())})
    if not results:
        return INDEX_FIXED_FALLBACK, INDEX_WEIGHT_CANDIDATES[INDEX_FIXED_FALLBACK], 'fixed_fallback_no_inner_validation', 0
    scores = pd.DataFrame(results)
    # All candidates must have the same inner cases; avoid advantage from selective abstention.
    common = scores.groupby('event').candidate.nunique()
    common = common[common.eq(len(INDEX_WEIGHT_CANDIDATES))].index
    scores = scores[scores.event.isin(common)]
    if scores.empty:
        return INDEX_FIXED_FALLBACK, INDEX_WEIGHT_CANDIDATES[INDEX_FIXED_FALLBACK], 'fixed_fallback_no_common_inner_cases', 0
    mean = scores.groupby(['candidate', 'cluster']).rho.mean().groupby('candidate').mean()
    best = max(INDEX_WEIGHT_CANDIDATES, key=lambda name: mean.get(name, -np.inf))
    return best, INDEX_WEIGHT_CANDIDATES[best], 'nested_forward_selected', scores.event.nunique()


def loss_index_forward_table(table, sales):
    """최종 지수와 동일한 특성·목표·가중치 선택 절차의 시간 순차 검증."""
    rows, predictions = [], []
    for event, test_all in table.groupby('event', sort=False):
        origin, cluster = test_all.start.min(), test_all.cluster.iloc[0]
        history = table[table.index_available_end.lt(origin) & table.cluster.ne(cluster) & table.total_D.notna()]
        test = test_all[test_all.mask_risk.ne('high') & test_all.total_D.notna()]
        name, weights, selection, inner_n = choose_forward_index_weights(history, sales, origin)
        f, target = forward_index_pair(history, sales, test_all, origin)
        score = forward_index_score(f, weights).reindex(target.index)
        pair = pd.DataFrame({'score': score, 'target': target}).dropna()
        enough = history.cluster.nunique() >= LOSS_MIN_TRAIN_CLUSTERS and len(pair) >= 10
        identified = enough and pair.score.nunique() > 1 and pair.target.nunique() > 1
        rows.append({'event': event, 'cluster': cluster, 'start': origin,
            'train_end': history.index_available_end.max(), 'eligible_units': len(test), 'units': len(pair),
            'coverage_pct': len(pair) / len(test) * 100 if len(test) else np.nan,
            'rho': pair.score.rank().corr(pair.target.rank()) if identified else np.nan,
            'top20_hit': top_hit(pair.score, pair.target) if identified else np.nan,
            'random_top20_hit': max(1, int(round(len(pair) * .2))) / len(pair) if len(pair) else np.nan,
            'status': 'ok' if identified else 'insufficient_history_units_or_rank_variation',
            'weight_name': name, 'weights': str(weights), 'weight_selection': selection,
            'inner_validation_events': inner_n, 'target': INDEX_TARGET})
        if identified:
            p = pair.rename_axis('uid').reset_index()
            p['event'], p['cluster'], p['start'] = event, cluster, origin
            p['train_end'] = history.index_available_end.max()
            p['weight_name'] = name
            predictions.append(p)
    cols = ['event', 'cluster', 'start', 'train_end', 'eligible_units', 'units', 'coverage_pct',
            'rho', 'top20_hit', 'random_top20_hit', 'status', 'weight_name', 'weights',
            'weight_selection', 'inner_validation_events', 'target']
    return pd.DataFrame(rows, columns=cols), (pd.concat(predictions, ignore_index=True) if predictions else
        pd.DataFrame(columns=['uid', 'score', 'target', 'event', 'cluster', 'start', 'train_end', 'weight_name']))


def index_forward_summary(results, repetitions=500):
    out = []
    rng = np.random.default_rng(42)
    valid = results[results.status.eq('ok')]
    for metric in ['rho', 'top20_hit', 'random_top20_hit']:
        values = valid.groupby('cluster')[metric].mean().dropna().to_numpy()
        if not len(values):
            continue
        bs = np.array([rng.choice(values, len(values), replace=True).mean() for _ in range(repetitions)])
        out.append({'metric': metric, 'mean': values.mean(),
                    'ci_low': np.quantile(bs, .025) if len(values) >= 3 else np.nan,
                    'ci_high': np.quantile(bs, .975) if len(values) >= 3 else np.nan,
                    'weather_clusters': len(values), 'events': valid.event.nunique()})
    return pd.DataFrame(out, columns=['metric', 'mean', 'ci_low', 'ci_high', 'weather_clusters', 'events'])


def build_final_forward_index(table, sales, unit_info, origin):
    """배포 시점까지 완전히 관측된 사건으로 최종 지수를 산출한다. 평가와 동일 절차."""
    history = table[table.index_available_end.lt(origin) & table.total_D.notna()]
    name, weights, selection, inner_n = choose_forward_index_weights(history, sales, origin)
    frames = []
    for region in unit_info.MCT_SGG_CD.unique():
        f = forward_index_features(history, sales, region, origin)
        f['취약점수'] = forward_index_score(f, weights)
        if history.cluster.nunique() < LOSS_MIN_TRAIN_CLUSTERS or f.취약점수.nunique() < 2:
            f['취약점수'] = np.nan
        rank = f.취약점수.rank(method='average')
        n = f.취약점수.notna().sum()
        f['취약 백분위(0~100)'] = 100 * (rank - 1) / n if n else np.nan
        f['회복탄력성 지수'] = (100 - f['취약 백분위(0~100)']).round(1)
        frames.append(f)
    features = pd.concat(frames) if frames else pd.DataFrame()
    result = unit_info.set_index('uid').join(features)
    result['등급'] = pd.cut(result['회복탄력성 지수'], [-1, 20, 40, 101], labels=['우선지원', '주의', '양호']).astype('string').fillna('미평가')
    result['유형'] = '미평가'
    for region, g in result.groupby('MCT_SGG_CD'):
        for uid, r in g.iterrows():
            if pd.notna(r['회복탄력성 지수']):
                result.loc[uid, '유형'] = typ(r, g[['비민감도', '회복지연']].median())
    result['신뢰도'] = np.where(result.mask_risk.eq('high') | result.과거사건수.fillna(0).lt(3)
                               | result['회복탄력성 지수'].isna(), '낮음(표본·잡음)', '보통')
    result['비 온 날 매출(%)'] = np.expm1(-history.groupby('uid').damage_D_raw.mean()).reindex(result.index) * 100
    result['그친 뒤 1~3일(%)'] = np.expm1(-history.groupby('uid').after_D_raw.mean()).reindex(result.index) * 100
    result['산출기준일'] = origin
    result['지수버전'] = LOSS_MODEL_VERSION
    result['가중치선택'] = selection
    result['가중치내부검증사건'] = inner_n
    return result, name, weights


def predict_shortfall_scenario(sales, event_table, region, start_date, rain_mm, wet_hours,
                               days=1, heat=False, tropical_night=False):
    """사용자가 넣은 강수 시나리오의 업종별 부족액 예측. 예보 정확도 검증 아님.

    기간 중 매일 동일한 mm·강수시간을 가정한다. 계절·기상 외삽을 별도로 표시한다.
    반환된 원화는 지급액이 아니며 실제 지급에는 자격·영업 확인·상한·정산이 필요하다.
    """
    start = pd.Timestamp(start_date)
    if days < 1 or not isinstance(days, (int, np.integer)) or not 0 <= wet_hours <= 13 or rain_mm < 0:
        raise ValueError('days는 양의 정수, 영업시간 강수는 0 이상, 강수시간은 0~13이어야 합니다.')
    train = event_table[event_table.end.lt(start)].copy()
    if train.cluster.nunique() < LOSS_MIN_TRAIN_CLUSTERS:
        return pd.DataFrame(), 'insufficient_prior_weather_clusters'
    dates = pd.date_range(start, periods=days)
    records = []
    for uid, g in sales[sales.MCT_SGG_CD.eq(region)].groupby('uid'):
        tr = g[g.date.lt(start) & g.date.ge(start-pd.Timedelta(days=LOSS_LOOKBACK_DAYS))
               & g.use & ~g.rain & g.amt.gt(0)]
        if len(tr) < LOSS_MIN_CALM_DAYS:
            continue
        te = pd.DataFrame({'date': dates})
        te['month_end'] = te.date.dt.is_month_end
        te['school_vac'] = in_ranges(te.date, SCHOOL_VAC)
        te['festival'] = in_ranges(te.date, FESTIVAL_CC) & (region == CC)
        te['heat'], te['trop'] = heat, tropical_night
        model = loss_fit_ridge(loss_calendar_matrix(tr), np.log(tr.amt))
        res = np.log(tr.amt) - loss_predict_ridge(model, loss_calendar_matrix(tr))
        reference = float((np.exp(loss_predict_ridge(model, loss_calendar_matrix(te))) * np.exp(res).mean()).sum())
        row = {'uid': uid, 'event': 'user_scenario', 'cluster': 'user_scenario',
            'MCT_SGG_CD': region, 'start': start, 'end': dates.max(),
            '분야': g['분야'].iloc[0], '유형': g['유형'].iloc[0], 'mask_risk': g.mask_risk.iloc[0],
            'n_days': days, 'rain_mm': rain_mm, 'wet_h': wet_hours, 'baseline_krw': reference}
        for dow in range(7):
            row[f'dow_{dow}'] = float((dates.dayofweek == dow).mean())
        records.append(row)
    if not records:
        return pd.DataFrame(), 'insufficient_prior_reference_sales'
    result = loss_fold_predictions(train, pd.DataFrame(records))
    region_history = train[train.MCT_SGG_CD.eq(region)]
    if region_history.empty:
        result['weather_extrapolation'] = True
    else:
        result['weather_extrapolation'] = not (region_history.rain_mm.min() <= rain_mm <= region_history.rain_mm.max()
            and region_history.wet_h.min() <= wet_hours <= region_history.wet_h.max())
    result['prediction_context'] = '사용자 강수 시나리오·고정 업종표본; 지급액 아님'
    return result, 'ok'


def diagnose_loss_reference(sales, weather, horizon_days=3):
    """매주 고정한 3일 맑은 구간을 순차 홀드아웃하여 기준선 자체를 진단한다.

    시작일·기간은 매출을 보기 전에 고정한다. 기상·사용 조건이 안 맞으면 제외 사유를 남긴다.
    맑은 날 예측오차를 보는 음성 대조이며 비가 없었을 실제 매출의 정답은 아니다.
    """
    records, audit = [], []
    for region, w in weather.groupby('MCT_SGG_CD'):
        w = w.set_index('date').sort_index()
        origins = pd.date_range(w.index.min() + pd.Timedelta(days=28), w.index.max(), freq='7D')
        for origin in origins:
            dates = pd.date_range(origin, periods=horizon_days)
            holdout = w.reindex(dates)
            suitable = holdout.use.fillna(False).astype(bool) & holdout.rn_biz.eq(0)
            for uid, g in sales[sales.MCT_SGG_CD.eq(region)].groupby('uid'):
                g = g.set_index('date', drop=False)
                tr = g[g.date.lt(origin) & g.date.ge(origin-pd.Timedelta(days=LOSS_LOOKBACK_DAYS))
                       & g.use & ~g.rain & g.amt.gt(0)]
                te = g.reindex(dates).copy()
                valid = te.amt.notna() & np.isfinite(te.amt) & te.amt.ge(0)
                reason = ('not_complete_dry_weather_window' if not suitable.all() else
                          'incomplete_holdout_sales' if not valid.all() else
                          'insufficient_prior_calm_days' if len(tr) < LOSS_MIN_CALM_DAYS else 'ok')
                audit.append({'uid': uid, 'MCT_SGG_CD': region, 'origin': origin, 'horizon_days': horizon_days,
                              'prior_days': len(tr), 'observed_days': int(valid.sum()), 'status': reason})
                if reason != 'ok':
                    continue
                pred, _, _ = loss_reference(tr, te)
                for j, date in enumerate(dates):
                    records.append({'uid': uid, 'MCT_SGG_CD': region, 'origin': origin, 'date': date,
                                    'horizon': j+1, 'train_end': tr.date.max(),
                                    'observed_krw': float(te.amt.iloc[j]), 'reference_krw': float(pred[j]),
                                    'error_krw': float(pred[j] - te.amt.iloc[j]),
                                    'context': '고정 3일 맑은 구간 순차 홀드아웃; 인과 기준선 정답 아님'})
    cols = ['uid', 'MCT_SGG_CD', 'origin', 'date', 'horizon', 'train_end', 'observed_krw', 'reference_krw', 'error_krw', 'context']
    return pd.DataFrame(records, columns=cols), pd.DataFrame(audit, columns=['uid', 'MCT_SGG_CD', 'origin', 'horizon_days', 'prior_days', 'observed_days', 'status'])


def summarize_loss_reference(predictions):
    cols = ['level', 'MCT_SGG_CD', 'uid', 'month', 'holdouts', 'days', 'MAE_krw', 'bias_krw', 'WAPE_pct', 'bias_pct']
    if predictions.empty:
        return pd.DataFrame(columns=cols)
    d = predictions.copy()
    d['month'] = d.date.dt.to_period('M').astype(str)
    rows = []
    for level, keys in [('region', ['MCT_SGG_CD', 'month']), ('unit', ['MCT_SGG_CD', 'uid', 'month'])]:
        for key, g in d.groupby(keys):
            info = dict(zip(keys, key))
            den = g.observed_krw.sum()
            rows.append({**info, 'level': level, 'holdouts': g.origin.nunique(), 'days': len(g),
                         'MAE_krw': g.error_krw.abs().mean(), 'bias_krw': g.error_krw.mean(),
                         'WAPE_pct': 100 * g.error_krw.abs().sum() / den if den > 0 else np.nan,
                         'bias_pct': 100 * g.error_krw.sum() / den if den > 0 else np.nan})
    return pd.DataFrame(rows, columns=cols)


In [ ]:
# 취합 1회차 추가 결과. 아래 성능은 기준선 추정 목표에 대한 조건부 예측 성능이다.
SU_WITH_INFO = SU.merge(units[['uid', '분야', '유형', 'mask_risk']], on='uid', how='left', validate='many_to_one')
LOSS_EVENTS, LOSS_REFERENCE_AUDIT = build_loss_event_table(SU_WITH_INFO, EVENTS, RH, wx)
LOSS_EVENTS.to_csv(os.path.join(OUT, '10-1_사건별_매출부족액_목표.csv'), index=False, encoding='utf-8-sig')
LOSS_REFERENCE_AUDIT.to_csv(os.path.join(OUT, '10-1_기준선_사용가능성.csv'), index=False, encoding='utf-8-sig')
LOSS_PRED, LOSS_METRICS, LOSS_COVERAGE = evaluate_loss_models(LOSS_EVENTS)
LOSS_PRED.to_csv(os.path.join(OUT, '10-2_피해규모_검증예측.csv'), index=False, encoding='utf-8-sig')
LOSS_METRICS.to_csv(os.path.join(OUT, '10-3_피해규모_사건별성능.csv'), index=False, encoding='utf-8-sig')
LOSS_COVERAGE.to_csv(os.path.join(OUT, '10-4_피해규모_검증가능성.csv'), index=False, encoding='utf-8-sig')
LOSS_SUMMARY = loss_cluster_summary(LOSS_METRICS)
LOSS_SUMMARY.to_csv(os.path.join(OUT, '10-5_피해규모_성능과불확실성.csv'), index=False, encoding='utf-8-sig')
# Final published index and forward validation share one procedure and one target.
LOSS_INDEX_FORWARD, LOSS_INDEX_PRED = loss_index_forward_table(LOSS_EVENTS, SU_WITH_INFO)
LOSS_INDEX_FORWARD.to_csv(os.path.join(OUT, '10-6_최종지수_동일목표_순차검증.csv'), index=False, encoding='utf-8-sig')
LOSS_INDEX_PRED.to_csv(os.path.join(OUT, '10-6_최종지수_업종별_순차예측.csv'), index=False, encoding='utf-8-sig')
INDEX_FORWARD_SUMMARY = index_forward_summary(LOSS_INDEX_FORWARD)
INDEX_FORWARD_SUMMARY.to_csv(os.path.join(OUT, '10-7_최종지수_순차검증_불확실성.csv'), index=False, encoding='utf-8-sig')
F_LEGACY_RETROSPECTIVE = F.copy()
INDEX_ORIGIN = wx.date.max() + pd.Timedelta(days=1)
F, W_NAME, W = build_final_forward_index(LOSS_EVENTS, SU_WITH_INFO, units, INDEX_ORIGIN)
F = F.join(UE.set_index('uid')[['원화(억)']])
F.to_csv(os.path.join(OUT, '5-5_회복탄력성지수_순차동일공식.csv'), encoding='utf-8-sig')
# Rebuild support candidates from the new index, while the amount remains the retrospective association estimate.
rows = []
for sgg in REG:
    s = F[F.MCT_SGG_CD.eq(sgg) & F.신뢰도.eq('보통')].copy()
    s['손실'] = -s['원화(억)'].clip(upper=0)
    q = s.손실.quantile(.8)
    for _, r in s.iterrows():
        lo, big = r['회복탄력성 지수'] <= 20, r.손실 >= q
        rows.append({'지역': SHORT[sgg], '업종': r.MCT_RY_CD, '지수': r['회복탄력성 지수'],
                     '손실(억)': round(r.손실, 1), '유형': r.유형,
                     '구분': '1순위(약하고 손실 큼)' if lo and big else '선별 지원(약하지만 손실 작음)' if lo else '규모 지원(강하지만 손실 큼)' if big else '-',
                     '금액의미': '전체기간 회귀 연관 추정; 인과 피해·예측 지급액 아님'})
MX = pd.DataFrame(rows, columns=['지역', '업종', '지수', '손실(억)', '유형', '구분', '금액의미'])
MX.to_csv(os.path.join(OUT, '5-6_우선지원_매트릭스_순차동일공식.csv'), index=False, encoding='utf-8-sig')
REFERENCE_PRED, REFERENCE_COVERAGE = diagnose_loss_reference(SU_WITH_INFO, wx)
REFERENCE_MONTHLY = summarize_loss_reference(REFERENCE_PRED)
REFERENCE_PRED.to_csv(os.path.join(OUT, '10-8_기준선_맑은구간_순차예측.csv'), index=False, encoding='utf-8-sig')
REFERENCE_COVERAGE.to_csv(os.path.join(OUT, '10-8_기준선_홀드아웃_관측상태.csv'), index=False, encoding='utf-8-sig')
REFERENCE_MONTHLY.to_csv(os.path.join(OUT, '10-9_기준선_지역업종월별_진단.csv'), index=False, encoding='utf-8-sig')
print('최종 지수:', W_NAME, W, '| 표준화 종합목표와 동일 공식 순차 검증; 원화 부족률 순위와 구별')
print(INDEX_FORWARD_SUMMARY.to_string(index=False))
print('기준선 맑은구간 진단과 강수입력 제외 모형 대비 개선 차이를 함께 확인하세요.')

if not LOSS_SUMMARY.empty:
    print(LOSS_SUMMARY.round(3).to_string(index=False))
else:
    print('기준선·학습 사건이 부족하여 평가할 수 없습니다. 정확도·손실을 0으로 채우지 않습니다.')
print('주 평가: forward_past_only. LOEO_shared_weather_block는 미래 사건 학습이 가능한 사후 비교입니다.')
print('입력 강수는 관측값입니다. 예보 오차를 포함한 재난 전 예측 성능은 아닙니다.')

# 방법에 따른 손실 민감도 범위. 신뢰구간이나 소상공인 확정 손실이 아니다.
ue = UE.copy()
ue['원화_raw_억'] = ue.rain_amt * -np.expm1(-ue.b) / 1e8
sensitivity_rows = []
for sgg, g in ue.groupby('MCT_SGG_CD'):
    for scope, sub in [('전체 분석업종', g), ('골프장·할인점 제외 대리지표', g[~g.MCT_RY_CD.isin(DIRECT_EXCLUDED_RY)])]:
        for method, col in [('원계수', '원화_raw_억'), ('베이즈 안정화', '원화(억)')]:
            sensitivity_rows.append({'지역': SHORT[sgg], '대상': scope, '방법': method,
                '순매출부족_억': -sub[col].sum(), '총감소분_억': -sub.loc[sub[col].lt(0), col].sum(),
                '자격확인': '사업체별 소상공인 자격 미확인'})
SENSITIVITY_LOSS = pd.DataFrame(sensitivity_rows)
SENSITIVITY_LOSS.to_csv(os.path.join(OUT, '11-1_손실_방법민감도.csv'), index=False, encoding='utf-8-sig')
LOSS_RANGES = SENSITIVITY_LOSS.groupby(['지역', '대상']).순매출부족_억.agg(최솟값='min', 최댓값='max').reset_index()
LOSS_RANGES['범위의미'] = '원계수/안정화계수 방법 민감도; 통계적 신뢰구간 아님'
LOSS_RANGES.to_csv(os.path.join(OUT, '11-1_손실_방법범위.csv'), index=False, encoding='utf-8-sig')

# 보전율별 연간 비용: 실제 자격이 확인되지 않은 대리지표, 예산 비교 가정.
BUDGET_REFERENCE_EOK = 562.0  # 취합본 제공 비교치. 확정 재원·가용 예산 검증 아님.
compensation_rows = []
cal = CALIBRATION.set_index('MCT_SGG_CD')
ea = pd.read_csv(os.path.join(WORK, 'climate', 'era5_annual.csv'))
for sgg, site in [(GN, 'gangnam'), (CC, 'chuncheon')]:
    g = ue[ue.MCT_SGG_CD.eq(sgg)]
    n_rain = float(g.n_rain.max())
    n_annual = ea[ea.site.eq(site) & ea.date.between(2016, 2025)].rain5.mean() * cal.loc[sgg, 'count_ratio']
    for scope, sub in [('전체 분석업종', g), ('골프장·할인점 제외 대리지표', g[~g.MCT_RY_CD.isin(DIRECT_EXCLUDED_RY)])]:
        annual_gross_loss = -sub.loc[sub['원화(억)'].lt(0), '원화(억)'].sum() / n_rain * n_annual
        for fraction in [.2, .3, .5]:
            compensation_rows.append({'지역': SHORT[sgg], '대상': scope, '보전율': fraction,
                '평년_연간총감소분_억': annual_gross_loss,
                '연간비용_억': annual_gross_loss * fraction,
                '562억_비교비율_pct': annual_gross_loss * fraction / BUDGET_REFERENCE_EOK * 100,
                '가정': '하반기 1일 효과 연중 동일; 상한·실제 자격 미적용; 비교용 예산'})
COMPENSATION = pd.DataFrame(compensation_rows)
COMPENSATION.to_csv(os.path.join(OUT, '11-2_보전율별_연간비용_시나리오.csv'), index=False, encoding='utf-8-sig')
if not COMPENSATION.empty:
    TOTAL_COMPENSATION = COMPENSATION.groupby(['대상', '보전율'], as_index=False).agg(연간비용_억=('연간비용_억', 'sum'))
    TOTAL_COMPENSATION['562억_비교비율_pct'] = TOTAL_COMPENSATION.연간비용_억 / BUDGET_REFERENCE_EOK * 100
    TOTAL_COMPENSATION.to_csv(os.path.join(OUT, '11-2_보전율별_두지역합계.csv'), index=False, encoding='utf-8-sig')
    print(TOTAL_COMPENSATION.round(2).to_string(index=False))

# 업종 제외 대리지표와 일반 상권의 지원 경로를 구별한다. 실제 지급·자격 판정은 아니다.
SUPPORT_REVIEW = MX.copy()
SUPPORT_REVIEW['지원경로'] = np.where(SUPPORT_REVIEW.업종.isin(DIRECT_EXCLUDED_RY),
    '직접 보전 후보 제외·자구책/공동사업 검토', '직접 보전 검토 후보·사업체 자격 확인 필요')
SUPPORT_REVIEW.to_csv(os.path.join(OUT, '11-3_지원경로_구분.csv'), index=False, encoding='utf-8-sig')

# 장마형 가중치: 당일 강수 반응과 시간당 계수를 로그 매출비에서 결합한다.
DURATION_EXAMPLES = []
for sgg in REG:
    eligible = panel[panel.MCT_SGG_CD.eq(sgg) & ~panel.MCT_RY_CD.isin(DIRECT_EXCLUDED_RY)]
    daily_sales = eligible[eligible.use & ~eligible.rain].groupby('date').amt.sum()
    base = eligible[eligible.use].groupby('date').agg(amt=('amt', 'sum')).reset_index()
    base = base.merge(wx[wx.MCT_SGG_CD.eq(sgg)], on='date', suffixes=('', '_wx'))
    if not len(base) or daily_sales.empty:
        continue
    base['y'] = np.log(base.amt)
    effect = pct(fit(base.y, calendar_X(base).join(base[['rain', 'heat', 'trop']].astype(float))), 'rain')['효과']
    # 지속시간 계수도 동일한 제외 표본에서 재추정한다.
    duration_data = base.merge(RH[RH.MCT_SGG_CD.eq(sgg)][['date', 'wet_h', 'rn_max']], on='date')
    duration_data = duration_data[duration_data.rn_biz.gt(0)]
    X = calendar_X(duration_data).join(pd.DataFrame({'log총량': np.log1p(duration_data.rn_biz),
        '비온시간': duration_data.wet_h, '최대강도': duration_data.rn_max}, index=duration_data.index)).join(duration_data[['heat', 'trop']].astype(float))
    duration_effect = pct(fit(duration_data.y, X), '비온시간')['효과']
    reference_hours = duration_data.loc[duration_data.rn_biz.ge(RAIN_BIZ_MM), 'wet_h'].mean()
    for forecast_hours in [3., 6., 9.]:
        DURATION_EXAMPLES.append({'지역': SHORT[sgg], '대상': '골프장·할인점 제외 대리지표',
            '시간당효과_pct': duration_effect, '기준비온시간': reference_hours,
            '시나리오비온시간': forecast_hours, '기준일매출_원': daily_sales.mean(),
            '부족액시나리오_원': duration_scenario_loss(effect, duration_effect, forecast_hours, reference_hours, daily_sales.mean()),
            '해석': '동일 총량 조건의 탐색 계수; 인과효과·검증된 예보예측 아님'})
pd.DataFrame(DURATION_EXAMPLES).to_csv(os.path.join(OUT, '11-4_지속시간_시나리오.csv'), index=False, encoding='utf-8-sig')

# 숫자를 고정하지 않고 현재 계산 결과에 연결한 보고서 대조표.
report_lines = [
    '# 취합 1회차: 이번 실행에서 확인할 사항',
    '- 기존 16/18 및 v2 중첩검증은 전체기간 사후 분석이다. 새 최종 지수의 성능은 10-6·10-7 순차검증에서 확인한다.',
    '- 피해 규모 모형의 목표는 과거 기준선 대비 카드매출 부족액이다. 인과적 피해 정답이 아니다.',
    '- 월별 통신 분석은 일별 방문 감소의 검증이 아니다. 6개월 전체 결과를 먼저 제시한다.',
    '- 손실 범위는 11-1, 보전율별 비용은 11-2의 실제 재실행 결과를 사용한다.',
    '- 골프장·할인점 제외분은 실제 소상공인 자격 확인값이 아닌 대리지표다.',
    '- 연간 비용은 하반기 일별 효과를 연중 적용한 시나리오이며 예산 562억은 취합본 비교 가정이다.',
    '- 회복 시작일은 7일까지이며 8일째로 연속 회복을 확인한다. 누적손실은 사건 중+사후5일로 별도 집계한다.',
    '- 10-1에서 기상 사건은 고정되며 매출 완전 관측만 원화 주 평가에 사용한다. 관측 0원은 포함한다.',
    '- 10-9에서 맑은 날 기준선의 지역·업종·월별 편향을 확인하고 10-5에서 강수입력 제외 모형과 같은 사건으로 비교한다.',
    '- 예측 입력에는 당시 발표된 예보가 아니라 실제 관측 기상이 사용됐다.',
    '- 외부 지역·연도 검증은 향후 과제다. 현재 전체기간 선정 업종표본이라는 한계를 남긴다.',
]
if not LOSS_SUMMARY.empty:
    report_lines += ['\n## 피해 규모 성능(원)', LOSS_SUMMARY.to_string(index=False)]
report_lines += ['\n## 최종 지수 동일목표 순차 검증', INDEX_FORWARD_SUMMARY.to_string(index=False)]
report_lines += ['\n## 손실 방법 민감도(신뢰구간 아님)', LOSS_RANGES.to_string(index=False)]
with open(os.path.join(OUT, '취합1회차_보고서대조_메모.md'), 'w', encoding='utf-8') as stream:
    stream.write('\n'.join(report_lines))


## 사용자가 넣는 강수 시나리오

실제 관측 기상으로 얻은 검증 성능과 예보 성능을 구별합니다. 예시 함수는 실행자가 넣은 강수량·시간이 실현된다고 가정합니다.
```python
scenario, status = predict_shortfall_scenario(
    SU_WITH_INFO, LOSS_EVENTS, CC, '2026-01-02', rain_mm=12., wet_hours=6., days=1)
# status, weather_extrapolation, 기준매출과 예측값을 함께 확인합니다.
```

### 지원 운영안(연구 제안)
예보는 사전 안내 신호로 사용합니다. 직접 보전에는 사업체 자격·영업 확인, 관측 강수, 실제 카드매출 감소의 사후 확인, 점포별·사건별 상한과 중복 지원 점검이 필요합니다. 신고 손실만으로 지급하지 않습니다. 골프장·할인점은 자구책·공동사업 검토 경로로 분리합니다. 관측소와 실제 점포 날씨 차이 및 손실 없는 점포 지급 위험을 기저위험으로 명시합니다. 비용 시나리오는 확정 지원액이 아니며 상한·자격 자료를 확보하면 재산정합니다.


## 부록 F. 공식 상권 자료 수집 (소상공인시장진흥공단 상가(상권)정보 API)
맨 위 `SERVICE_KEY`가 있어야 실행된다. 강남구 주요상권 21곳, 춘천시 6곳의 경계와 점포를 받는다.

In [ ]:
RUN_SANGKWON = bool(SERVICE_KEY) or os.path.exists(os.path.join(WORK, 'sangkwon', 'zones.json'))
if not RUN_SANGKWON:
    print('SERVICE_KEY가 비어 있어 이 단계는 건너뜁니다.')
else:
    import os, json, time, urllib.parse, urllib.request
    SK = os.path.join(WORK, 'sangkwon'); os.makedirs(SK, exist_ok=True)
    key = SERVICE_KEY
    key = urllib.parse.unquote(key)                    # 인코딩 키도 받아들인다
    API = 'https://apis.data.go.kr/B553077/api/open/sdsc2/'

    def call(op, **params):
        q = urllib.parse.urlencode({'serviceKey': key, 'type': 'json', **params})
        for i in range(4):
            try:
                with urllib.request.urlopen(f'{API}{op}?{q}', timeout=60) as r: body = r.read().decode('utf-8')
                return json.loads(body)
            except Exception as e:
                print(f'  재시도 {i + 1}: {op} {str(e)[:120]}'); time.sleep(3)
        raise RuntimeError(f'{op} 실패: {body[:300] if "body" in dir() else ""}')

    # 1) 주요상권 경계: 강남구 11680, 춘천시 51110(강원특별자치도 출범 후) / 42110(이전 코드)
    zf = os.path.join(SK, 'zones.json')
    if not os.path.exists(zf):
        zones = []
        for name, codes in [('강남', ['11680']), ('춘천', ['51110', '42110'])]:
            for c in codes:
                j = call('storeZoneInAdmi', divId='signguCd', key=c)
                items = (j.get('body') or {}).get('items') or []
                print(name, c, '상권', len(items), '|', j.get('header', {}).get('resultMsg'))
                if items:
                    for it in items: it['지역'] = name
                    zones += items; break
        json.dump(zones, open(zf, 'w', encoding='utf-8'), ensure_ascii=False)
    zones = json.load(open(zf, encoding='utf-8'))
    print('상권 수:', len(zones), {z['지역'] for z in zones})

    # 2) 상권별 상가업소 (1000개씩 페이지)
    for z in zones:
        f = os.path.join(SK, f"stores_{z['trarNo']}.json")
        if os.path.exists(f): continue
        out, page = [], 1
        while True:
            j = call('storeListInArea', key=z['trarNo'], numOfRows=1000, pageNo=page)
            body = j.get('body') or {}; items = body.get('items') or []
            out += items
            if len(out) >= int(body.get('totalCount', 0) or 0) or not items: break
            page += 1
        json.dump(out, open(f, 'w', encoding='utf-8'), ensure_ascii=False)
        print(f"  {z['지역']} {z['mainTrarNm']}: 점포 {len(out)}")
    print('완료')

## 부록 G. 공식 상권별 회복탄력성과 지도 (보고서 8-4절, 그림 13·14)

In [ ]:
RUN_SANGKWON = bool(SERVICE_KEY) or os.path.exists(os.path.join(WORK, 'sangkwon', 'zones.json'))
if not RUN_SANGKWON:
    print('SERVICE_KEY가 비어 있어 이 단계는 건너뜁니다.')
else:
    import os, json, glob, re
    import numpy as np, pandas as pd
    import matplotlib
    import matplotlib.pyplot as plt
    from matplotlib.patches import Polygon as MPoly
    from matplotlib.collections import PatchCollection
    from matplotlib import cm, colors

    SK = os.path.join(WORK, 'sangkwon')
    EV = FIG
    plt.rcParams.update({'axes.unicode_minus': False, 'font.size': 10, 'axes.titleweight': 'bold'})

    # 상가업소 소분류 → 카드 업종 (분석 대상 업종만. 전문서비스·의료·교육·정기결제형 등은 매핑하지 않음)
    MAP = {
        '한식': ['백반/한정식', '돼지고기 구이/찜', '국/탕/찌개류', '국수/칼국수', '닭/오리고기 구이/찜', '횟집', '해산물 구이/찜', '곱창 전골/구이',
                 '소고기 구이/찜', '족발/보쌈', '냉면/밀면', '기타 한식 음식점', '전/부침개', '복 요리 전문'],
        '커피전문점': ['카페'], '유흥업소': ['요리 주점', '일반 유흥 주점', '생맥주 전문', '무도 유흥 주점'],
        '양식': ['경양식', '파스타/스테이크', '기타 서양식 음식점', '패밀리레스토랑'],
        '일식': ['회/초밥', '일식 카레/돈가스/덮밥', '일식 면 요리', '기타 일식 음식점'], '중식': ['중국집', '마라탕/훠궈'],
        '기타요식': ['김밥/만두/분식', '베트남식 전문', '기타 동남아식 전문', '그 외 기타 간이 음식점', '토스트/샌드위치/샐러드', '떡/한과',
                     '구내식당', '뷔페', '분류 안된 외국식 음식점'],
        '패스트푸드': ['치킨', '피자', '버거'], '제과점': ['빵/도넛', '아이스크림/빙수'],
        '편의점': ['편의점'], '할인점/슈퍼마켓/양판점': ['슈퍼마켓', '그 외 기타 종합 소매업'],
        '기타식품': ['건강보조식품 소매업', '반찬/식료품 소매업', '건어물/젓갈 소매업', '곡물/곡분 소매업', '생수/음료 소매업', '아이스크림 할인점'],
        '농수산물': ['채소/과일 소매업', '수산물 소매업'], '정육점': ['정육점'], '주류판매': ['주류 소매업'], '화원': ['꽃집'],
        '애완동물': ['애완동물/애완용품 소매업'], '완구/아동용자전거': ['장난감 소매업', '자전거 소매업'], '문화용품': ['문구/회화용품 소매업'],
        '서점': ['서점'], '악기/음반': ['악기 소매업', '음반/비디오물 소매업'], '스포츠/레저용품': ['운동용품 소매업'],
        '중고품판매점': ['중고 상품 소매업'], '생활잡화/수입상품점': ['주방/가정용품 소매업', '그 외 기타 상품 전문 소매업'],
        '의복/의류': ['기타 의류 소매업', '여성 의류 소매업', '남성 의류 소매업', '유아용 의류 소매업', '한복 소매업'],
        '패션잡화': ['액세서리/잡화 소매업', '가방 소매업', '신발 소매업'], '화장품': ['화장품 소매업'], '시계/귀금속': ['시계/귀금속 소매업'],
        '미용실': ['미용실'], '미용서비스': ['피부 관리실', '네일숍', '체형/비만 관리'], '안마/마사지': ['마사지/안마'],
        '싸우나/목욕탕': ['목욕탕/사우나'], '세탁소': ['세탁소', '셀프 빨래방'], '노래방': ['노래방'],
        '게임방/오락실': ['PC방', '전자 게임장', '기타 오락장'], '실내/실외골프장': ['골프 연습장'], '모텔,여관,기타숙박': ['여관/모텔'],
    }
    S2R = {s: r for r, ss in MAP.items() for s in ss}

    zones = pd.DataFrame(json.load(open(os.path.join(SK, 'zones.json'), encoding='utf-8')))
    st = []
    for f in glob.glob(os.path.join(SK, 'stores_*.json')):
        t = pd.DataFrame(json.load(open(f, encoding='utf-8'))); t['trarNo'] = os.path.basename(f)[7:-5]; st.append(t)
    st = pd.concat(st, ignore_index=True)
    st['업종'] = st.indsSclsNm.map(S2R)
    st = st.merge(zones[['trarNo', 'mainTrarNm', '지역']], on='trarNo')

    F = pd.read_csv(os.path.join(OUT, '5-5_회복탄력성지수_순차동일공식.csv'), encoding='utf-8-sig')
    F['지역'] = F.MCT_SGG_CD.map({'서울 강남구': '강남', '강원 춘천시': '춘천'})
    idx = F.set_index(['지역', 'MCT_RY_CD'])[['회복탄력성 지수', '비 온 날 매출(%)', '신뢰도']]
    st = st.join(idx, on=['지역', '업종'])
    ok = st['회복탄력성 지수'].notna()

    rows = []
    for (tr, nm, reg), g in st.groupby(['trarNo', 'mainTrarNm', '지역']):
        m = g[g['회복탄력성 지수'].notna()]
        top = m.업종.value_counts().head(3)
        rows.append({'지역': reg, '상권번호': tr, '상권명': nm, '전체 점포': len(g), '지수 연결 점포': len(m), '범위': '선정 업종 점포 구성 추정·상권 매출 직접 관측 아님',
                     '낮은신뢰도_연결점포_pct': (m['신뢰도'].ne('보통').mean() * 100) if len(m) else np.nan,
                     '연결 비율(%)': len(m) / len(g) * 100,
                     '상권 회복탄력성': m['회복탄력성 지수'].mean(),
                     '보통신뢰도만_점수': m.loc[m['신뢰도'].eq('보통'), '회복탄력성 지수'].mean(),
                     '취약 업종 점포 비중(%)': (m['회복탄력성 지수'] <= 20).mean() * 100,
                     '비 온 날 예상 매출 변화(%)': m['비 온 날 매출(%)'].mean(),
                     '주요 업종': ', '.join(f'{k}({v})' for k, v in top.items())})
    Z = pd.DataFrame(rows).sort_values(['지역', '상권 회복탄력성'])
    Z.round(1).to_csv(os.path.join(OUT, '7-1_공식상권_회복탄력성.csv'), index=False, encoding='utf-8-sig')
    pd.set_option('display.width', 250); print(Z.round(1).to_string(index=False))
    print('점포 연결 비율:', st.groupby('지역').apply(lambda g: round(g['회복탄력성 지수'].notna().mean() * 100, 1)).to_dict())
    miss = st[st.업종.isna()].groupby('indsLclsNm').size().sort_values(ascending=False)
    print('연결 안 된 점포(대분류):', miss.head(8).to_dict())

    # 지도
    def polys(wkt):
        out = []
        for ring in re.findall(r'\(\(([^()]+)\)', wkt):
            xy = np.array([[float(a) for a in p.split()] for p in ring.split(',')]); out.append(xy)
        return out
    Zg = Z.merge(zones[['trarNo', 'coords']], left_on='상권번호', right_on='trarNo')
    cmap = colors.LinearSegmentedColormap.from_list('res', ['#B2182B', '#F4A582', '#F7F7F7', '#92C5DE', '#2166AC'])
    norm = colors.Normalize(30, 70)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6.4), gridspec_kw={'width_ratios': [1.5, 1]})
    for ax, reg in zip(axes, ['강남', '춘천']):
        g = Zg[Zg.지역 == reg]; patches, vals = [], []
        for _, r in g.iterrows():
            for xy in polys(r.coords): patches.append(MPoly(xy, closed=True)); vals.append(r['상권 회복탄력성'])
        pc = PatchCollection(patches, cmap=cmap, norm=norm, edgecolor='#444444', lw=.6); pc.set_array(np.array(vals)); ax.add_collection(pc)
        gs = g.sort_values('상권 회복탄력성')
        named = set(gs.상권명.head(3)) | set(gs.상권명.tail(2)) if len(g) > 8 else set(g.상권명)
        for _, r in g.iterrows():
            xy = np.vstack(polys(r.coords)); cx, cy = xy.mean(axis=0)
            lab = f"{r.상권명}\n{r['상권 회복탄력성']:.0f}" if r.상권명 in named else f"{r['상권 회복탄력성']:.0f}"
            ax.text(cx, cy, lab, ha='center', va='center', fontsize=7.5, fontweight='bold' if r.상권명 in named else 'normal',
                    bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=.75))
        allxy = np.vstack([np.vstack(polys(c)) for c in g.coords]); pad = .004
        ax.set_xlim(allxy[:, 0].min() - pad, allxy[:, 0].max() + pad); ax.set_ylim(allxy[:, 1].min() - pad, allxy[:, 1].max() + pad)
        ax.set_aspect(1 / np.cos(np.deg2rad(allxy[:, 1].mean()))); ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_color('#DDDDDD')
        ax.set_title(f'{reg}구 주요상권 ({len(g)}곳)' if reg == '강남' else f'{reg}시 주요상권 ({len(g)}곳)')
    cb = fig.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=axes, orientation='horizontal', fraction=.04, pad=.04, shrink=.5)
    cb.set_label('상권 기후 회복탄력성 (점포 업종 구성으로 가중 평균, 낮을수록 비에 약함)')
    fig.suptitle('공식 상권 지도로 본 기후 회복탄력성: 외식·장보기 점포가 몰린 상권이 비에 약하다 (숫자 = 상권 점수, 강남은 가장 약한 3곳·강한 2곳만 이름 표시)', fontsize=11)
    plt.savefig(os.path.join(EV, '그림13_공식상권_지도.png'), dpi=170, bbox_inches='tight', facecolor='white'); (plt.show() if IN_COLAB else None); plt.close('all')

    # 상권 순위 막대
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.6), gridspec_kw={'width_ratios': [1.6, 1]})
    for ax, reg in zip(axes, ['강남', '춘천']):
        g = Z[Z.지역 == reg].sort_values('상권 회복탄력성', ascending=False)
        ax.barh(range(len(g)), g['상권 회복탄력성'], color=[cmap(norm(v)) for v in g['상권 회복탄력성']], edgecolor='#666666', lw=.5)
        ax.set_yticks(range(len(g))); ax.set_yticklabels(g.상권명, fontsize=8.5); ax.axvline(50, color='#999', ls=':', lw=1)
        for i, (v, p) in enumerate(zip(g['상권 회복탄력성'], g['취약 업종 점포 비중(%)'])): ax.text(v + .5, i, f'{v:.0f} (취약 점포 {p:.0f}%)', va='center', fontsize=7.5)
        ax.set_xlim(0, 85); ax.set_title(reg); ax.set_xlabel('상권 회복탄력성 (낮을수록 우선 지원)'); ax.grid(axis='x', alpha=.2)
        for s in ['top', 'right']: ax.spines[s].set_visible(False)
    fig.suptitle('공식 상권별 순위: 위쪽 상권부터 비 예보 시 우선 지원 (괄호 = 우선지원 등급 업종 점포 비중)', fontsize=11)
    plt.tight_layout(); plt.savefig(os.path.join(EV, '그림14_공식상권_순위.png'), dpi=170, bbox_inches='tight', facecolor='white'); (plt.show() if IN_COLAB else None); plt.close('all')
    print('그림 완료')

## 부록 H. 통신(SKT 유동인구) × 공식 상권 (보고서 8-5절, 그림 15)
유동인구는 월별 자료다(12월 파일의 중복 격자는 지운다). 비 효과를 직접 재지 않고 상권의 노출 규모와 성격을 잰다.

In [ ]:
RUN_SANGKWON = bool(SERVICE_KEY) or os.path.exists(os.path.join(WORK, 'sangkwon', 'zones.json'))
if not RUN_SANGKWON:
    print('SERVICE_KEY가 비어 있어 이 단계는 건너뜁니다.')
else:
    import os, re, json, glob
    import numpy as np, pandas as pd
    from scipy import stats
    from pyproj import Transformer
    from matplotlib.path import Path
    import matplotlib
    import matplotlib.pyplot as plt

    SK = os.path.join(WORK, 'sangkwon')
    EV = FIG
    plt.rcParams.update({'axes.unicode_minus': False, 'font.size': 10, 'axes.titleweight': 'bold',
                         'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .2})
    COLR = {'강남': '#C8553D', '춘천': '#2B6CB0'}
    SKT = glob.glob(os.path.join(RAW, '*sk통신인구*'))[0]; KMA = glob.glob(os.path.join(RAW, 'kma_data*'))[0]

    zones = pd.DataFrame(json.load(open(os.path.join(SK, 'zones.json'), encoding='utf-8')))
    ZP = {r.trarNo: [Path(np.array([[float(a) for a in p.split()] for p in ring.split(',')])) for ring in re.findall(r'\(\(([^()]+)\)', r.coords)]
          for r in zones.itertuples()}
    score = pd.read_csv(os.path.join(OUT, '7-1_공식상권_회복탄력성.csv'), encoding='utf-8-sig', dtype={'상권번호': str}).set_index('상권번호')
    tf = Transformer.from_crs(5179, 4326, always_xy=True)

    def load(kind, ym):
        INPUT_FILES.append(os.path.join(SKT, f'flow_{kind}_pop_{ym}.csv'))
        d = pd.read_csv(os.path.join(SKT, f'flow_{kind}_pop_{ym}.csv'), sep='|', encoding='utf-8-sig', low_memory=False)
        return d.drop_duplicates(['X_COORD', 'Y_COORD']).reset_index(drop=True)
    def assign(d):
        lon, lat = tf.transform(d.X_COORD.values, d.Y_COORD.values); pts = np.c_[lon, lat]
        lab = np.full(len(d), None, dtype=object)
        for tr, paths in ZP.items():
            inside = np.zeros(len(d), bool)
            for p in paths: inside |= p.contains_points(pts)
            lab[inside] = tr
        return lab

    rows = []
    for ym in ['202507', '202508', '202509', '202510', '202511', '202512']:
        t, w, a = load('time', ym), load('wkdy', ym), load('age', ym)
        t['z'] = assign(t); w['z'] = assign(w); a['z'] = assign(a)
        t = t[t.z.notna()]; w = w[w.z.notna()]; a = a[a.z.notna()]
        T = t.groupby('z').agg(**{'영업시간 유동': ('z', 'size')}).astype(float)
        T['영업시간 유동'] = t.groupby('z')[[f'TMST_{h:02d}' for h in range(9, 22)]].sum().sum(axis=1)
        T['하루 유동'] = t.groupby('z')[[f'TMST_{h:02d}' for h in range(24)]].sum().sum(axis=1)
        T['저녁 유동'] = t.groupby('z')[[f'TMST_{h:02d}' for h in range(18, 23)]].sum().sum(axis=1)
        wd = w.groupby('z')[['FLOW_POP_CNT_MON', 'FLOW_POP_CNT_TUS', 'FLOW_POP_CNT_WED', 'FLOW_POP_CNT_THU', 'FLOW_POP_CNT_FRI']].sum().mean(axis=1)
        we = w.groupby('z')[['FLOW_POP_CNT_SAT', 'FLOW_POP_CNT_SUN']].sum().mean(axis=1)
        T['주말/평일'] = we / wd
        ac = [c for c in a.columns if 'FLOW_POP_CNT' in c]
        T['20~30대 비중'] = a.groupby('z')[[c for c in ac if c.endswith(('20G', '30G'))]].sum().sum(axis=1) / a.groupby('z')[ac].sum().sum(axis=1)
        T['월'] = int(ym[4:]); rows.append(T.reset_index().rename(columns={'z': '상권번호'}))
    FL = pd.concat(rows, ignore_index=True).merge(zones[['trarNo', 'mainTrarNm', '지역']], left_on='상권번호', right_on='trarNo').drop(columns='trarNo')

    # ① 상권 특성(6개월 평균)과 카드 기반 점수
    P = FL.groupby(['지역', '상권번호', 'mainTrarNm']).agg(**{'영업시간 유동(일평균)': ('영업시간 유동', 'mean'), '저녁 비중': ('저녁 유동', 'sum'),
                                                           '하루': ('하루 유동', 'sum'), '주말/평일': ('주말/평일', 'mean'), '20~30대 비중': ('20~30대 비중', 'mean')}).reset_index()
    P['저녁 비중'] = P['저녁 비중'] / P.pop('하루')
    P = P.join(score[['상권 회복탄력성', '취약 업종 점포 비중(%)', '전체 점포']], on='상권번호')
    res = []
    for reg, g in P.groupby('지역'):
        for c in ['영업시간 유동(일평균)', '주말/평일', '저녁 비중', '20~30대 비중']:
            r, p = stats.spearmanr(g['상권 회복탄력성'], g[c]); res.append({'지역': reg, '특성': c, '점수와 순위상관': r, 'p': p})
    R1 = pd.DataFrame(res); print(R1.round(3).to_string(index=False))
    # 우선지원 구분: 점수 지역 중앙값 미만(약함) × 영업시간 유동 지역 중앙값 이상(사람 많음)
    P['약함'] = P['상권 회복탄력성'] < P.groupby('지역')['상권 회복탄력성'].transform('median')
    P['붐빔'] = P['영업시간 유동(일평균)'] >= P.groupby('지역')['영업시간 유동(일평균)'].transform('median')
    P['구분'] = np.select([P.약함 & P.붐빔, P.약함, P.붐빔], ['1순위(약하고 사람 많음)', '선별(약하지만 한산)', '규모(덜 약하지만 사람 많음)'], '관찰')
    P.round(4).to_csv(os.path.join(OUT, '8-1_유동인구_상권특성.csv'), index=False, encoding='utf-8-sig')
    print(P.sort_values(['지역', '상권 회복탄력성'])[['지역', 'mainTrarNm', '상권 회복탄력성', '영업시간 유동(일평균)', '주말/평일', '구분']].round(2).to_string(index=False))

    # ② 비 오는 달: 지역 공통 흐름과 상권 고유 수준을 뺀 편차를 월별 비 온 날 비율에 회귀
    def rain_days(fname, col):
        h = pd.read_csv(os.path.join(KMA, fname), encoding='utf-8-sig', parse_dates=['TM']); r = h[col].where(h[col] >= 0, 0).fillna(0)
        d = pd.DataFrame({'date': (h.TM - pd.Timedelta(minutes=1)).dt.normalize(), 'hr': h.TM.dt.hour, 'rn': r})
        b = d[d.hr.between(10, 22)].groupby('date').rn.sum(); b = b[(b.index >= '2025-07-01') & (b.index <= '2025-12-31')]
        return pd.DataFrame({'비 온 날 비율': (b >= 5).groupby(b.index.month).mean()})
    RD = pd.concat({'강남': rain_days('gangnam_aws_hourly.csv', 'RN_HR1'), '춘천': rain_days('chuncheon_asos_hourly.csv', 'RN')}, names=['지역', '월']).reset_index()
    F = FL.merge(RD, on=['지역', '월']).join(score[['상권 회복탄력성']], on='상권번호')
    F['ly'] = np.log(F['영업시간 유동'])
    F['dev'] = F.ly - F.groupby(['지역', '월']).ly.transform('mean'); F['dev'] = F.dev - F.groupby('상권번호').dev.transform('mean')
    out = []
    for lab, months in [('6개월 전체', [7, 8, 9, 10, 11, 12]), ('8월(휴가) 제외', [7, 9, 10, 11, 12]), ('8월·11~12월(쇼핑철) 제외', [7, 9, 10])]:
        g = F[F.월.isin(months)].copy()
        g['dev'] = g.dev - g.groupby('상권번호').dev.transform('mean')
        b = g.groupby(['지역', '상권번호']).apply(lambda x: np.polyfit(x['비 온 날 비율'], x.dev, 1)[0], include_groups=False).rename('b').reset_index().join(score[['상권 회복탄력성']], on='상권번호')
        for reg, h in b.groupby('지역'):
            r, p = stats.spearmanr(h['상권 회복탄력성'], h.b); out.append({'표본': lab, '지역': reg, '점수 vs 비 오는 달 유동 민감도': r, 'p': p})
    # 계절 요인 점검: 8월(휴가철)과 11~12월(쇼핑철)의 상대 유동이 점수와 어떤 관계인가
    for lab, months in [('8월 상대 유동', [8]), ('11~12월 상대 유동', [11, 12])]:
        s = F[F.월.isin(months)].groupby(['지역', '상권번호']).dev.mean().rename('d').reset_index().join(score[['상권 회복탄력성']], on='상권번호')
        for reg, h in s.groupby('지역'):
            r, p = stats.spearmanr(h['상권 회복탄력성'], h.d); out.append({'표본': lab, '지역': reg, '점수 vs 비 오는 달 유동 민감도': r, 'p': p})
    R2 = pd.DataFrame(out); print('월별 통신 전체기간 결과를 먼저 봅니다. 일부 기간 제외 결과는 참고이며 일별 강수 검증이 아닙니다.'); print(R2.round(3).to_string(index=False))
    R1.round(3).to_csv(os.path.join(OUT, '8-2_유동인구_특성_vs_점수.csv'), index=False, encoding='utf-8-sig')
    R2.round(3).to_csv(os.path.join(OUT, '8-3_유동인구_비오는달_점검.csv'), index=False, encoding='utf-8-sig')
    F.round(4).to_csv(os.path.join(OUT, '8-0_유동인구_상권월별.csv'), index=False, encoding='utf-8-sig')

    # 그림 15: 상권 우선지원 매트릭스 (카드 점수 × 통신 유동 규모)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), gridspec_kw={'width_ratios': [1.5, 1]})
    for ax, reg in zip(axes, ['강남', '춘천']):
        g = P[P.지역 == reg]; xm, ym = g['상권 회복탄력성'].median(), g['영업시간 유동(일평균)'].median()
        top = g['영업시간 유동(일평균)'].max() / 1000 * 1.1
        ax.fill_between([g['상권 회복탄력성'].min() - 2, xm], ym / 1000, top, color='#FBE3D6', zorder=0); ax.set_ylim(0, top)
        ax.scatter(g['상권 회복탄력성'], g['영업시간 유동(일평균)'] / 1000, s=np.clip(g['전체 점포'] / 8, 20, 400), color=COLR[reg], alpha=.75, zorder=3, edgecolor='white')
        ax.axhline(ym / 1000, color='#999', ls='--', lw=1); ax.axvline(xm, color='#999', ls='--', lw=1)
        for _, r in g.iterrows():
            if r.구분.startswith('1순위') or len(g) < 8 or r['상권 회복탄력성'] > g['상권 회복탄력성'].quantile(.9):
                ax.annotate(r.mainTrarNm, (r['상권 회복탄력성'], r['영업시간 유동(일평균)'] / 1000), fontsize=8, xytext=(5, 3 if r.mainTrarNm != '우석중학교' else 12), textcoords='offset points',
                            fontweight='bold' if r.구분.startswith('1순위') else 'normal')
        ax.set(xlabel='상권 기후 회복탄력성 (카드, 낮을수록 비에 약함)', ylabel='영업시간 유동인구 (SKT, 하루 평균, 천 명·시간)', title=f'{reg}')
        ax.text(.02, .97, '왼쪽 위 = 1순위\n(비에 약하고 사람도 많음)', transform=ax.transAxes, va='top', fontsize=8.5, color='#8A3B1F')
    fig.suptitle('카드(얼마나 약한가) × 통신(얼마나 많은 사람이 오가는가)로 먼저 도울 공식 상권을 고른다 (원 크기 = 점포 수)', fontsize=11)
    plt.tight_layout(); plt.savefig(os.path.join(EV, '그림15_상권_카드x통신.png'), dpi=170, bbox_inches='tight', facecolor='white'); (plt.show() if IN_COLAB else None); plt.close('all')
    print('완료')

## 부록 I. 보고서 그림 다시 만들기 (그림 0~12, 부록 그림 1)
결과는 `report_figs` 폴더에 저장된다.

In [ ]:
import os, json, shutil
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

CLI = os.path.join(WORK, 'climate')
EV = os.path.join(WORK, 'report_figs'); os.makedirs(EV, exist_ok=True)
plt.rcParams.update({'axes.unicode_minus': False, 'figure.dpi': 110, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .2,
                     'axes.titleweight': 'bold', 'axes.titlesize': 11})
C = {'강남': '#C8553D', '춘천': '#2B6CB0'}                         # 검증 통과 (CVD ΔE 18.0)
G = {'핵심 합계': '#222222', '방문·선택형': '#D35F1F', '생활밀착형': '#2B6CB0'}
INK2 = '#555555'
def save(name):
    plt.tight_layout(); plt.savefig(os.path.join(EV, name + '.png'), dpi=170, bbox_inches='tight', facecolor='white'); (plt.show() if IN_COLAB else None); plt.close('all')
def rd(f): return pd.read_csv(os.path.join(OUT, f), encoding='utf-8-sig')

# 그림 1. 이야기의 흐름
steps = [('가설', '폭염이 상권을\n흔들 것이다', '#EEEEEE'), ('검증', '비와 더위의 효과를\n분리해서 비교', '#EEEEEE'),
         ('발견', '매출을 깎은 건\n영업시간의 비', '#FBE3D6'), ('확인', '관측 강수 기준과\n장기 추세를 비교', '#FBE3D6'),
         ('측정', '관측 매출 감소와\n연간 시나리오', '#DCE8F5'), ('해결', '회복탄력성 지수로\n먼저 도울 곳을 고른다', '#DCE8F5')]
fig, ax = plt.subplots(figsize=(13, 2.6)); ax.set_xlim(0, 6); ax.set_ylim(0, 1); ax.axis('off')
for i, (t, s, c) in enumerate(steps):
    ax.add_patch(FancyBboxPatch((i + .06, .12), .82, .76, boxstyle='round,pad=0.02,rounding_size=0.06', fc=c, ec='#999999', lw=1))
    ax.text(i + .47, .72, t, ha='center', va='center', fontsize=12, fontweight='bold')
    ax.text(i + .47, .4, s, ha='center', va='center', fontsize=10)
    if i < 5: ax.annotate('', xy=(i + 1.05, .5), xytext=(i + .9, .5), arrowprops=dict(arrowstyle='-|>', color='#666666', lw=1.5))
for x0, x1, lab in [(0, 2, '목차 1~3: 문제와 방향 전환'), (2, 4, '목차 4: 기후 반응'), (4, 6, '목차 5~8: 피해·회복·지수·활용')]:
    ax.text((x0 + x1) / 2 + .0, .02, lab, ha='center', va='bottom', fontsize=9, color=INK2)
save('그림01_이야기흐름')

# 그림 2. 더위 vs 비
h = rd('1-4_더위_반응.csv'); r = rd('1-3_비_효과_그룹별.csv')
h = h[(h.그룹 == '핵심 합계') & (h['방학 통제'] == '포함')].set_index('지역'); r = r[r.그룹 == '핵심 합계'].set_index('지역')
fig, ax = plt.subplots(figsize=(7.5, 4))
for i, reg in enumerate(['강남', '춘천']):
    vals = [h.loc[reg, '폭염일(%)'], r.loc[reg, '매출(%)']]; ps = [h.loc[reg, '폭염 p'], r.loc[reg, 'p']]
    x = np.arange(2) + (i - .5) * .36
    ax.bar(x, vals, width=.34, color=C[reg], label=reg)
    for xx, v, p in zip(x, vals, ps):
        ax.text(xx, v - .6, f'{v:+.1f}%\n' + ('유의함' if p < .05 else '유의하지 않음'), ha='center', va='top', fontsize=9)
ax.axhline(0, color='k', lw=.8); ax.set_xticks(range(2)); ax.set_xticklabels(['폭염일\n(일최고 33℃ 이상)', '비 온 날\n(영업시간 강수 5mm 이상)'])
ax.set(ylabel='평범한 날 대비 핵심 업종 매출 (%)', ylim=(-17, 2)); ax.legend(frameon=False)
ax.set_title('선정 업종에서 추정한 폭염·강수일의 매출 변화')
save('그림02_더위vs비')

# 그림 3. 강수량 세분 구간별 반응 (6-1) + 강수량 2배당 효과 (6-2)
q = rd('6-1_강수량_세분구간.csv'); bins = ['0~2', '2~5', '5~10', '10~20', '20~40', '40+']
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), sharey=True)
for ax, reg in zip(axes, ['춘천', '강남']):
    for grp, colr in G.items():
        s_ = q[(q.지역 == reg) & (q.그룹 == grp)].set_index('영업시간 강수(mm)').reindex(bins)
        x = np.arange(7); yv = np.r_[0, s_.효과.values]
        ax.plot(x, yv, marker='o', color=colr, lw=2.6 if grp == '핵심 합계' else 1.5, ms=6, label=grp)
        sig = np.r_[False, (s_.p < .05).values]; ax.scatter(x[sig], yv[sig], s=110, facecolor='none', edgecolor=colr, lw=1.2)
    n = q[(q.지역 == reg) & (q.그룹 == '핵심 합계')].set_index('영업시간 강수(mm)').reindex(bins).일수.astype(int).values
    ax.set_xticks(range(7)); ax.set_xticklabels(['없음'] + [f'{b_}\n({k}일)' for b_, k in zip(bins, n)], fontsize=9)
    ax.axhline(0, color='k', lw=.8); ax.axvline(2.5, color='#999', ls='--', lw=1); ax.text(2.55, 6, '5mm', fontsize=9, color=INK2)
    ax.set_title(reg); ax.set_xlabel('영업시간(09~22시) 강수량 (mm)')
axes[0].set_ylabel('비 없는 날 대비 매출 (%)'); axes[0].legend(frameon=False, loc='lower left')
fig.suptitle('강수 구간별 매출 반응 (5mm는 공통 실무 기준, 빈 원 = 유의)', fontsize=10.5)
save('그림03_강수량별_반응')

# 그림 4. 강수 임계값 (노트북 그림)
shutil.copy(os.path.join(FIG, '1-2_강수_임계값.png'), os.path.join(EV, '그림04_강수_임계값.png'))

# 그림 5. 오래 오는 비 vs 세게 오는 비
d = rd('1-5_지속vs강도.csv'); d = d[d.그룹.isin(list(G))]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, (col, pc, lab) in zip(axes, [('비 1시간 더(%)', 'p_시간', '같은 양의 비가 1시간 더 오래 올 때'), ('강도 1mm/h 더(%)', 'p_강도', '같은 양의 비가 시간당 1mm 더 세게 올 때')]):
    for i, reg in enumerate(['강남', '춘천']):
        s = d[d.지역 == reg].set_index('그룹').reindex(list(G))
        x = np.arange(3) + (i - .5) * .36
        ax.bar(x, s[col], width=.34, color=C[reg], label=reg, edgecolor=['k' if p < .05 else 'none' for p in s[pc]], lw=1.6)
    ax.axhline(0, color='k', lw=.8); ax.set_xticks(range(3)); ax.set_xticklabels(list(G)); ax.set_title(lab, fontsize=10)
axes[0].set_ylabel('매출 변화 (%)'); axes[0].legend(frameon=False)
fig.suptitle('지속시간과 강도에 따른 매출 반응 (총량 통제 / 검은 테두리 = 유의)', fontsize=10.5)
save('그림05_지속vs강도')

# 그림 6. 재난 기준으로 보이는 손실
L = rd('5-1_재난기준별_손실.csv')
cats = ['영업시간 0~5mm', '영업시간 5mm 이상(특보·30mm 미만)', '일강수 30mm 이상(주의보 미만)']
labs = ['약한 비\n(영업시간 0~5mm)', '보통 비\n(5mm 이상, 일 30mm 미만)', '많은 비\n(일 30mm 이상, 주의보 미만)']
cols = ['#BFD3EA', '#6C9BD2', '#2B6CB0']
fig, ax = plt.subplots(figsize=(11, 3.2))
for j, reg in enumerate(['강남', '춘천']):
    s = L[L.MCT_SGG_CD == reg].set_index('기준').reindex(cats); left = 0
    for k, (cat, colr) in enumerate(zip(cats, cols)):
        w = s.loc[cat, '손실비중(%)']; ax.barh(j, w, left=left, color=colr, edgecolor='white', lw=2, height=.6)
        ax.text(left + w / 2, j, f"{w:.0f}%\n{s.loc[cat, '손실억']:.0f}억 · {int(s.loc[cat, '일수'])}일", ha='center', va='center', fontsize=9,
                color='white' if k == 2 else '#222222')
        left += w
    ax.text(101, j, f"합계 {s.손실억.sum():.0f}억", va='center', fontsize=9.5, fontweight='bold')
ax.set_yticks([0, 1]); ax.set_yticklabels(['강남', '춘천']); ax.set_xlim(0, 112); ax.set_xlabel('6개월(2025.7~12) 비로 사라진 핵심 업종 매출의 비중 (%)')
ax.grid(False); ax.set_xticks([0, 25, 50, 75, 100])
for k, (lab, colr) in enumerate(zip(labs, cols)): ax.bar(0, 0, color=colr, label=lab)
ax.legend(frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(.45, -.32), fontsize=9)
ax.set_title('관측소 강수 수치별 매출 감소 추정 (실제 호우특보 발효 이력과 다름)')
save('그림06_재난기준별_손실')

# 그림 7. 분야별 순손실
u = rd('2-1_유닛별_비효과.csv'); u['지역'] = u.MCT_SGG_CD.map({'서울 강남구': '강남', '강원 춘천시': '춘천'})
p = u.groupby(['분야', '지역'])['원화(억)'].sum().unstack(); order = p.sum(axis=1).sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(8.5, 4.2)); yy = np.arange(len(order))
for i, reg in enumerate(['강남', '춘천']):
    ax.barh(yy + (i - .5) * .38, p.loc[order, reg], height=.36, color=C[reg], label=f"{reg} (합계 {p[reg].sum():.0f}억)")
ax.set_yticks(yy); ax.set_yticklabels(order); ax.axvline(0, color='k', lw=.8)
ax.set(xlabel='비 온 날 순변화 (억 원, 2025년 7~12월)'); ax.legend(frameon=False, loc='lower left')
ax.set_title('선정 업종 내 감소·증가 추정 (다른 카드·채널 이동은 미관측)')
save('그림07_분야별_순손실')

# 그림 8. 회복
q = rd('3-2_그친뒤_여파.csv')
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, reg in zip(axes, ['춘천', '강남']):
    for grp, colr in G.items():
        s = q[(q.지역 == reg) & (q.그룹 == grp)].set_index('시점').reindex(['당일', '+1일', '+2일', '+3일'])
        ax.plot(range(4), s.효과, marker='o', color=colr, lw=2.4 if grp == '핵심 합계' else 1.6, label=grp)
        sig = (s.p < .05).values; ax.scatter(np.arange(4)[sig], s.효과.values[sig], s=110, facecolor='none', edgecolor=colr)
    ax.axhline(0, color='k', lw=.8); ax.set_xticks(range(4)); ax.set_xticklabels(['비 온 날', '그친 뒤 1일', '2일', '3일'])
    ax.set_title(reg)
axes[0].set_ylabel('평범한 날 대비 매출 (%)'); axes[0].legend(frameon=False, loc='lower right')
fig.suptitle('비 사건 종료 후 1~3일 매출 변화 (빈 원 = 유의, 이후 소비 이동은 미관측)', fontsize=10.5)
save('그림08_회복')

# 그림 9. 기후 기록과 전망
y = pd.read_csv(os.path.join(CLI, 'era5_annual.csv'), encoding='utf-8-sig')
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={'width_ratios': [1.6, 1]})
ax = axes[0]
for site, reg in [('chuncheon', '춘천'), ('gangnam', '강남')]:
    s = y[y.site == site].set_index('date').rain5
    ax.plot(s.index, s, color=C[reg], lw=.8, alpha=.45); ax.plot(s.index, s.rolling(10, center=True, min_periods=5).mean(), color=C[reg], lw=2.4, label=f'{reg} (10년 이동평균)')
for yr in (2020, 2023): ax.axvline(yr, color='#999', ls=':', lw=1); ax.text(yr, ax.get_ylim()[1] if False else 57, str(yr), ha='center', fontsize=8, color=INK2)
ax.set(title='영업시간 5mm 이상 비 온 날 (연간, ERA5 1981~2025)', ylabel='일수', ylim=(15, 60)); ax.legend(frameon=False)
ax = axes[1]; rows = []
for site, reg in [('chuncheon', '춘천'), ('gangnam', '강남')]:
    j = json.load(open(os.path.join(CLI, f'cmip6_{site}.json'))); dd = pd.DataFrame(j['daily']); dd['time'] = pd.to_datetime(dd.time); dd = dd.set_index('time')
    for c in dd.columns:
        s = dd[c].dropna(); g = s.groupby(s.index.year); n5, n30 = g.apply(lambda v: (v >= 5).sum()), g.apply(lambda v: (v >= 30).sum())
        rows.append({'지역': reg, 'a': (n5.loc[2041:2050].mean() / n5.loc[1991:2020].mean() - 1) * 100, 'b': (n30.loc[2041:2050].mean() / n30.loc[1991:2020].mean() - 1) * 100})
t = pd.DataFrame(rows)
for i, reg in enumerate(['강남', '춘천']):
    s = t[t.지역 == reg]
    for j_, col in enumerate(['a', 'b']):
        x = j_ + (i - .5) * .36; ax.scatter(np.full(len(s), x), s[col], color=C[reg], s=20, alpha=.6)
        ax.hlines(s[col].median(), x - .15, x + .15, color=C[reg], lw=2.6, label=reg if j_ == 0 else None)
ax.axhline(0, color='k', lw=.8); ax.set_xticks([0, 1]); ax.set_xticklabels(['비 온 날\n(일 5mm 이상)', '강한 비 온 날\n(일 30mm 이상)'])
ax.set(title='2041~2050년 전망 (기후모델 7개)', ylabel='1991~2020년 대비 (%)'); ax.legend(frameon=False)
fig.suptitle('영업시간 강수의 과거 기록과 일강수 미래 시나리오 (서로 다른 기준)', fontsize=10.5)
save('그림09_기후기록과전망')

# 그림 10. 연간 손실 시나리오
a = rd('5-2_연간손실_시나리오.csv'); sc = ['최근 10년 평균(2016~2025)', '2020년', '2023년', '45년 최대']
fig, ax = plt.subplots(figsize=(9, 3.8)); x = np.arange(len(sc))
for i, reg in enumerate(['강남', '춘천']):
    s = a[a.지역 == reg].set_index('시나리오').reindex(sc)
    ax.bar(x + (i - .5) * .38, s['연간 손실(억)'], .36, color=C[reg], label=reg)
    for xx, v in zip(x + (i - .5) * .38, s['연간 손실(억)']): ax.text(xx, v + 8, f'{v:.0f}', ha='center', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(['평년\n(최근 10년 평균)', '2020년', '2023년', '45년 중\n가장 비가 많던 해']); ax.set_ylabel('연간 손실 (억 원)')
ax.legend(frameon=False); ax.set_title('연간 시나리오: 하반기 하루 감소액 × 보정된 연간 강수일 (계절효과 고정)')
save('그림10_연간손실_시나리오')

# 그림 11. 최종 지수와 동일한 목표·산출 절차의 순차 검증
check_index = LOSS_INDEX_FORWARD[LOSS_INDEX_FORWARD.status.eq('ok')].sort_values(['start', 'event'])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
if check_index.empty:
    for ax in axes:
        ax.text(.5, .5, '과거 사건·관측 업종 부족으로 평가 불가', ha='center', transform=ax.transAxes)
else:
    x = np.arange(len(check_index))
    labels = check_index.event.astype(str).tolist()
    axes[0].bar(x, check_index.rho, color='#2B6CB0')
    axes[0].axhline(0, color='k', lw=.8)
    axes[0].set(ylim=(-1, 1), ylabel='Spearman ρ', title='사건별 표준화 종합피해 순위상관')
    axes[1].bar(x, check_index.top20_hit * 100, color='#2B6CB0')
    axes[1].plot(x, check_index.random_top20_hit * 100, 'k:', label='동일 표본 무작위 기준')
    axes[1].set(ylim=(0, 105), ylabel='적중률 (%)', title='피해 상위 20%의 동점 보정 적중률')
    axes[1].legend(fontsize=8)
    for ax in axes:
        ax.set_xticks(x); ax.set_xticklabels(labels, rotation=55, ha='right', fontsize=7)
fig.suptitle('과거만으로 기준선·노출·가중치를 산출한 최종 지수의 순차 검증\n평균과 기상 블록 부트스트랩 구간: 10-7 결과표', fontsize=10.5)
save('그림11_지수검증')

# 그림 12. 우선지원 매트릭스
SHORTN = lambda n: {'실내/실외골프장': '골프장', '할인점/슈퍼마켓/양판점': '할인점·슈퍼', '종합레저타운/놀이동산': '놀이동산', '생활잡화/수입상품점': '생활잡화'}.get(n, n.split('/')[0])
F = pd.read_csv(os.path.join(OUT, '5-5_회복탄력성지수_순차동일공식.csv'), encoding='utf-8-sig'); F = F[F.신뢰도.eq('보통') & ~F.MCT_RY_CD.isin(DIRECT_EXCLUDED_RY)]
F['지역'] = F.MCT_SGG_CD.map({'서울 강남구': '강남', '강원 춘천시': '춘천'}); F['손실'] = -F['원화(억)'].clip(upper=0)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))
for ax, reg in zip(axes, ['춘천', '강남']):
    s = F[F.지역 == reg]
    if s.empty:
        ax.text(.5, .5, '충분한 과거 사건·신뢰도 있는 지수 없음', ha='center', transform=ax.transAxes)
        ax.set_title(reg)
        continue
    q80 = s.손실.quantile(.8); top = max(1., s.손실.max() * 1.15)
    ax.axvspan(0, 20, color='#FBE3D6', zorder=0); ax.axhspan(q80, top, xmin=20/102, color='#EFEFEF', zorder=0)
    ax.axhspan(q80, top, xmax=20/102, color='#F5C6AE', zorder=0)
    ax.scatter(s['회복탄력성 지수'], s.손실, s=38, color=C[reg], alpha=.85, zorder=3)
    for _, rr in s[s.손실 >= q80].iterrows():
        ax.annotate(SHORTN(rr.MCT_RY_CD), (rr['회복탄력성 지수'], rr.손실), fontsize=8.5, xytext=(5, 2), textcoords='offset points')
    sel = s[(s['회복탄력성 지수'] <= 20) & (s.손실 < q80)].sort_values('회복탄력성 지수').MCT_RY_CD.map(SHORTN)
    ax.text(.98, .55, '선별 지원 대상 (분홍 아래칸)\n' + '\n'.join(sel), transform=ax.transAxes, fontsize=8.5, color='#333333', va='center', ha='right',
            bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='#DDDDDD'))
    ax.set(xlim=(0, 102), ylim=(0, top), xlabel='기후 회복탄력성 지수 (낮을수록 비에 약함)', ylabel='강수 관련 매출 감소 추정 (억 원, 사후)', title=reg)
from matplotlib.patches import Patch
fig.legend(handles=[Patch(color='#F5C6AE', label='1순위: 약하고 손실도 큼'), Patch(color='#FBE3D6', label='선별 검토: 약하지만 감소액 작음 (사업체 자격 확인 필요)'),
                    Patch(color='#EFEFEF', label='규모 지원: 덜 약하지만 손실 큼 (상권 단위 소비 촉진)')], loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(.5, -.04))
fig.suptitle('지수(얼마나 약한가) × 손실(얼마나 큰가)로 지원 대상과 방식을 나눈다 (기준: 지수 하위 20%, 손실 상위 20%)', fontsize=10.5)
plt.tight_layout(rect=(0, .05, 1, 1)); plt.savefig(os.path.join(EV, '그림12_우선지원_매트릭스.png'), dpi=170, bbox_inches='tight', facecolor='white'); (plt.show() if IN_COLAB else None); plt.close('all')

# 부록 그림
shutil.copy(os.path.join(FIG, '1-4_더위_기울기.png'), os.path.join(EV, '부록1_기온_1도당_변화.png'))
print(sorted(os.listdir(EV)))

# 그림 0. 분석 대상 깔때기
steps0 = list(zip(funnel['단계'], funnel['유닛']))
fig, ax = plt.subplots(figsize=(9, 3.4)); yy = np.arange(len(steps0))[::-1]
for (lab, n), y_ in zip(steps0, yy):
    last = lab.startswith('정기')
    ax.barh(y_, n, color='#2B6CB0' if last else '#BFD3EA', height=.62)
    ax.text(n + 2, y_, f'{n}개', va='center', fontsize=10, fontweight='bold' if last else 'normal')
ax.set_yticks(yy); ax.set_yticklabels([s for s, _ in steps0]); ax.set_xlim(0, 200); ax.grid(False); ax.set_xticks([])
ax.spines['bottom'].set_visible(False)
ax.set_title(f'당일 결제 해석이 가능한 선정 업종 (최종 {len(units)}개, 분석 제외 업종으로 일반화하지 않음)')
save('그림00_분석대상')
print('그림00 완료')

# 마지막 단계. 재현 기록과 제출 묶음
실제 입력 해시, 패키지 버전, 실행 옵션, 결과 파일 대응표를 저장합니다. 이 셀까지 실자료로 실행한 다음 출력이 들어 있는 노트북을 다운로드합니다. 합성 검증은 실자료 실행을 대신하지 않습니다.


In [ ]:
# 재현 기록: 비밀 인증키는 저장하지 않는다. 출력이 없으면 성능도 기록하지 않는다.
import hashlib, platform, importlib.metadata, zipfile

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

packages = {}
for package in ['numpy', 'pandas', 'scipy', 'statsmodels', 'matplotlib', 'pyproj', 'openpyxl']:
    try:
        packages[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        packages[package] = 'not_installed'
climate_inputs = glob.glob(os.path.join(WORK, 'climate', 'era5_*.json')) + glob.glob(os.path.join(WORK, 'climate', 'cmip6_*.json'))
inputs = []
for path in sorted(set(INPUT_FILES + climate_inputs)):
    if os.path.isfile(path):
        inputs.append({'name': os.path.basename(path), 'bytes': os.path.getsize(path), 'sha256': file_sha256(path)})
outputs = []
for path in sorted(glob.glob(os.path.join(OUT, '*'))):
    if os.path.isfile(path) and not path.endswith('실행기록.json'):
        outputs.append({'name': os.path.basename(path), 'bytes': os.path.getsize(path), 'sha256': file_sha256(path)})
manifest = {'code_version': LOSS_MODEL_VERSION, 'executed_at': pd.Timestamp.now(tz='Asia/Seoul').isoformat(),
    'data_context': os.environ.get('BIGCON_DATA_CONTEXT', 'user_supplied_data_not_independently_verified'),
    'python': platform.python_version(), 'packages': packages, 'inputs': inputs, 'outputs': outputs,
    'calendar_long_break': USE_LONG_BREAK, 'rain_business_threshold_mm': RAIN_BIZ_MM,
    'loss_min_prior_days': LOSS_MIN_CALM_DAYS, 'loss_min_train_clusters': LOSS_MIN_TRAIN_CLUSTERS,
    'ridge_alpha_fixed': LOSS_RIDGE_ALPHA, 'final_index_weight_name': W_NAME, 'final_index_weights': W,
    'index_target': INDEX_TARGET, 'index_as_of': str(INDEX_ORIGIN),
    'index_validation': 'nested_forward_same_procedure_and_target',
    'reference_diagnostic': 'fixed_weekly_3day_dry_holdout_past_only',
    'event_sales_requirement': 'complete_event_sales_zero_included_missing_excluded',
    'recovery_start_horizon': 7, 'recovery_confirmation_horizon': 8,
    'scope': '2025H2 강남·춘천의 고정 분석업종; 관측기상 조건부 예측',
    'not_verified': ['인과적 피해 정답', '실제 발행 예보 성능', '사업체별 소상공인 자격', '562억의 확정 가용 재원'],
    'climate_requests': [json.load(open(path, encoding='utf-8')) for path in glob.glob(os.path.join(WORK, 'climate', '*.request.json'))],
    'code_assistance': 'Codex: 코드 작성·검증 보조; 실자료 결과 확인은 실행자 수행',
    'run_status': 'all_notebook_analysis_cells_reached_finalize'}
with open(os.path.join(OUT, '실행기록.json'), 'w', encoding='utf-8') as stream:
    json.dump(manifest, stream, ensure_ascii=False, indent=2, default=str)
with open(os.path.join(OUT, '실행환경_버전.txt'), 'w', encoding='utf-8') as stream:
    stream.write('\n'.join(f'{k}=={v}' for k, v in packages.items()))

mapping = [
    ('4-2 비 효과', '1-3_비_효과_그룹별.csv'), ('4-3 강수 임계', '1-2_강수_임계값.csv'),
    ('4-6 강수 판정', '../climate/era5_observation_calibration.csv'),
    ('4-6 연간 변동성', '../climate/climate_trends.csv'),
    ('5-1 손실', '2-1_원화피해.csv'), ('5-1 방법 범위', '11-1_손실_방법범위.csv'),
    ('5-3 연간 시나리오', '5-2_연간손실_시나리오.csv'), ('6 회복', '3-1_회복_요약.csv'),
    ('7-4 사후 중첩검증', '5-3_지수v2_중첩교차검증_요약.csv'),
    ('7 최종 지수 동일목표 순차검증', '10-6_최종지수_동일목표_순차검증.csv'),
    ('7 최종 지수', '5-5_회복탄력성지수_순차동일공식.csv'),
    ('7 지수 불확실성', '10-7_최종지수_순차검증_불확실성.csv'),
    ('8 기준선 진단', '10-9_기준선_지역업종월별_진단.csv'),
    ('8 피해 규모 예측', '10-5_피해규모_성능과불확실성.csv'),
    ('8 보전율별 비용', '11-2_보전율별_두지역합계.csv'), ('8 지속시간 시나리오', '11-4_지속시간_시나리오.csv'),
    ('8-4 공식 상권', '7-1_공식상권_회복탄력성.csv'), ('8-5 월별 통신 전체/참고기간', '8-3_유동인구_비오는달_점검.csv')]
map_table = pd.DataFrame(mapping, columns=['보고서', '결과파일'])
map_table['생성됨'] = map_table.결과파일.map(lambda name: os.path.exists(os.path.normpath(os.path.join(OUT, name))))
map_table.to_csv(os.path.join(OUT, '보고서_결과파일_대응표.csv'), index=False, encoding='utf-8-sig')
external_files = [{'파일': os.path.basename(path), 'sha256': file_sha256(path), '포함여부': '원본은 별도 제출 여부 확인', '경로': path} for path in climate_inputs if not path.endswith('.request.json')]
pd.DataFrame(external_files).to_csv(os.path.join(OUT, '외부기후자료_원본목록.csv'), index=False, encoding='utf-8-sig')

readme = '''# 제출 재현 안내
1. 원본 3종 zip을 설정한 ZIP_DIR에 둡니다. 기상 파일 3종의 이름과 열 정의를 확인합니다.
2. 맨 위 설정 셀부터 실행합니다. 공식 상권 단계만 SERVICE_KEY가 필요합니다.
3. out_round2의 실행기록·환경 버전·결과파일 대응표를 확인합니다.
4. 10-1 관측률·10-8 기준선·10-9 월별 편향을 확인합니다. 10-4 검증가능성 표를 먼저 보고 10-5 성능을 읽습니다. 자료 부족은 성능 0이 아닙니다.
5. Colab 메뉴 파일 > 다운로드 > .ipynb를 사용해 실제 실행 출력이 들어 있는 노트북을 저장합니다.
6. 아래 결과 zip과 함께 제출합니다. 코드 목록이나 합성 테스트 출력만으로 실자료 재현을 주장하지 않습니다.
입력 데이터는 결과 zip에 포함하지 않습니다. 외부 데이터 원본 제출 가능 범위·출처는 별도 확인합니다.
연간 추정은 시나리오이며 일강수 전망을 영업시간 전망으로 변환하지 않습니다.
'''
with open(os.path.join(OUT, '제출_재현안내.md'), 'w', encoding='utf-8') as stream:
    stream.write(readme)
archive = os.path.join(WORK, '취합1회차_4대수정_실행결과.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for directory in [OUT, FIG, os.path.join(WORK, 'report_figs')]:
        if not os.path.isdir(directory):
            continue
        for path in glob.glob(os.path.join(directory, '**', '*'), recursive=True):
            if os.path.isfile(path):
                z.write(path, os.path.relpath(path, WORK))
    for name in ['era5_annual.csv', 'era5_observation_calibration.csv', 'climate_trends.csv', 'cmip6_daily_scenarios.csv']:
        path = os.path.join(WORK, 'climate', name)
        if os.path.isfile(path):
            z.write(path, 'climate/' + name)
print('결과 묶음:', archive)
print('마지막으로 이 노트북을 실제 실행 출력이 포함된 .ipynb로 다운로드하세요.')
